# RetailPulse - MAIN NOTEBOOK (all stages end-to-end)
**AI-Powered Customer Analytics & Demand Forecasting Platform**

| | |
|---|---|
| **Project made by** | **Santosh Kumar** |
| **Organisation** | Zidio Development - Data Science & Analytics Domain |
| **Phase** | All 28 days - complete pipeline |
| **Version** | 2.0 - Industry Edition (March 2026) |

> Run order: `Runtime -> Run all`. The first 3 cells install missing packages, write the helper library `retailpulse_utils.py` and import it.

## Table of contents
1. **01 - Data Collection & Exploratory Data Analysis (EDA)**
2. **02 - Data Cleaning, Validation & Feature Engineering**
3. **03 - Customer Segmentation (RFM, K-Means, DBSCAN, Cohorts)**
4. **04 - Time-Series Preparation, Stationarity & Decomposition**
5. **05 - Baseline Demand Forecasting with Prophet**
6. **06 - Deep Learning Forecasting with LSTM (PyTorch)**
7. **07 - MLflow Experiment Tracking (Week 1 Baselines)**
8. **08 - Hybrid Forecasting Ensemble (Prophet + LSTM)**
9. **09 - Churn Prediction with XGBoost + SHAP Explainability**
10. **10 - Inventory Optimization using Forecasted Demand**
11. **11 - Hyper-parameter Tuning with Optuna & Feature Importance**
12. **12 - Data & Model Drift Detection (PSI, KS, Evidently)**
13. **13 - Model Refinement & Automated Retraining Pipeline (Airflow)**
14. **14 - MLflow Week-2 Tracking, Acceptance Checklist & Model Card**
15. **15 - Streamlit Dashboard Launch (Colab) & Final DOCX Report**

> This single notebook runs the complete RetailPulse project: EDA -> cleaning -> segmentation -> forecasting -> churn -> inventory -> tuning -> drift -> MLflow -> report.

### Setup (run once)
1) install missing packages  2) write helper library  3) import & create workspace folders

In [ ]:

# Installs ONLY what is missing (so Colab does not need a restart in most cases)
import importlib.util, subprocess, sys
need = {"openpyxl": "openpyxl", "kagglehub": "kagglehub", "xgboost": "xgboost", "shap": "shap", "optuna": "optuna",
        "mlflow": "mlflow", "statsmodels": "statsmodels", "prophet": "prophet", "docx": "python-docx"}
missing = [pip for mod, pip in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
    print("If Colab shows 'Restart runtime', click it once and run all cells again.")
else:
    print("All packages already installed.")


In [ ]:
%%writefile retailpulse_utils.py
"""
RetailPulse - AI-Powered Customer Analytics & Demand Forecasting Platform
Shared utility library (single file, works in Google Colab + local + Streamlit).

Author : Santosh Kumar
Project: Zidio Development - Data Science & Analytics (March 2026)

Design rules
------------
* Every public function is defensive: optional libraries (Prophet, PyTorch, SHAP,
  MLflow, Evidently) are wrapped so that the notebook NEVER crashes - it falls back
  to a simpler method and prints a clear message.
* Works with 3 data sources: (1) Kaggle via kagglehub, (2) your own uploaded
  CSV / XLSX, (3) built-in synthetic demo data (automatic fallback).
"""
from __future__ import annotations

import importlib
import importlib.util
import json
import math
import os
import pickle
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# 0. CONFIG
# ----------------------------------------------------------------------------
AUTHOR = "Santosh Kumar"
PROJECT = "RetailPulse - AI-Powered Customer Analytics & Demand Forecasting Platform"
SEED = 42

CONFIG = {
    # "auto" | "uci" | "synthetic" | "upload" | "demo"
    "DATASET": "auto",
    "MAX_ROWS": 400_000,          # sampling cap (customer-level) for huge files
    "FORECAST_HORIZON": 30,       # days ahead (F-03)
    "CHURN_WINDOW_DAYS": 90,      # label window for churn
    "TOP_N_SKUS": 40,             # SKUs used in inventory optimisation
    "LEAD_TIME_DAYS": 5,
    "REVIEW_PERIOD_DAYS": 7,
    "SERVICE_LEVEL": 0.95,
    "ORDER_COST": 40.0,
    "HOLDING_RATE": 0.20,
    "PATH": None,                 # e.g. "/content/drive/MyDrive/online_retail_II.xlsx"
}

KAGGLE_HANDLES = {
    "uci": "cgrymn/online-retail-ii-uci-dataset",
    "synthetic": "mrayyanshehzad/synthetic-retail-dataset-10-million-transactions",
}

ROOT = Path(os.environ.get("RETAILPULSE_HOME", "retailpulse_work")).resolve()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
CHARTS = ROOT / "outputs" / "charts"
MODELS = ROOT / "models"
REPORTS = ROOT / "reports"
METRICS_FILE = ROOT / "outputs" / "metrics.json"
CHART_INDEX = ROOT / "outputs" / "charts_index.json"

PALETTE = {
    "navy": "#1F3A5F", "teal": "#14B8A6", "orange": "#F59E0B",
    "red": "#EF4444", "purple": "#6366F1", "green": "#22C55E",
    "grey": "#64748B", "pink": "#EC4899", "sky": "#0EA5E9",
}
COLORS = list(PALETTE.values())

_CACHE: dict = {}


def setup(root: str | None = None, show: bool = True):
    """Create the workspace folders and apply the plotting theme."""
    global ROOT, DATA_RAW, DATA_PROC, CHARTS, MODELS, REPORTS, METRICS_FILE, CHART_INDEX
    if root:
        ROOT = Path(root).resolve()
        DATA_RAW, DATA_PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
        CHARTS, MODELS, REPORTS = ROOT / "outputs" / "charts", ROOT / "models", ROOT / "reports"
        METRICS_FILE, CHART_INDEX = ROOT / "outputs" / "metrics.json", ROOT / "outputs" / "charts_index.json"
    for d in (DATA_RAW, DATA_PROC, CHARTS, MODELS, REPORTS):
        d.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid", palette=COLORS, font_scale=1.0)
    plt.rcParams.update({
        "figure.figsize": (10, 5), "figure.dpi": 100, "savefig.dpi": 130,
        "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 11,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.prop_cycle": matplotlib.cycler(color=COLORS),
    })
    if show:
        print(f"RetailPulse workspace ready -> {ROOT}")


setup(show=False)


def ensure_packages(pkgs: dict[str, str] | None = None):
    """Install only the packages that are really missing (avoids Colab restarts)."""
    pkgs = pkgs or {
        "openpyxl": "openpyxl", "kagglehub": "kagglehub", "xgboost": "xgboost",
        "shap": "shap", "optuna": "optuna", "mlflow": "mlflow",
        "statsmodels": "statsmodels", "prophet": "prophet",
    }
    missing = [pip for mod, pip in pkgs.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing missing packages:", missing)
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
    else:
        print("All required packages already available.")


# ----------------------------------------------------------------------------
# 1. LOGGING HELPERS (metrics + chart index feed the final DOCX report)
# ----------------------------------------------------------------------------
def _read_json(path: Path, default):
    try:
        return json.loads(path.read_text())
    except Exception:
        return default


def _json_safe(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, (np.ndarray,)):
        return o.tolist()
    if isinstance(o, (pd.Timestamp,)):
        return str(o)
    return str(o)


def log_metrics(section: str, values: dict):
    """Merge a dict of results into outputs/metrics.json (used by the report)."""
    data = _read_json(METRICS_FILE, {})
    data.setdefault(section, {}).update(values)
    METRICS_FILE.parent.mkdir(parents=True, exist_ok=True)
    METRICS_FILE.write_text(json.dumps(data, indent=2, default=_json_safe))


def get_metrics() -> dict:
    return _read_json(METRICS_FILE, {})


def save_fig(fig, fname: str, title: str = "", note: str = "", section: str = "General", show: bool = True):
    """Save a matplotlib figure to outputs/charts and register it for the report."""
    CHARTS.mkdir(parents=True, exist_ok=True)
    path = CHARTS / f"{fname}.png"
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    idx = _read_json(CHART_INDEX, {})
    idx[fname] = {"title": title or fname, "note": note, "section": section, "file": str(path)}
    CHART_INDEX.write_text(json.dumps(idx, indent=2, default=_json_safe))
    if show:
        plt.show()
    else:
        plt.close(fig)
    return path


def fmt_money(x):
    return f"{x:,.0f}"


# ----------------------------------------------------------------------------
# 2. SYNTHETIC DEMO DATA (automatic fallback - mimics UCI Online Retail II)
# ----------------------------------------------------------------------------
_ADJ = ["VINTAGE", "RETRO", "CLASSIC", "MODERN", "PINK", "BLUE", "RED", "GREEN", "WHITE", "GOLD",
        "SILVER", "FLORAL", "STRIPED", "SPOTTED", "HANDMADE", "LARGE", "SMALL", "MINI", "JUMBO", "LUXURY"]
_ITEMS = {
    "Kitchen": ["MUG", "TEA POT", "CAKE TIN", "LUNCH BOX", "JAM JAR", "SPOON SET", "BOWL", "APRON", "CUP", "PLATE"],
    "Home Decor": ["CANDLE HOLDER", "PHOTO FRAME", "CUSHION COVER", "LANTERN", "WALL CLOCK", "VASE", "DOORMAT", "MIRROR"],
    "Stationery": ["NOTEBOOK", "PENCIL SET", "GREETING CARD", "WRAPPING PAPER", "STICKER SET", "DIARY"],
    "Gifts & Toys": ["TEDDY BEAR", "GAME", "PUZZLE", "DOLL", "TOY CAR", "GIFT BOX", "MUSIC BOX"],
    "Garden": ["PLANT POT", "BIRD FEEDER", "GARDEN TROWEL", "WATERING CAN", "SEED TIN"],
    "Bags & Accessories": ["SHOPPER BAG", "PURSE", "UMBRELLA", "WASH BAG", "KEY RING", "SCARF"],
}
_COUNTRIES = ["United Kingdom", "Germany", "France", "EIRE", "Netherlands", "Spain", "Belgium",
              "Switzerland", "Portugal", "Australia", "Norway", "Italy"]
_COUNTRY_P = np.array([0.88, 0.025, 0.022, 0.02, 0.008, 0.006, 0.005, 0.004, 0.004, 0.0035, 0.0015, 0.001])
_COUNTRY_P = _COUNTRY_P / _COUNTRY_P.sum()


def generate_demo_data(n_customers: int = 1900, start: str = "2010-01-04", end: str = "2011-12-09",
                       seed: int = SEED) -> pd.DataFrame:
    """Synthetic UCI-style transactions: 8 columns exactly like Online Retail II."""
    rng = np.random.default_rng(seed)
    days = pd.date_range(start, end, freq="D")
    n_days = len(days)

    # product catalogue
    products = []
    code = 20000
    for cat, items in _ITEMS.items():
        for it in items:
            for adj in rng.choice(_ADJ, size=4, replace=False):
                code += int(rng.integers(1, 9))
                price = float(np.round(rng.lognormal(mean=0.9, sigma=0.55), 2))
                products.append((str(code), f"{adj} {it}", max(price, 0.29)))
    prod = pd.DataFrame(products, columns=["stock_code", "description", "price"])
    n_prod = len(prod)
    pop = 1.0 / np.arange(1, n_prod + 1) ** 0.9
    rng.shuffle(pop)
    pop = pop / pop.sum()

    # seasonality: Nov peak, Jan trough; weekday: no Saturday, Sunday lighter
    month_mult = {1: .7, 2: .75, 3: .9, 4: .85, 5: .95, 6: .95, 7: .95, 8: .95, 9: 1.15, 10: 1.3, 11: 1.8, 12: 1.2}
    dow_mult = {0: 1.0, 1: 1.15, 2: 1.1, 3: 1.2, 4: 0.9, 5: 0.0, 6: 0.55}
    trend = np.linspace(1.0, 1.3, n_days)
    day_w = np.array([month_mult[d.month] * dow_mult[d.dayofweek] for d in days]) * trend
    day_w = day_w / day_w.sum()

    rows = []
    inv_no = 530000

    def make_order(cust_id, day, country):
        nonlocal inv_no
        inv_no += 1
        n_lines = 1 + rng.geometric(1 / 5.0)
        n_lines = min(n_lines, 40)
        pids = rng.choice(n_prod, size=n_lines, replace=False, p=pop)
        hour = int(rng.choice(np.arange(8, 19), p=np.array([.04, .08, .13, .14, .10, .10, .12, .11, .09, .06, .03]) / 1.0))
        ts = pd.Timestamp(day) + pd.Timedelta(hours=hour, minutes=int(rng.integers(0, 60)))
        for pid in pids:
            sc, desc, pr = prod.iloc[pid]
            qty = int(np.clip(np.round(rng.lognormal(1.3, 0.9)), 1, 120))
            if rng.random() < 0.12:
                qty = int(rng.choice([6, 12, 24, 48]))
            price = float(np.round(pr * rng.choice([1.0, 1.0, 1.0, 0.95, 1.05]), 2))
            rows.append((str(inv_no), sc, desc, qty, ts, price, cust_id, country))
        # cancellation
        if rng.random() < 0.02:
            inv_no += 1
            sc, desc, pr = prod.iloc[rng.choice(n_prod, p=pop)]
            rows.append(("C" + str(inv_no), sc, desc, -int(rng.integers(1, 12)), ts + pd.Timedelta(days=int(rng.integers(1, 9))), pr, cust_id, country))

    for c in range(n_customers):
        cid = 12000 + c
        country = rng.choice(_COUNTRIES, p=_COUNTRY_P)
        first_idx = int(rng.beta(1.2, 2.2) * n_days * 0.92)
        lifetime = int(rng.exponential(520)) + 45
        last_idx = min(n_days - 1, first_idx + lifetime)
        rate = rng.gamma(1.3, 0.017)         # orders / day
        n_orders = max(1, rng.poisson(rate * (last_idx - first_idx + 1)))
        w = day_w[first_idx:last_idx + 1]
        if w.sum() <= 0:
            continue
        w = w / w.sum()
        chosen = rng.choice(np.arange(first_idx, last_idx + 1), size=n_orders, p=w)
        for di in chosen:
            make_order(float(cid), days[di], country)

    # guest orders (no customer id)
    n_guest = int(n_days * 3.0)
    for di in rng.choice(np.arange(n_days), size=n_guest, p=day_w):
        make_order(np.nan, days[di], "United Kingdom")

    df = pd.DataFrame(rows, columns=["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID", "Country"])

    # realistic dirt
    dup = df.sample(frac=0.003, random_state=seed)
    zero = df.sample(frac=0.001, random_state=seed + 1).copy()
    zero["Price"] = 0.0
    fee = df.sample(n=40, random_state=seed + 2).copy()
    fee["StockCode"], fee["Description"], fee["Quantity"], fee["Price"] = "POST", "POSTAGE", 1, 18.0
    big = df.sample(n=5, random_state=seed + 3).copy()
    big["Quantity"] = rng.integers(3000, 9000, size=5)
    df = pd.concat([df, dup, zero, fee, big], ignore_index=True)
    miss = df.sample(frac=0.004, random_state=seed + 4).index
    df.loc[miss, "Description"] = np.nan
    df = df.sort_values("InvoiceDate").reset_index(drop=True)
    return df


# ----------------------------------------------------------------------------
# 3. DATA LOADING (Kaggle / upload / demo) + column standardisation
# ----------------------------------------------------------------------------
_COLMAP = {
    "invoice": ["invoice", "invoiceno", "invoiceid", "orderid", "transactionid", "orderno", "billno"],
    "stock_code": ["stockcode", "productid", "sku", "itemcode", "productcode", "itemid"],
    "description": ["description", "productname", "itemname", "product", "item"],
    "quantity": ["quantity", "qty", "units", "unitssold", "quantitysold"],
    "invoice_date": ["invoicedate", "orderdate", "transactiondate", "date", "timestamp", "datetime", "purchasedate"],
    "unit_price": ["price", "unitprice", "sellingprice", "priceperunit", "unitcost"],
    "customer_id": ["customerid", "custid", "userid", "clientid", "customer"],
    "country": ["country", "region", "location", "storelocation", "city", "state"],
    "category": ["category", "productcategory", "department"],
    "total": ["totalamount", "totalprice", "total", "revenue", "sales", "amount", "linetotal"],
}


def _norm(c: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(c).lower())


def standardize(df: pd.DataFrame) -> pd.DataFrame:
    """Map any retail-like schema to the 8 canonical columns. Raises ValueError if impossible."""
    norm = {_norm(c): c for c in df.columns}
    out = pd.DataFrame(index=df.index)
    for target, cands in _COLMAP.items():
        for cand in cands:
            if cand in norm:
                out[target] = df[norm[cand]]
                break
    if "invoice_date" not in out:
        raise ValueError("No date column found")
    out["invoice_date"] = pd.to_datetime(out["invoice_date"], errors="coerce")
    if out["invoice_date"].notna().mean() < 0.8:
        raise ValueError("Date column could not be parsed")
    if "quantity" not in out:
        out["quantity"] = 1
    out["quantity"] = pd.to_numeric(out["quantity"], errors="coerce")
    if "unit_price" not in out:
        if "total" in out:
            out["unit_price"] = pd.to_numeric(out["total"], errors="coerce") / out["quantity"].replace(0, np.nan)
        else:
            raise ValueError("No price / amount column found")
    out["unit_price"] = pd.to_numeric(out["unit_price"], errors="coerce")
    if "invoice" not in out:
        key = out["customer_id"].astype(str) if "customer_id" in out else pd.Series("na", index=out.index)
        out["invoice"] = key + "_" + out["invoice_date"].dt.strftime("%Y%m%d%H")
    if "stock_code" not in out:
        out["stock_code"] = out["description"] if "description" in out else "UNKNOWN"
    if "description" not in out:
        out["description"] = out["stock_code"]
    if "customer_id" not in out:
        out["customer_id"] = np.nan
    if "country" not in out:
        out["country"] = "Unknown"
    cols = ["invoice", "stock_code", "description", "quantity", "invoice_date", "unit_price", "customer_id", "country"]
    if "category" in out:
        cols.append("category")
    out = out[cols].copy()
    out["invoice"] = out["invoice"].astype(str)
    out["stock_code"] = out["stock_code"].astype(str)
    return out


def _read_any(path: Path, max_rows: int) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix in (".xlsx", ".xls"):
        sheets = pd.read_excel(path, sheet_name=None)
        return pd.concat(sheets.values(), ignore_index=True)
    if suffix in (".csv", ".gz", ".txt"):
        try:
            head = pd.read_csv(path, nrows=2000, encoding="utf-8")
            enc = "utf-8"
        except Exception:
            enc = "ISO-8859-1"
            head = pd.read_csv(path, nrows=2000, encoding=enc)
        size = path.stat().st_size
        est_rows = size / max(len(head.to_csv(index=False)) / len(head), 1)
        if est_rows <= max_rows * 1.2:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        # big file: customer-level hash sampling keeps whole customer histories
        keep_every = int(math.ceil(est_rows / max_rows))
        print(f"Large file (~{est_rows:,.0f} rows). Sampling 1/{keep_every} of customers for Colab memory safety ...")
        chunks = []
        for chunk in pd.read_csv(path, encoding=enc, chunksize=500_000, low_memory=False):
            try:
                std = standardize(chunk)
                if std["customer_id"].notna().mean() > 0.5:
                    cid = pd.to_numeric(std["customer_id"], errors="coerce").fillna(0).astype("int64")
                    chunk = chunk.loc[(cid % keep_every == 0).values]
                else:
                    chunk = chunk.sample(frac=1 / keep_every, random_state=SEED)
            except Exception:
                chunk = chunk.sample(frac=1 / keep_every, random_state=SEED)
            chunks.append(chunk)
        return pd.concat(chunks, ignore_index=True)
    raise ValueError(f"Unsupported file type: {path.name}")


def _candidate_files() -> list[Path]:
    dirs = [DATA_RAW, Path.cwd(), Path("/content"), Path.cwd() / "data", Path.cwd() / "data" / "raw",
            Path.cwd().parent / "data" / "sample", Path.cwd() / "data" / "sample"]
    found, seen = [], set()
    for d in dirs:
        if d.exists() and d.is_dir():
            for p in sorted(d.iterdir()):
                if p.is_file() and p.suffix.lower() in (".csv", ".xlsx", ".xls") and p not in seen:
                    seen.add(p)
                    found.append(p)
    return found


def _kaggle_files(handle: str) -> list[Path]:
    import kagglehub
    path = Path(kagglehub.dataset_download(handle))
    files = [p for p in path.rglob("*") if p.suffix.lower() in (".csv", ".xlsx", ".xls")]
    return sorted(files, key=lambda p: -p.stat().st_size)


def load_raw(dataset: str | None = None, path: str | None = None, max_rows: int | None = None, verbose: bool = True):
    """
    Load the raw dataset. Returns (df_standardised, source_label).
    Order for dataset="auto": uploaded/local file -> Kaggle UCI Online Retail II -> Kaggle synthetic -> demo.
    """
    dataset = dataset or CONFIG["DATASET"]
    max_rows = max_rows or CONFIG["MAX_ROWS"]

    def _try(p: Path, label: str):
        try:
            raw = _read_any(p, max_rows)
            std = standardize(raw)
            if verbose:
                print(f"[OK] Loaded {label}: {p.name} -> {len(std):,} rows")
            return std, f"{label} ({p.name})"
        except Exception as e:  # noqa
            if verbose:
                print(f"[skip] {p.name}: {e}")
            return None

    if path:
        r = _try(Path(path), "User file")
        if r:
            return r

    if dataset in ("auto", "upload"):
        for p in _candidate_files():
            if p.name.startswith("retailpulse_"):
                continue
            r = _try(p, "Uploaded/local file")
            if r:
                return r

    if dataset in ("auto", "uci", "synthetic") and importlib.util.find_spec("kagglehub") is not None:
        order = ["uci", "synthetic"] if dataset in ("auto", "uci") else ["synthetic"]
        for key in order:
            try:
                if verbose:
                    print(f"Downloading from Kaggle: {KAGGLE_HANDLES[key]} ...")
                for p in _kaggle_files(KAGGLE_HANDLES[key]):
                    r = _try(p, f"Kaggle {key}")
                    if r:
                        return r
            except Exception as e:  # noqa
                if verbose:
                    print(f"[skip] Kaggle {key} failed ({type(e).__name__}: {str(e)[:90]})")

    if verbose:
        print("[INFO] Real dataset not available -> using built-in synthetic demo data (same schema as UCI Online Retail II).")
    return standardize(generate_demo_data()), "Synthetic demo data"


# ----------------------------------------------------------------------------
# 4. CLEANING + FEATURE ENGINEERING
# ----------------------------------------------------------------------------
_NON_PRODUCT = {"POST", "D", "M", "DOT", "CRUK", "C2", "BANK CHARGES", "AMAZONFEE", "ADJUST", "ADJUST2", "S", "B",
                "PADS", "TEST001", "TEST002", "GIFT", "SAMPLES", "CARRIAGE", "DISCOUNT"}

_CATEGORY_RULES = [
    ("Kitchen", ["MUG", "TEA", "CAKE", "LUNCH", "JAM", "SPOON", "BOWL", "APRON", "CUP", "PLATE", "KITCHEN", "BAKING", "JUG", "TIN"]),
    ("Home Decor", ["CANDLE", "FRAME", "CUSHION", "LANTERN", "CLOCK", "VASE", "DOORMAT", "MIRROR", "HEART", "DECORATION", "LIGHT", "HOLDER"]),
    ("Stationery", ["NOTEBOOK", "PENCIL", "CARD", "WRAP", "STICKER", "DIARY", "PAPER", "PEN", "BOOK"]),
    ("Gifts & Toys", ["BEAR", "GAME", "PUZZLE", "DOLL", "TOY", "GIFT", "MUSIC", "CHRISTMAS", "BOX"]),
    ("Garden", ["PLANT", "BIRD", "GARDEN", "WATERING", "SEED", "FLOWER"]),
    ("Bags & Accessories", ["BAG", "PURSE", "UMBRELLA", "WASH", "KEY RING", "SCARF", "SHOPPER"]),
]


def infer_category(desc: pd.Series) -> pd.Series:
    d = desc.fillna("").astype(str).str.upper()
    out = pd.Series("Other", index=desc.index, dtype=object)
    for cat, words in reversed(_CATEGORY_RULES):          # first rule wins
        mask = d.str.contains("|".join(re.escape(w) for w in words), regex=True)
        out[mask] = cat
    return out


def clean(df_std: pd.DataFrame, verbose: bool = True):
    """
    Returns (sales, returns, report).
      sales   - valid positive sales lines with revenue + calendar features
      returns - cancellations / negative quantity lines
      report  - dict of rows removed at each step (for the waterfall chart)
    """
    rep = {"raw_rows": len(df_std)}
    df = df_std.copy()
    df = df.dropna(subset=["invoice_date"])
    rep["after_drop_bad_dates"] = len(df)

    before = len(df)
    df = df.drop_duplicates(subset=["invoice", "stock_code", "quantity", "invoice_date", "unit_price", "customer_id"])
    rep["duplicates_removed"] = before - len(df)

    df["is_cancel"] = df["invoice"].astype(str).str.upper().str.startswith("C") | (df["quantity"] < 0)
    returns = df[df["is_cancel"]].copy()
    returns["quantity"] = returns["quantity"].abs()
    rep["returns_separated"] = len(returns)
    df = df[~df["is_cancel"]].copy()

    before = len(df)
    df = df[(df["quantity"] > 0) & (df["unit_price"] > 0)]
    rep["invalid_qty_or_price_removed"] = before - len(df)

    before = len(df)
    df = df[~df["stock_code"].astype(str).str.upper().isin(_NON_PRODUCT)]
    rep["non_product_codes_removed"] = before - len(df)

    df["description"] = df["description"].fillna("UNKNOWN").astype(str).str.strip().str.upper()
    df["revenue"] = df["quantity"] * df["unit_price"]
    # NEW COLUMN 1: invoice_no -> numeric part of the invoice id (clean key for distinct-order counts / joins)
    df["invoice_no"] = pd.to_numeric(df["invoice"].astype(str).str.extract(r"(\d+)")[0], errors="coerce").astype("Int64")
    # NEW COLUMN 2: order_value -> total revenue of the whole invoice (gives AOV / basket value without re-grouping)
    df["order_value"] = df.groupby("invoice")["revenue"].transform("sum")

    # extreme outliers (beyond 99.95th percentile of line revenue / quantity)
    before = len(df)
    q_hi, r_hi = df["quantity"].quantile(0.9995), df["revenue"].quantile(0.9995)
    df = df[(df["quantity"] <= max(q_hi, 1)) & (df["revenue"] <= max(r_hi, 1))]
    rep["extreme_outliers_removed"] = before - len(df)

    df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce")
    rep["rows_without_customer_id"] = int(df["customer_id"].isna().sum())

    if "category" not in df.columns:
        df["category"] = infer_category(df["description"])
    df["category"] = df["category"].fillna("Other")

    d = df["invoice_date"]
    df["date"] = d.dt.normalize()
    df["year"], df["month"], df["day"] = d.dt.year, d.dt.month, d.dt.day
    df["dow"], df["hour"] = d.dt.dayofweek, d.dt.hour
    df["week"] = d.dt.isocalendar().week.astype(int)
    df["quarter"] = d.dt.quarter
    df["is_weekend"] = (df["dow"] >= 5).astype(int)
    df["year_month"] = d.dt.to_period("M").astype(str)
    df["month_name"] = d.dt.strftime("%b")
    df = df.sort_values("invoice_date").reset_index(drop=True)
    rep["final_rows"] = len(df)
    if verbose:
        for k, v in rep.items():
            print(f"{k:<32}: {v:>12,}")
    return df, returns.reset_index(drop=True), rep


def validate_data(df: pd.DataFrame) -> pd.DataFrame:
    """Great-Expectations-style checks implemented natively (no API-breaking dependency)."""
    checks = [
        ("No missing invoice_date", df["invoice_date"].notna().all()),
        ("quantity > 0 for all rows", (df["quantity"] > 0).all()),
        ("unit_price > 0 for all rows", (df["unit_price"] > 0).all()),
        ("revenue == quantity * unit_price", np.allclose(df["revenue"], df["quantity"] * df["unit_price"])),
        ("invoice_date is sorted ascending", df["invoice_date"].is_monotonic_increasing),
        ("No duplicate full rows", not df.duplicated(["invoice", "stock_code", "quantity", "invoice_date", "unit_price", "customer_id"]).any()),
        ("customer_id missing < 40%", df["customer_id"].isna().mean() < 0.40),
        ("country has no nulls", df["country"].notna().all()),
        ("unit_price 99.9th pct < 1000", df["unit_price"].quantile(0.999) < 1000),
        ("Date range covers > 180 days", (df["date"].max() - df["date"].min()).days > 180),
    ]
    out = pd.DataFrame(checks, columns=["expectation", "passed"])
    out["passed"] = out["passed"].astype(bool)
    return out


def _cached(name: str, fn):
    if name not in _CACHE:
        _CACHE[name] = fn()
    return _CACHE[name]


def get_data(force: bool = False, verbose: bool = True):
    """Load + clean once per session; also stored on disk (pickle) for fast re-use between notebooks."""
    pk = DATA_PROC / "retailpulse_clean.pkl"
    if not force and "data" in _CACHE:
        return _CACHE["data"]
    if not force and pk.exists():
        try:
            _CACHE["data"] = pickle.load(open(pk, "rb"))
            return _CACHE["data"]
        except Exception:
            pass
    raw, source = load_raw(path=CONFIG.get("PATH"), verbose=verbose)
    sales, returns, rep = clean(raw, verbose=False)
    if verbose:
        print(f"Source: {source} | raw rows {rep['raw_rows']:,} -> clean rows {rep['final_rows']:,}")
    bundle = {"raw": raw, "sales": sales, "returns": returns, "report": rep, "source": source}
    pickle.dump(bundle, open(pk, "wb"))
    _CACHE["data"] = bundle
    log_metrics("data", {"source": source, **{k: int(v) for k, v in rep.items()},
                         "date_min": str(sales["date"].min().date()), "date_max": str(sales["date"].max().date()),
                         "customers": int(sales["customer_id"].nunique()), "products": int(sales["stock_code"].nunique()),
                         "countries": int(sales["country"].nunique()), "total_revenue": float(sales["revenue"].sum()),
                         "orders": int(sales["invoice"].nunique())})
    return bundle


def get_sales():
    return get_data()["sales"]


def get_customer_sales():
    """Sales with a customer id; if the dataset has none, falls back to demo data (with a warning)."""
    s = get_sales()
    sc = s[s["customer_id"].notna()]
    if sc["customer_id"].nunique() >= 100:
        return sc
    print("[WARN] Dataset has too few customer IDs for customer analytics -> using synthetic demo customers for this section.")
    raw = standardize(generate_demo_data())
    demo, _, _ = clean(raw, verbose=False)
    return demo[demo["customer_id"].notna()]


# ----------------------------------------------------------------------------
# 5. CUSTOMER ANALYTICS: RFM, K-Means, DBSCAN, cohorts
# ----------------------------------------------------------------------------
def rfm_table(df: pd.DataFrame, snapshot=None) -> pd.DataFrame:
    snapshot = pd.Timestamp(snapshot) if snapshot is not None else df["invoice_date"].max().normalize() + pd.Timedelta(days=1)
    g = df.groupby("customer_id")
    t = pd.DataFrame({
        "recency": (snapshot - g["invoice_date"].max()).dt.days,
        "frequency": g["invoice"].nunique(),
        "monetary": g["revenue"].sum(),
        "n_products": g["stock_code"].nunique(),
        "first_purchase": g["invoice_date"].min(),
        "last_purchase": g["invoice_date"].max(),
    })
    t["avg_order_value"] = t["monetary"] / t["frequency"]
    t["tenure_days"] = (snapshot - t["first_purchase"]).dt.days
    return t


def rfm_scores(t: pd.DataFrame) -> pd.DataFrame:
    t = t.copy()
    t["R"] = pd.qcut(t["recency"].rank(method="first", ascending=False), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    t["F"] = pd.qcut(t["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    t["M"] = pd.qcut(t["monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    t["RFM_score"] = t["R"] * 100 + t["F"] * 10 + t["M"]
    t["RFM_total"] = t["R"] + t["F"] + t["M"]
    r, fm = t["R"], (t["F"] + t["M"]) / 2
    conds = [
        (r >= 4) & (fm >= 4.5), (r == 1) & (fm >= 4), (r <= 2) & (fm >= 3), (r >= 3) & (fm >= 3.5),
        (r >= 4) & (fm >= 2.5), (r >= 4), (r == 3) & (fm >= 2.5), (r == 3), (r == 2), (r == 1) & (fm <= 2),
    ]
    names = ["Champions", "Can't Lose Them", "At Risk", "Loyal Customers", "Potential Loyalists",
             "New Customers", "Need Attention", "Promising", "About To Sleep", "Lost"]
    t["rule_segment"] = np.select(conds, names, default="Hibernating")
    return t


def kmeans_search(rfm: pd.DataFrame, ks=range(2, 11)):
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score, davies_bouldin_score
    from sklearn.preprocessing import StandardScaler
    X = StandardScaler().fit_transform(np.log1p(rfm[["recency", "frequency", "monetary"]]))
    rows = []
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(X), size=min(len(X), 4000), replace=False)
    for k in ks:
        km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
        rows.append({"k": k, "inertia": km.inertia_,
                     "silhouette": silhouette_score(X[idx], km.labels_[idx]),
                     "davies_bouldin": davies_bouldin_score(X, km.labels_)})
    return pd.DataFrame(rows), X


def fit_kmeans(rfm: pd.DataFrame, X: np.ndarray, k: int):
    from sklearn.cluster import KMeans
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(X)
    out = rfm.copy()
    out["cluster"] = km.labels_
    prof = out.groupby("cluster")[["recency", "frequency", "monetary"]].mean()
    med = out[["recency", "frequency", "monetary"]].median()
    names = {}
    table = {
        (1, 1, 1): "Champions", (1, 1, 0): "Loyal Regulars", (1, 0, 1): "Big Spenders", (1, 0, 0): "New / Promising",
        (0, 1, 1): "Can't Lose Them", (0, 1, 0): "At-Risk Regulars", (0, 0, 1): "Dormant Big Spenders", (0, 0, 0): "Hibernating / Lost",
    }
    base_of = {}
    for c, r in prof.iterrows():
        key = (int(r["recency"] < med["recency"]), int(r["frequency"] >= med["frequency"]), int(r["monetary"] >= med["monetary"]))
        base_of[c] = table[key]
    descr = {2: ["Top Value", "Entry Value"], 3: ["Top Value", "Mid Value", "Entry Value"],
             4: ["Top Value", "Upper-Mid Value", "Lower-Mid Value", "Entry Value"]}
    for base in set(base_of.values()):
        members = [c for c, b in base_of.items() if b == base]
        if len(members) == 1:
            names[members[0]] = base
        else:
            members = sorted(members, key=lambda c: -prof.loc[c, "monetary"])
            labels = descr.get(len(members), [f"Group {i + 1}" for i in range(len(members))])
            for c, lab in zip(members, labels):
                names[c] = f"{base} - {lab}"
    out["segment"] = out["cluster"].map(names)
    return out, km, names


def fit_dbscan(X: np.ndarray, min_samples: int = 6, pct: float = 92):
    from sklearn.cluster import DBSCAN
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=min_samples).fit(X)
    kdist = np.sort(nn.kneighbors(X)[0][:, -1])
    eps = float(np.percentile(kdist, pct))
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
    return labels, eps, kdist


def cohort_matrix(df: pd.DataFrame) -> pd.DataFrame:
    d = df[["customer_id", "invoice_date"]].copy()
    d["order_month"] = d["invoice_date"].dt.to_period("M")
    d["cohort"] = d.groupby("customer_id")["invoice_date"].transform("min").dt.to_period("M")
    d["offset"] = (d["order_month"].dt.year - d["cohort"].dt.year) * 12 + (d["order_month"].dt.month - d["cohort"].dt.month)
    m = d.groupby(["cohort", "offset"])["customer_id"].nunique().unstack(fill_value=0)
    ret = m.div(m[0], axis=0) * 100
    last = d["order_month"].max()
    for coh in ret.index:                     # months that have not happened yet -> NaN (not 0)
        max_off = (last.year - coh.year) * 12 + (last.month - coh.month)
        ret.loc[coh, [c for c in ret.columns if c > max_off]] = np.nan
    ret.index = ret.index.astype(str)
    return ret.round(1)


# ----------------------------------------------------------------------------
# 6. TIME SERIES + FORECASTING (Prophet, LSTM, Hybrid)
# ----------------------------------------------------------------------------
def daily_series(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("date").agg(quantity=("quantity", "sum"), revenue=("revenue", "sum"),
                               orders=("invoice", "nunique"), customers=("customer_id", "nunique"))
    idx = pd.date_range(g.index.min(), g.index.max(), freq="D")
    g = g.reindex(idx).fillna(0.0)
    g.index.name = "ds"
    return g.reset_index()


def mape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    m = y > 0
    return float(np.mean(np.abs(y[m] - p[m]) / y[m]) * 100) if m.any() else float("nan")


def wape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return float(np.abs(y - p).sum() / max(np.abs(y).sum(), 1e-9) * 100)


def smape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    den = (np.abs(y) + np.abs(p)) / 2
    m = den > 0
    return float(np.mean(np.abs(y[m] - p[m]) / den[m]) * 100)


def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y, float) - np.asarray(p, float)) ** 2)))


def mae(y, p):
    return float(np.mean(np.abs(np.asarray(y, float) - np.asarray(p, float))))


def block_mape(y, p, block=7):
    """MAPE of non-overlapping 7-day totals (weekly demand planning accuracy)."""
    y, p = np.asarray(y, float), np.asarray(p, float)
    n = (len(y) // block) * block
    if n == 0:
        return float("nan")
    ys, ps = y[:n].reshape(-1, block).sum(1), p[:n].reshape(-1, block).sum(1)
    return mape(ys, ps)


def forecast_metrics(y, p) -> dict:
    return {"MAPE_daily_%": mape(y, p), "MAPE_weekly_%": block_mape(y, p), "WAPE_%": wape(y, p),
            "sMAPE_%": smape(y, p), "RMSE": rmse(y, p), "MAE": mae(y, p), "Bias": float(np.mean(np.asarray(p) - np.asarray(y)))}


def split_series(ds: pd.DataFrame, horizon: int | None = None):
    h = horizon or CONFIG["FORECAST_HORIZON"]
    n = len(ds)
    return ds.iloc[: n - 2 * h].copy(), ds.iloc[n - 2 * h: n - h].copy(), ds.iloc[n - h:].copy()


def closed_dows(train: pd.DataFrame, target: str) -> list[int]:
    t = train.copy()
    t["dow"] = pd.to_datetime(t["ds"]).dt.dayofweek
    share_zero = t.groupby("dow")[target].apply(lambda s: (s <= 0).mean())
    return [int(d) for d, v in share_zero.items() if v >= 0.9]


def _apply_closed(pred, dates, closed):
    pred = np.clip(np.asarray(pred, float), 0, None)
    dow = pd.to_datetime(pd.Series(dates)).dt.dayofweek.values
    pred[np.isin(dow, closed)] = 0.0
    return pred


def naive_forecast(train: pd.DataFrame, future_dates, target: str = "quantity", weeks: int = 4):
    t = train.copy()
    t["dow"] = pd.to_datetime(t["ds"]).dt.dayofweek
    recent = t.tail(7 * weeks)
    means = recent.groupby("dow")[target].mean()
    dow = pd.to_datetime(pd.Series(future_dates)).dt.dayofweek
    return dow.map(means).fillna(recent[target].mean()).values.astype(float)


def holt_winters_forecast(train: pd.DataFrame, future_dates, target: str = "quantity"):
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    y = train[target].astype(float).values
    try:
        fit = ExponentialSmoothing(y, trend="add", damped_trend=True, seasonal="add", seasonal_periods=7).fit()
        pred = fit.forecast(len(future_dates))
    except Exception:
        pred = naive_forecast(train, future_dates, target)
    return _apply_closed(pred, future_dates, closed_dows(train, target))


def prophet_forecast(train: pd.DataFrame, future_dates, target: str = "quantity",
                     changepoint_prior_scale: float = 0.05, seasonality_prior_scale: float = 10.0,
                     seasonality_mode: str = "multiplicative", return_model: bool = False):
    """Prophet forecast. Falls back to Holt-Winters if Prophet is unavailable. Returns dict."""
    closed = closed_dows(train, target)
    try:
        import logging
        for name in ("cmdstanpy", "prophet", "stan"):
            logging.getLogger(name).setLevel(logging.ERROR)
        from prophet import Prophet
        m = Prophet(yearly_seasonality=len(train) >= 540, weekly_seasonality=True, daily_seasonality=False,
                    seasonality_mode=seasonality_mode, changepoint_prior_scale=changepoint_prior_scale,
                    seasonality_prior_scale=seasonality_prior_scale, interval_width=0.8)
        try:
            m.add_country_holidays(country_name="GB")
        except Exception:
            pass
        m.fit(train[["ds", target]].rename(columns={target: "y"}))
        fut = pd.DataFrame({"ds": pd.to_datetime(list(future_dates))})
        fc = m.predict(fut)
        out = {"pred": _apply_closed(fc["yhat"].values, fut["ds"], closed),
               "lower": _apply_closed(fc["yhat_lower"].values, fut["ds"], closed),
               "upper": _apply_closed(fc["yhat_upper"].values, fut["ds"], closed),
               "engine": "Prophet"}
        if return_model:
            out["model"], out["forecast_df"] = m, fc
        return out
    except Exception as e:  # noqa
        print(f"[INFO] Prophet unavailable ({type(e).__name__}); using Holt-Winters fallback.")
        p = holt_winters_forecast(train, future_dates, target)
        sd = float(np.std(train[target].tail(60)))
        return {"pred": p, "lower": np.clip(p - 1.28 * sd, 0, None), "upper": p + 1.28 * sd, "engine": "Holt-Winters (fallback)"}


def _calendar_feats(dates, closed):
    d = pd.to_datetime(pd.Series(list(dates)))
    dow, mon = d.dt.dayofweek.values, d.dt.month.values
    return np.column_stack([np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7),
                            np.sin(2 * np.pi * mon / 12), np.cos(2 * np.pi * mon / 12),
                            np.isin(dow, closed).astype(float)])


def lstm_forecast(train: pd.DataFrame, future_dates, target: str = "quantity", window: int = 28,
                  epochs: int = 60, hidden: int = 32, lr: float = 0.01, seed: int = SEED, return_history: bool = False):
    """PyTorch LSTM with known-future calendar covariates, recursive multi-step forecast.
    Falls back to a gradient-boosting lag model when PyTorch is not installed."""
    closed = closed_dows(train, target)
    y = np.log1p(train[target].astype(float).values)
    mu, sd = y.mean(), y.std() + 1e-9
    ys = (y - mu) / sd
    cal = _calendar_feats(train["ds"], closed)
    try:
        import torch
        import torch.nn as nn
        torch.manual_seed(seed)
        np.random.seed(seed)
        feats = np.column_stack([ys, cal])
        Xw, yw, cw = [], [], []
        for t in range(window, len(ys)):
            Xw.append(feats[t - window:t])
            yw.append(ys[t])
            cw.append(cal[t])
        Xw, yw, cw = (torch.tensor(np.array(a), dtype=torch.float32) for a in (Xw, yw, cw))
        n_val = max(int(len(Xw) * 0.12), 20)
        tr_idx, va_idx = slice(0, len(Xw) - n_val), slice(len(Xw) - n_val, len(Xw))

        class Net(nn.Module):
            def __init__(self):
                super().__init__()
                self.lstm = nn.LSTM(6, hidden, batch_first=True)
                self.head = nn.Sequential(nn.Linear(hidden + 5, 16), nn.ReLU(), nn.Linear(16, 1))

            def forward(self, x, c):
                o, _ = self.lstm(x)
                return self.head(torch.cat([o[:, -1, :], c], dim=1)).squeeze(-1)

        net = Net()
        opt = torch.optim.Adam(net.parameters(), lr=lr)
        loss_fn = nn.MSELoss()
        hist, best, best_state, wait = [], 1e9, None, 0
        for ep in range(epochs):
            net.train()
            perm = torch.randperm(len(Xw[tr_idx]))
            tl = 0.0
            for i in range(0, len(perm), 32):
                b = perm[i:i + 32]
                opt.zero_grad()
                loss = loss_fn(net(Xw[tr_idx][b], cw[tr_idx][b]), yw[tr_idx][b])
                loss.backward()
                opt.step()
                tl += loss.item() * len(b)
            net.eval()
            with torch.no_grad():
                vl = loss_fn(net(Xw[va_idx], cw[va_idx]), yw[va_idx]).item()
            hist.append((tl / len(perm), vl))
            if vl < best - 1e-5:
                best, wait = vl, 0
                best_state = {k: v.clone() for k, v in net.state_dict().items()}
            else:
                wait += 1
                if wait >= 12:
                    break
        if best_state:
            net.load_state_dict(best_state)
        net.eval()
        fut_cal = _calendar_feats(future_dates, closed)
        window_feats = feats[-window:].copy()
        preds = []
        with torch.no_grad():
            for i in range(len(fut_cal)):
                x = torch.tensor(window_feats[None, :, :], dtype=torch.float32)
                c = torch.tensor(fut_cal[i][None, :], dtype=torch.float32)
                p = net(x, c).item()
                preds.append(p)
                window_feats = np.vstack([window_feats[1:], np.concatenate([[p], fut_cal[i]])])
        pred = np.expm1(np.array(preds) * sd + mu)
        out = {"pred": _apply_closed(pred, future_dates, closed), "engine": "PyTorch LSTM", "history": hist}
        return out
    except ImportError:
        print("[INFO] PyTorch not installed -> using gradient-boosting lag model instead of LSTM.")
    except Exception as e:  # noqa
        print(f"[INFO] LSTM failed ({type(e).__name__}: {e}); using gradient-boosting lag model.")
    from sklearn.ensemble import HistGradientBoostingRegressor
    lags = [1, 2, 3, 7, 14, 21, 28]
    full = list(ys)
    Xl, yl = [], []
    for t in range(max(lags), len(ys)):
        Xl.append([ys[t - l] for l in lags] + list(cal[t]))
        yl.append(ys[t])
    gb = HistGradientBoostingRegressor(max_iter=250, learning_rate=0.05, random_state=seed).fit(np.array(Xl), np.array(yl))
    fut_cal = _calendar_feats(future_dates, closed)
    preds = []
    for i in range(len(fut_cal)):
        row = [full[-l] for l in lags] + list(fut_cal[i])
        p = float(gb.predict(np.array([row]))[0])
        full.append(p)
        preds.append(p)
    pred = np.expm1(np.array(preds) * sd + mu)
    return {"pred": _apply_closed(pred, future_dates, closed), "engine": "GBM lag model (LSTM fallback)", "history": []}


def make_lag_features(daily: pd.DataFrame, target: str = "quantity") -> pd.DataFrame:
    """Lag, rolling and calendar features for supervised demand models."""
    d = daily[["ds", target]].copy().rename(columns={target: "y"})
    for l in (1, 2, 3, 7, 14, 21, 28):
        d[f"lag_{l}"] = d["y"].shift(l)
    for w in (7, 14, 28):
        d[f"roll_mean_{w}"] = d["y"].shift(1).rolling(w).mean()
        d[f"roll_std_{w}"] = d["y"].shift(1).rolling(w).std()
    d["roll_max_7"] = d["y"].shift(1).rolling(7).max()
    d["ewm_7"] = d["y"].shift(1).ewm(span=7).mean()
    d["expanding_mean"] = d["y"].shift(1).expanding().mean()
    dt = pd.to_datetime(d["ds"])
    d["dow"], d["month"], d["week"] = dt.dt.dayofweek, dt.dt.month, dt.dt.isocalendar().week.astype(int)
    d["day_of_month"], d["is_weekend"] = dt.dt.day, (dt.dt.dayofweek >= 5).astype(int)
    d["is_month_end"] = dt.dt.is_month_end.astype(int)
    return d.dropna().reset_index(drop=True)


def optimize_blend(y_val, p_prophet, p_lstm):
    best_w, best = 0.5, 1e18
    rows = []
    for w in np.linspace(0, 1, 21):
        s = wape(y_val, w * np.asarray(p_prophet) + (1 - w) * np.asarray(p_lstm))
        rows.append((w, s))
        if s < best:
            best, best_w = s, w
    return float(best_w), pd.DataFrame(rows, columns=["w_prophet", "WAPE_%"])


def forecast_bundle(daily: pd.DataFrame, target: str = "quantity", horizon: int | None = None,
                    force: bool = False, lstm_epochs: int = 60) -> dict:
    """Full pipeline: val fit -> blend weight -> test fit -> final future forecast. Cached."""
    h = horizon or CONFIG["FORECAST_HORIZON"]
    key = f"bundle_{target}_{h}_{len(daily)}"
    f = MODELS / f"{key}.pkl"
    if not force and key in _CACHE:
        return _CACHE[key]
    if not force and f.exists():
        try:
            _CACHE[key] = pickle.load(open(f, "rb"))
            return _CACHE[key]
        except Exception:
            pass
    tr, va, te = split_series(daily, h)
    pv = prophet_forecast(tr, va["ds"], target)
    lv = lstm_forecast(tr, va["ds"], target, epochs=lstm_epochs)
    w, blend_curve = optimize_blend(va[target].values, pv["pred"], lv["pred"])
    trv = pd.concat([tr, va])
    pt = prophet_forecast(trv, te["ds"], target, return_model=True)
    lt = lstm_forecast(trv, te["ds"], target, epochs=lstm_epochs)
    hyb = w * pt["pred"] + (1 - w) * lt["pred"]
    nv = naive_forecast(trv, te["ds"], target)
    hw = holt_winters_forecast(trv, te["ds"], target)
    y = te[target].values
    table = pd.DataFrame({
        "Seasonal Naive": forecast_metrics(y, nv), "Holt-Winters": forecast_metrics(y, hw),
        f"Prophet": forecast_metrics(y, pt["pred"]), "LSTM": forecast_metrics(y, lt["pred"]),
        "Hybrid (Prophet+LSTM)": forecast_metrics(y, hyb)}).T
    # final future forecast (fit on everything)
    fut_dates = pd.date_range(daily["ds"].max() + pd.Timedelta(days=1), periods=h, freq="D")
    pf = prophet_forecast(daily, fut_dates, target)
    lf = lstm_forecast(daily, fut_dates, target, epochs=lstm_epochs)
    hf = w * pf["pred"] + (1 - w) * lf["pred"]
    err = rmse(y, hyb)
    future = pd.DataFrame({"ds": fut_dates, "prophet": pf["pred"], "lstm": lf["pred"], "hybrid": hf,
                           "lower": np.clip(hf - 1.28 * err, 0, None), "upper": hf + 1.28 * err})
    bundle = {"target": target, "horizon": h, "weight_prophet": w, "blend_curve": blend_curve,
              "train": tr, "val": va, "test": te, "test_preds": {"Seasonal Naive": nv, "Holt-Winters": hw, "Prophet": pt["pred"],
                                                               "LSTM": lt["pred"], "Hybrid": hyb},
              "prophet_lower": pt["lower"], "prophet_upper": pt["upper"], "prophet_model": None,
              "prophet_forecast_df": pt.get("forecast_df"), "lstm_history": lt.get("history", []),
              "metrics": table, "future": future, "engines": {"prophet": pt["engine"], "lstm": lt["engine"]}}
    try:
        pickle.dump({**bundle, "prophet_model": None}, open(f, "wb"))
    except Exception:
        pass
    _CACHE[key] = bundle
    log_metrics("forecast", {"target": target, "horizon_days": h, "blend_weight_prophet": w,
                             "engines": bundle["engines"], "table": table.round(3).to_dict(orient="index")})
    return bundle


# ----------------------------------------------------------------------------
# 7. CHURN PREDICTION
# ----------------------------------------------------------------------------
def customer_features(hist: pd.DataFrame, snapshot, returns: pd.DataFrame | None = None) -> pd.DataFrame:
    snapshot = pd.Timestamp(snapshot)
    f = rfm_table(hist, snapshot)[["recency", "frequency", "monetary", "n_products", "avg_order_value", "tenure_days"]].copy()
    orders = (hist.groupby(["customer_id", "invoice"])
              .agg(date=("invoice_date", "min"), value=("revenue", "sum"), items=("quantity", "sum"), lines=("stock_code", "count"))
              .reset_index().sort_values(["customer_id", "date"]))
    orders["gap"] = orders.groupby("customer_id")["date"].diff().dt.days
    gap = orders.groupby("customer_id")["gap"].agg(avg_gap_days="mean", std_gap_days="std")
    f = f.join(gap)
    f["single_order"] = (f["frequency"] == 1).astype(int)
    f["avg_gap_days"] = f["avg_gap_days"].fillna(f["avg_gap_days"].median())
    f["std_gap_days"] = f["std_gap_days"].fillna(0)
    f["avg_items_per_order"] = orders.groupby("customer_id")["items"].mean()
    f["avg_lines_per_order"] = orders.groupby("customer_id")["lines"].mean()
    for w in (30, 90):
        o = orders[orders["date"] > snapshot - pd.Timedelta(days=w)]
        f[f"orders_last_{w}d"] = o.groupby("customer_id").size().reindex(f.index).fillna(0)
    prev = orders[(orders["date"] <= snapshot - pd.Timedelta(days=90)) & (orders["date"] > snapshot - pd.Timedelta(days=180))]
    f["orders_prev_90d"] = prev.groupby("customer_id").size().reindex(f.index).fillna(0)
    f["order_trend"] = (f["orders_last_90d"] - f["orders_prev_90d"]) / (f["orders_prev_90d"] + 1)
    r90 = orders[orders["date"] > snapshot - pd.Timedelta(days=90)].groupby("customer_id")["value"].sum()
    f["revenue_last_90d"] = r90.reindex(f.index).fillna(0)
    f["recency_gap_ratio"] = f["recency"] / f["avg_gap_days"].clip(lower=1)
    f["weekend_share"] = hist.groupby("customer_id")["is_weekend"].mean()
    f["avg_hour"] = hist.groupby("customer_id")["hour"].mean()
    f["is_uk"] = (hist.groupby("customer_id")["country"].agg(lambda s: s.iloc[0]) == "United Kingdom").astype(int)
    f["n_categories"] = hist.groupby("customer_id")["category"].nunique()
    if returns is not None and len(returns):
        r = returns[(returns["invoice_date"] < snapshot) & returns["customer_id"].notna()]
        f["n_returns"] = r.groupby("customer_id").size().reindex(f.index).fillna(0)
    else:
        f["n_returns"] = 0
    return f.fillna(0)


def build_churn_dataset(sales: pd.DataFrame, returns: pd.DataFrame | None = None, window_days: int | None = None):
    """Time-based label (no leakage): features from history <= cutoff; churn = no purchase in the next `window_days`."""
    window_days = window_days or CONFIG["CHURN_WINDOW_DAYS"]
    end = sales["invoice_date"].max()
    cutoff = (end - pd.Timedelta(days=window_days)).normalize()
    hist = sales[sales["invoice_date"] < cutoff + pd.Timedelta(days=1)]
    fut = sales[sales["invoice_date"] >= cutoff + pd.Timedelta(days=1)]
    X = customer_features(hist, cutoff + pd.Timedelta(days=1), returns)
    y = (~X.index.isin(fut["customer_id"].unique())).astype(int)
    return X, pd.Series(y, index=X.index, name="churn"), cutoff


def precision_at_k(y, p, frac=0.2):
    y, p = np.asarray(y), np.asarray(p)
    k = max(int(len(y) * frac), 1)
    top = np.argsort(-p)[:k]
    return float(y[top].mean())


def recall_at_k(y, p, frac=0.2):
    y, p = np.asarray(y), np.asarray(p)
    k = max(int(len(y) * frac), 1)
    top = np.argsort(-p)[:k]
    return float(y[top].sum() / max(y.sum(), 1))


def make_xgb(**kw):
    params = dict(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
                  min_child_weight=3, reg_lambda=2.0, eval_metric="auc", random_state=SEED, n_jobs=-1)
    params.update(kw)
    try:
        from xgboost import XGBClassifier
        return XGBClassifier(**params), "XGBoost"
    except Exception:
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED), "HistGradientBoosting (XGBoost fallback)"


def split_churn(X, y, seed: int = SEED):
    from sklearn.model_selection import train_test_split
    Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=seed)
    Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.50, stratify=ytmp, random_state=seed)
    return Xtr, Xva, Xte, ytr, yva, yte


def train_churn_models(X, y):
    """Logistic Regression, Random Forest, XGBoost. Returns dict(models, splits, metrics DataFrame, probs)."""
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
    Xtr, Xva, Xte, ytr, yva, yte = split_churn(X, y)
    pos_w = float((ytr == 0).sum() / max((ytr == 1).sum(), 1))
    xgb, xname = make_xgb(scale_pos_weight=pos_w)
    models = {
        "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
        "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=5, class_weight="balanced_subsample",
                                                random_state=SEED, n_jobs=-1),
        xname: xgb,
    }
    rows, probs, thresholds = {}, {}, {}
    for name, m in models.items():
        m.fit(Xtr, ytr)
        pv, pt = m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
        ths = np.linspace(0.05, 0.95, 91)
        th = float(ths[np.argmax([f1_score(yva, pv >= t) for t in ths])])
        thresholds[name] = th
        probs[name] = pt
        rows[name] = {"AUC-ROC": roc_auc_score(yte, pt), "PR-AUC": average_precision_score(yte, pt),
                      "F1": f1_score(yte, pt >= th), "Precision@top20%": precision_at_k(yte, pt, 0.2),
                      "Recall@top20%": recall_at_k(yte, pt, 0.2), "Brier": brier_score_loss(yte, pt), "threshold": th}
    return {"models": models, "xgb_name": xname, "splits": (Xtr, Xva, Xte, ytr, yva, yte),
            "metrics": pd.DataFrame(rows).T, "probs": probs, "thresholds": thresholds}


def score_all_customers(model, sales: pd.DataFrame, returns: pd.DataFrame | None = None) -> pd.DataFrame:
    snap = sales["invoice_date"].max().normalize() + pd.Timedelta(days=1)
    X = customer_features(sales, snap, returns)
    out = X[["recency", "frequency", "monetary"]].copy()
    out["churn_probability"] = model.predict_proba(X)[:, 1]
    out["risk_band"] = pd.cut(out["churn_probability"], [-0.01, 0.4, 0.7, 1.01], labels=["Low", "Medium", "High"])
    return out.reset_index()


# ----------------------------------------------------------------------------
# 8. INVENTORY OPTIMISATION
# ----------------------------------------------------------------------------
def z_value(service_level: float) -> float:
    from scipy.stats import norm
    return float(norm.ppf(service_level))


def sku_daily_matrix(sales: pd.DataFrame, top_n: int | None = None) -> pd.DataFrame:
    top_n = top_n or CONFIG["TOP_N_SKUS"]
    top = sales.groupby("stock_code")["revenue"].sum().nlargest(top_n).index
    d = sales[sales["stock_code"].isin(top)].groupby(["date", "stock_code"])["quantity"].sum().unstack(fill_value=0)
    idx = pd.date_range(sales["date"].min(), sales["date"].max(), freq="D")
    return d.reindex(idx, fill_value=0)


def inventory_plan(sales: pd.DataFrame, growth_ratio: float = 1.0, lead_time: int | None = None,
                   review: int | None = None, service_level: float | None = None, top_n: int | None = None,
                   order_cost: float | None = None, holding_rate: float | None = None, seed: int = SEED) -> pd.DataFrame:
    """Reorder point / safety stock / EOQ / order-up-to per SKU.
    NOTE: real datasets contain no stock-on-hand column, so current stock is SIMULATED for the demo."""
    L = lead_time or CONFIG["LEAD_TIME_DAYS"]
    R = review or CONFIG["REVIEW_PERIOD_DAYS"]
    sl = service_level or CONFIG["SERVICE_LEVEL"]
    S_cost = order_cost or CONFIG["ORDER_COST"]
    H_rate = holding_rate or CONFIG["HOLDING_RATE"]
    z = z_value(sl)
    mat = sku_daily_matrix(sales, top_n)
    recent = mat.tail(90)
    desc = sales.groupby("stock_code")["description"].agg(lambda s: s.mode().iloc[0])
    price = sales.groupby("stock_code")["unit_price"].median()
    rng = np.random.default_rng(seed)
    rows = []
    for sku in mat.columns:
        mu_h, sd_h = float(recent[sku].mean()), float(recent[sku].std())
        mu_f = mu_h * growth_ratio
        ss = z * sd_h * math.sqrt(L + R)
        rop = mu_f * L + z * sd_h * math.sqrt(L)
        s_up = mu_f * (L + R) + ss
        annual = mu_f * 365
        unit_cost = 0.6 * float(price[sku])
        eoq = math.sqrt(2 * annual * S_cost / max(H_rate * unit_cost, 1e-6))
        stock = float(np.round(rng.uniform(0.15, 4.0) * mu_h * (L + R)))
        cover = stock / max(mu_f, 1e-6)
        order_qty = max(0.0, math.ceil(s_up - stock))
        status = "Reorder Now" if stock < rop else ("Overstock" if cover > 45 else "Healthy")
        rows.append({"stock_code": sku, "description": desc[sku], "unit_price": float(price[sku]),
                     "avg_daily_demand_hist": mu_h, "forecast_daily_demand": mu_f, "demand_std": sd_h,
                     "safety_stock": ss, "reorder_point": rop, "order_up_to": s_up, "EOQ": eoq,
                     "current_stock_simulated": stock, "days_of_cover": cover, "recommended_order_qty": order_qty,
                     "status": status, "order_value": order_qty * unit_cost})
    return pd.DataFrame(rows).sort_values("order_value", ascending=False).reset_index(drop=True)


def _simulate_policy(demand, target_level, L, R, init):
    stock, pipe = float(init), {}
    lost, lost_days, inv_sum, over_days = 0.0, 0, 0.0, 0
    cap = max(float(np.mean(demand)) * 30, 1.0)
    for t, d in enumerate(demand):
        stock += pipe.pop(t, 0.0)
        sold = min(stock, d)
        stock -= sold
        if d > sold:
            lost += d - sold
            lost_days += 1
        inv_sum += stock
        over_days += stock > cap
        if t % R == 0:
            q = target_level - (stock + sum(pipe.values()))
            if q > 0:
                pipe[t + L] = pipe.get(t + L, 0.0) + q
    n = len(demand)
    return {"lost_units": lost, "stockout_days": lost_days, "avg_inventory": inv_sum / n, "overstock_days": over_days,
            "fill_rate": 1 - lost / max(float(np.sum(demand)), 1e-9)}


def inventory_backtest(sales: pd.DataFrame, growth_ratio: float = 1.0, horizon: int | None = None, **kw) -> pd.DataFrame:
    """Back-test on the last `horizon` days: naive buffer rule vs forecast-driven policy (real demand replayed)."""
    h = horizon or CONFIG["FORECAST_HORIZON"]
    L, R = CONFIG["LEAD_TIME_DAYS"], CONFIG["REVIEW_PERIOD_DAYS"]
    z = z_value(CONFIG["SERVICE_LEVEL"])
    mat = sku_daily_matrix(sales, kw.get("top_n"))
    hist, test = mat.iloc[:-h], mat.iloc[-h:]
    rows = []
    for sku in mat.columns:
        mu, sd = float(hist[sku].tail(90).mean()), float(hist[sku].tail(90).std())
        s_naive = mu * (L + R) * 1.25
        s_opt = mu * growth_ratio * (L + R) + z * sd * math.sqrt(L + R)
        dem = test[sku].values.astype(float)
        a = _simulate_policy(dem, s_naive, L, R, s_naive)
        b = _simulate_policy(dem, s_opt, L, R, s_opt)
        price = float(sales.loc[sales["stock_code"] == sku, "unit_price"].median())
        margin, hold = 0.4 * price, 0.6 * price * CONFIG["HOLDING_RATE"] / 365
        row = {"stock_code": sku, "unit_price": price}
        for tag, r_ in (("naive", a), ("opt", b)):
            row.update({f"{tag}_{k}": v for k, v in r_.items()})
            row[f"{tag}_cost"] = r_["lost_units"] * margin + r_["avg_inventory"] * h * hold
        rows.append(row)
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# 9. DRIFT DETECTION & MONITORING
# ----------------------------------------------------------------------------
def psi(ref, cur, bins: int = 10) -> float:
    ref, cur = np.asarray(ref, float), np.asarray(cur, float)
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref)
    c = np.histogram(cur, edges)[0] / len(cur)
    r, c = np.clip(r, 1e-4, None), np.clip(c, 1e-4, None)
    return float(np.sum((c - r) * np.log(c / r)))


def drift_table(ref: pd.DataFrame, cur: pd.DataFrame, cols) -> pd.DataFrame:
    from scipy.stats import ks_2samp
    rows = []
    for c in cols:
        p = psi(ref[c], cur[c])
        ks = ks_2samp(ref[c], cur[c])
        shift = (cur[c].mean() - ref[c].mean()) / (abs(ref[c].mean()) + 1e-9) * 100
        rows.append({"feature": c, "PSI": p, "KS_stat": ks.statistic, "KS_p_value": ks.pvalue, "mean_shift_%": shift,
                     "status": "DRIFT" if p > 0.2 else ("WARNING" if p > 0.1 else "OK")})
    return pd.DataFrame(rows)


def retrain_decision(drift_df: pd.DataFrame, current_mape: float, mape_limit: float = 12.0, psi_limit: float = 0.2) -> dict:
    n_drift = int((drift_df["PSI"] > psi_limit).sum())
    reasons = []
    if n_drift >= 2:
        reasons.append(f"{n_drift} features with PSI > {psi_limit}")
    if current_mape > mape_limit * 2.5:
        reasons.append(f"rolling MAPE {current_mape:.1f}% is far above target {mape_limit}%")
    return {"retrain": bool(reasons), "reasons": reasons, "n_drifted_features": n_drift}


def evidently_report(ref: pd.DataFrame, cur: pd.DataFrame, path: str):
    """Optional Evidently AI HTML report - supports both old and new Evidently APIs; never raises."""
    try:
        try:  # evidently >= 0.7
            from evidently import Report
            from evidently.presets import DataDriftPreset
            snap = Report([DataDriftPreset()]).run(cur, ref)
            snap.save_html(path)
        except ImportError:  # evidently 0.4.x
            from evidently.report import Report
            from evidently.metric_preset import DataDriftPreset
            rep = Report(metrics=[DataDriftPreset()])
            rep.run(reference_data=ref, current_data=cur)
            rep.save_html(path)
        return path
    except Exception as e:  # noqa
        print(f"[INFO] Evidently report skipped ({type(e).__name__}). Native PSI/KS drift report above is the main output.")
        return None


# ----------------------------------------------------------------------------
# 10. MLFLOW (safe wrapper - never crashes the notebook)
# ----------------------------------------------------------------------------
def _mlflow_setup(experiment: str):
    import mlflow
    ROOT.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri(f"sqlite:///{(ROOT / 'mlflow.db').as_posix()}")
    exp = mlflow.get_experiment_by_name(experiment)
    if exp is None:
        mlflow.create_experiment(experiment, artifact_location=(ROOT / "mlartifacts" / experiment.replace(" ", "_")).as_uri())
    mlflow.set_experiment(experiment)
    return mlflow


def mlflow_log(experiment: str, run_name: str, params: dict | None = None, metrics: dict | None = None,
               artifacts: list | None = None, tags: dict | None = None):
    params, metrics = params or {}, metrics or {}
    clean_m = {str(k).replace("%", "pct").replace("@", "_at_").replace(" ", "_"): float(v) for k, v in metrics.items()
               if isinstance(v, (int, float, np.floating, np.integer)) and np.isfinite(v)}
    try:
        mlflow = _mlflow_setup(experiment)
        with mlflow.start_run(run_name=run_name) as run:
            mlflow.log_params({str(k)[:250]: str(v)[:250] for k, v in params.items()})
            mlflow.log_metrics(clean_m)
            if tags:
                mlflow.set_tags({str(k): str(v) for k, v in tags.items()})
            mlflow.set_tag("author", AUTHOR)
            for a in artifacts or []:
                if a and Path(a).exists():
                    mlflow.log_artifact(str(a))
            return run.info.run_id
    except Exception as e:  # noqa
        fb = ROOT / "outputs" / "mlflow_fallback.jsonl"
        fb.parent.mkdir(parents=True, exist_ok=True)
        with open(fb, "a") as fh:
            fh.write(json.dumps({"experiment": experiment, "run": run_name, "params": params, "metrics": clean_m}, default=_json_safe) + "\n")
        print(f"[INFO] MLflow unavailable ({type(e).__name__}); run saved to {fb.name} instead.")
        return None


def mlflow_runs(experiments: list[str] | None = None) -> pd.DataFrame:
    try:
        mlflow = _mlflow_setup("RetailPulse-Week1")
        exps = experiments or [e.name for e in mlflow.search_experiments() if e.name != "Default"]
        df = mlflow.search_runs(experiment_names=exps)
        return df
    except Exception as e:  # noqa
        print(f"[INFO] Could not read MLflow runs ({type(e).__name__}).")
        fb = ROOT / "outputs" / "mlflow_fallback.jsonl"
        if fb.exists():
            return pd.DataFrame([json.loads(l) for l in fb.read_text().splitlines()])
        return pd.DataFrame()


# ----------------------------------------------------------------------------
# 11. PROJECT VISUALS (banner, architecture, workflow) - drawn in code, no external images
# ----------------------------------------------------------------------------
def _gradient(ax, c1, c2, extent):
    g = np.linspace(0, 1, 256).reshape(1, -1)
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list("g", [c1, c2])
    ax.imshow(g, aspect="auto", cmap=cmap, extent=extent, zorder=0)


def draw_banner(show=True):
    fig, ax = plt.subplots(figsize=(13, 3.6))
    ax.set_xlim(0, 13)
    ax.set_ylim(0, 3.6)
    ax.axis("off")
    _gradient(ax, "#0F2744", "#14B8A6", [0, 13, 0, 3.6])
    ax.text(0.5, 2.45, "RetailPulse", fontsize=38, fontweight="bold", color="white", va="center")
    ax.text(0.5, 1.6, "AI-Powered Customer Analytics & Demand Forecasting Platform", fontsize=16, color="#E6FFFA", va="center")
    ax.text(0.5, 0.95, "Demand Forecasting  |  Customer Segmentation  |  Churn Prediction  |  Inventory Optimization",
            fontsize=11.5, color="#CCFBF1", va="center")
    ax.text(0.5, 0.4, f"Made by {AUTHOR}   |   Zidio Development - Data Science & Analytics   |   March 2026",
            fontsize=10.5, color="white", va="center", style="italic")
    xs = np.linspace(9.3, 12.4, 8)
    hs = np.array([0.6, 1.0, 0.8, 1.4, 1.2, 1.8, 1.6, 2.2])
    for x, hgt, c in zip(xs, hs, ["#F59E0B"] * 8):
        ax.add_patch(plt.Rectangle((x, 0.5), 0.28, hgt, color=c, alpha=0.9, zorder=3))
    ax.plot(xs + 0.14, hs + 0.75, color="white", lw=2.5, marker="o", zorder=4)
    return save_fig(fig, "00_project_banner", "RetailPulse project banner", "Project hero image.", "Project Visuals", show)


def _box(ax, x, y, w, h, text, fc="#E0F2FE", ec="#0369A1", fs=9, bold=False):
    from matplotlib.patches import FancyBboxPatch
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08", fc=fc, ec=ec, lw=1.4, zorder=3))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs, fontweight="bold" if bold else "normal", zorder=4)


def draw_architecture(show=True):
    fig, ax = plt.subplots(figsize=(14, 7.2))
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 7.2)
    ax.axis("off")
    ax.text(7, 6.9, "RetailPulse - End-to-End Architecture", ha="center", fontsize=16, fontweight="bold", color="#1F3A5F")
    layers = [
        ("1. DATA LAYER", "#DBEAFE", "#1D4ED8", ["Kaggle / UCI Online Retail II", "CSV / XLSX upload", "ETL + Cleaning", "Data validation (GE-style)"]),
        ("2. FEATURE ENGINEERING", "#DCFCE7", "#15803D", ["RFM scores", "Lag features (1,7,14,28)", "Rolling averages", "Customer behaviour features"]),
        ("3. MODEL LAYER", "#FEF3C7", "#B45309", ["K-Means / DBSCAN", "Prophet + LSTM hybrid", "XGBoost churn + SHAP", "Inventory optimiser", "MLflow tracking"]),
        ("4. SERVING LAYER", "#FCE7F3", "#BE185D", ["Streamlit dashboard", "FastAPI endpoints", "Power BI report", "Docker + Kubernetes"]),
        ("5. MONITORING LAYER", "#EDE9FE", "#6D28D9", ["PSI / KS drift detection", "Evidently AI report", "Rolling MAPE tracking", "Auto-retrain trigger"]),
    ]
    w, gap = 2.4, 0.3
    for i, (title, fc, ec, items) in enumerate(layers):
        x = 0.3 + i * (w + gap)
        _box(ax, x, 5.7, w, 0.65, title, fc=ec, ec=ec, fs=10, bold=True)
        ax.texts[-1].set_color("white")
        for j, it in enumerate(items):
            _box(ax, x, 4.6 - j * 0.95, w, 0.75, it, fc=fc, ec=ec, fs=9)
        if i < len(layers) - 1:
            ax.annotate("", xy=(x + w + gap - 0.03, 6.02), xytext=(x + w + 0.03, 6.02),
                        arrowprops=dict(arrowstyle="-|>", color="#334155", lw=2))
    ax.annotate("", xy=(0.8, 0.45), xytext=(13.2, 0.45), arrowprops=dict(arrowstyle="-|>", color="#6D28D9", lw=2, ls="--"))
    ax.text(7, 0.15, "Feedback loop: drift detected -> retrain -> new model version in MLflow -> redeploy", ha="center", fontsize=10, color="#6D28D9", style="italic")
    return save_fig(fig, "00_architecture_diagram", "RetailPulse architecture diagram",
                    "Five layers: data, feature engineering, models, serving and monitoring with a retraining feedback loop.", "Project Visuals", show)


def draw_workflow(show=True):
    steps = ["Collect dataset", "Import libraries", "Load dataset", "Data cleaning", "Feature engineering", "EDA",
             "RFM segmentation", "Demand forecasting", "Churn prediction", "Inventory optimisation", "Dashboard (Streamlit / Power BI)", "Deploy (FastAPI / Streamlit)"]
    fig, ax = plt.subplots(figsize=(14, 4.6))
    ax.set_xlim(0, 14)
    ax.set_ylim(0, 4.6)
    ax.axis("off")
    ax.text(7, 4.25, "RetailPulse - 12 Step Workflow", ha="center", fontsize=15, fontweight="bold", color="#1F3A5F")
    for i, s in enumerate(steps):
        row, col = divmod(i, 6)
        x, y = 0.35 + col * 2.28, 2.65 - row * 1.75
        _box(ax, x, y, 1.95, 1.0, f"{i + 1}\n{s}", fc=["#DBEAFE", "#DCFCE7", "#FEF3C7"][min(i // 4, 2)], ec=COLORS[i % len(COLORS)], fs=9, bold=True)
        if col < 5:
            ax.annotate("", xy=(x + 2.25, y + 0.5), xytext=(x + 1.97, y + 0.5), arrowprops=dict(arrowstyle="-|>", color="#334155", lw=1.8))
    ax.annotate("", xy=(0.9, 2.55), xytext=(13.0, 3.6), arrowprops=dict(arrowstyle="-|>", color="#94A3B8", lw=1.2, connectionstyle="arc3,rad=-0.15", ls=":"))
    return save_fig(fig, "00_workflow", "RetailPulse 12-step workflow", "Workflow from data collection to deployment.", "Project Visuals", show)


In [ ]:

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
import retailpulse_utils as ru
ru.setup()
print("Author:", ru.AUTHOR, "|", ru.PROJECT)


---
# PART 1 - 01 - Data Collection & Exploratory Data Analysis (EDA)
*Week 1 - Day 1*

## Introduction
**RetailPulse** is an end-to-end retail analytics platform. In this notebook (Day 1) we **collect the dataset** and perform a deep **Exploratory Data Analysis**:
distribution analysis, missing values, seasonality, product / country / customer behaviour and correlation heatmap.

**Why EDA first?** Every later model (segmentation, forecasting, churn, inventory) is only as good as our understanding of the data. EDA tells us what cleaning is needed, which patterns (weekly / yearly seasonality) the forecaster must capture, and where the business revenue really comes from.

**How data is loaded (no zip file needed):**
1. `DATASET="auto"` -> uses a file you uploaded, else downloads from Kaggle with `kagglehub`, else falls back to built-in demo data (so the notebook never fails).
2. Set `UPLOAD_MY_FILE=True` to open the Colab file-picker and upload your own CSV / XLSX.
3. Set `DATA_PATH="/content/drive/MyDrive/yourfile.xlsx"` to read from Google Drive.

In [ ]:
ru.draw_banner()
ru.draw_architecture()
ru.draw_workflow()

### Step 1 - Dataset choice & loading
Supported Kaggle datasets: **Online Retail II (UCI)** `cgrymn/online-retail-ii-uci-dataset` and **Synthetic Retail 10M** `mrayyanshehzad/synthetic-retail-dataset-10-million-transactions` (very large -> automatically sampled by customer so Colab RAM is safe).

In [ ]:
# ---------- USER SETTINGS ----------
DATASET        = "auto"    # "auto" | "uci" | "synthetic" | "upload" | "demo"
UPLOAD_MY_FILE = False     # True -> Colab file picker opens (CSV / XLSX)
DATA_PATH      = None      # e.g. "/content/drive/MyDrive/online_retail_II.xlsx"
MAX_ROWS       = 400_000   # sampling cap for huge files
# -----------------------------------
ru.CONFIG.update(DATASET=DATASET, PATH=DATA_PATH, MAX_ROWS=MAX_ROWS)
if UPLOAD_MY_FILE:
    try:
        from google.colab import files
        up = files.upload()
        for name, content in up.items():
            (ru.DATA_RAW / name).write_bytes(content)
        print("Uploaded:", list(up))
    except ImportError:
        print("Not running in Colab -> copy your file into:", ru.DATA_RAW)

bundle = ru.get_data(force=True)          # load + clean (cached for next notebooks)
sales, returns, rep, raw = bundle["sales"], bundle["returns"], bundle["report"], bundle["raw"]
print("\nDATA SOURCE :", bundle["source"])
print("Raw rows    :", f"{len(raw):,}", "| Clean sales rows:", f"{len(sales):,}", "| Returns:", f"{len(returns):,}")

> **Comment (real dataset):** On the real *Online Retail II (UCI)* file we get ~1.07 million raw rows covering Dec-2009 -> Dec-2011 for a UK online gift-ware wholesaler. Roughly 20-25% rows have no `Customer ID` (guest checkouts) and ~2% are cancellation invoices starting with `C`. The numbers printed above are computed live from *your* loaded data.

In [ ]:
display(raw.head())
print(raw.dtypes)
display(raw.describe(include="all").T.head(12))

### Step 2 - Missing values (raw data)

In [ ]:
miss = (raw.isna().mean() * 100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(x=miss.values, y=miss.index, ax=ax, color=ru.PALETTE["navy"])
for i, v in enumerate(miss.values):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center")
ax.set_xlabel("% missing"); ax.set_title("Missing values by column (raw data)")
ru.save_fig(fig, "01_01_missing_values", "Missing values by column (raw data)",
            f"Highest missing share: {miss.index[0]} ({miss.iloc[0]:.1f}%). Missing customer IDs are guest checkouts and are excluded from customer analytics only.", "EDA")

### Step 3 - Data-quality issues found in raw data

In [ ]:
issues = {
    "Cancelled invoices (C...)": int(raw["invoice"].str.upper().str.startswith("C").sum()),
    "Negative quantity": int((raw["quantity"] < 0).sum()),
    "Zero / negative price": int((raw["unit_price"] <= 0).sum()),
    "Missing customer id": int(raw["customer_id"].isna().sum()),
    "Missing description": int(raw["description"].isna().sum()),
    "Non-product codes (POST, D, M...)": int(raw["stock_code"].astype(str).str.upper().isin(ru._NON_PRODUCT).sum()),
    "Duplicate rows": int(raw.duplicated().sum()),
}
s = pd.Series(issues).sort_values()
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(s.index, s.values, color=ru.COLORS[:len(s)])
for i, v in enumerate(s.values):
    ax.text(v, i, f" {v:,}", va="center")
ax.set_title("Data-quality issues detected in raw data"); ax.set_xlabel("rows affected")
ru.save_fig(fig, "01_02_data_quality_issues", "Data-quality issues detected in raw data",
            "Counts of cancellations, invalid prices, missing IDs and duplicates that the cleaning stage must handle.", "EDA")

### Step 4 - Business KPIs (cleaned sales)

In [ ]:
kpi = {
    "Total revenue": sales["revenue"].sum(), "Orders (invoices)": sales["invoice"].nunique(),
    "Customers": sales["customer_id"].nunique(), "Products (SKUs)": sales["stock_code"].nunique(),
    "Countries": sales["country"].nunique(), "Avg order value": sales.groupby("invoice")["revenue"].sum().mean(),
    "Units sold": sales["quantity"].sum(),
}
for k, v in kpi.items():
    print(f"{k:<20}: {v:>16,.2f}")
ru.log_metrics("eda", {k: float(v) for k, v in kpi.items()})

### Step 5 - Monthly revenue, orders and customers

In [ ]:
m = sales.groupby("year_month").agg(revenue=("revenue", "sum"), orders=("invoice", "nunique"), customers=("customer_id", "nunique")).reset_index()
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(m["year_month"], m["revenue"], color=ru.PALETTE["sky"], alpha=0.85, label="Revenue")
ax.set_ylabel("Revenue"); ax.tick_params(axis="x", rotation=60)
ax2 = ax.twinx()
ax2.plot(m["year_month"], m["orders"], color=ru.PALETTE["orange"], marker="o", lw=2.5, label="Orders")
ax2.set_ylabel("Orders"); ax2.grid(False)
ax.set_title("Monthly revenue (bars) and number of orders (line)")
peak = m.loc[m["revenue"].idxmax()]
ru.save_fig(fig, "01_03_monthly_revenue_orders", "Monthly revenue and orders",
            f"Peak month is {peak['year_month']} with revenue {peak['revenue']:,.0f}. Strong Q4 seasonality is typical for gift retail.", "EDA")

In [ ]:
g = m["revenue"].pct_change() * 100
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(m["year_month"], g, color=np.where(g >= 0, ru.PALETTE["green"], ru.PALETTE["red"]))
ax.axhline(0, color="black", lw=0.8); ax.tick_params(axis="x", rotation=60)
ax.set_ylabel("MoM growth %"); ax.set_title("Month-over-month revenue growth")
ru.save_fig(fig, "01_04_mom_growth", "Month-over-month revenue growth",
            "Green bars show growth, red bars show decline versus the previous month.", "EDA")

### Step 6 - Products and categories

In [ ]:
top_rev = sales.groupby("description")["revenue"].sum().nlargest(10).sort_values()
fig, ax = plt.subplots(figsize=(10, 5)); ax.barh(top_rev.index, top_rev.values, color=ru.PALETTE["teal"])
ax.set_title("Top 10 products by revenue"); ax.set_xlabel("Revenue")
ru.save_fig(fig, "01_05_top_products_revenue", "Top 10 products by revenue", f"Best seller by revenue: {top_rev.index[-1]}.", "EDA")

top_qty = sales.groupby("description")["quantity"].sum().nlargest(10).sort_values()
fig, ax = plt.subplots(figsize=(10, 5)); ax.barh(top_qty.index, top_qty.values, color=ru.PALETTE["purple"])
ax.set_title("Top 10 products by units sold"); ax.set_xlabel("Units")
ru.save_fig(fig, "01_06_top_products_quantity", "Top 10 products by units sold", f"Most units sold: {top_qty.index[-1]}.", "EDA")

In [ ]:
cat = sales.groupby("category")["revenue"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
ax.pie(cat.values, labels=cat.index, autopct="%1.1f%%", startangle=90, colors=ru.COLORS, wedgeprops=dict(width=0.45, edgecolor="white"))
ax.set_title("Revenue share by product category (keyword-inferred)")
ru.save_fig(fig, "01_07_category_share", "Revenue share by category", f"Largest category: {cat.index[0]} ({cat.iloc[0] / cat.sum() * 100:.1f}%).", "EDA")

### Step 7 - Geography

In [ ]:
ctry = sales.groupby("country")["revenue"].sum().sort_values(ascending=False)
share = ctry.iloc[0] / ctry.sum() * 100
rest = ctry.iloc[1:11].sort_values()
fig, ax = plt.subplots(figsize=(10, 5)); ax.barh(rest.index, rest.values, color=ru.PALETTE["orange"])
ax.set_title(f"Top 10 countries by revenue (excluding {ctry.index[0]})"); ax.set_xlabel("Revenue")
ru.save_fig(fig, "01_08_country_revenue", "Revenue by country (excluding the dominant market)",
            f"{ctry.index[0]} contributes {share:.1f}% of revenue; export markets are shown separately.", "EDA")

### Step 8 - Time patterns: day of week, hour of day, heatmap

In [ ]:
names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow = sales.groupby("dow")["revenue"].sum().reindex(range(7), fill_value=0)
fig, ax = plt.subplots(figsize=(9, 4.5)); ax.bar(names, dow.values, color=ru.COLORS[:7])
ax.set_title("Revenue by day of week"); ax.set_ylabel("Revenue")
ru.save_fig(fig, "01_09_revenue_by_dow", "Revenue by day of week", f"Strongest weekday: {names[int(dow.idxmax())]}. Days with zero revenue indicate the shop is closed.", "EDA")

hr = sales.groupby("hour")["invoice"].nunique()
fig, ax = plt.subplots(figsize=(10, 4.5)); ax.plot(hr.index, hr.values, marker="o", color=ru.PALETTE["navy"], lw=2.5)
ax.fill_between(hr.index, hr.values, alpha=0.15, color=ru.PALETTE["navy"]); ax.set_xlabel("Hour of day"); ax.set_ylabel("Orders")
ax.set_title("Orders by hour of day")
ru.save_fig(fig, "01_10_orders_by_hour", "Orders by hour of day", f"Peak ordering hour: {int(hr.idxmax())}:00.", "EDA")

ct = pd.crosstab(sales["dow"], sales["hour"]).reindex(range(7), fill_value=0)
fig, ax = plt.subplots(figsize=(12, 4.5)); sns.heatmap(ct, cmap="YlGnBu", ax=ax, yticklabels=names)
ax.set_title("Order lines heatmap: day of week x hour"); ax.set_xlabel("Hour"); ax.set_ylabel("")
ru.save_fig(fig, "01_11_dow_hour_heatmap", "Day x hour activity heatmap", "Darker cells mark busiest trading windows.", "EDA")

### Step 9 - Distributions & outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
sns.histplot(np.log10(sales["quantity"]), bins=40, ax=axes[0], color=ru.PALETTE["navy"]); axes[0].set_title("log10(Quantity)")
sns.histplot(np.log10(sales["unit_price"]), bins=40, ax=axes[1], color=ru.PALETTE["teal"]); axes[1].set_title("log10(Unit price)")
sns.histplot(np.log10(sales["revenue"]), bins=40, ax=axes[2], color=ru.PALETTE["orange"]); axes[2].set_title("log10(Line revenue)")
ru.save_fig(fig, "01_12_distributions", "Distributions of quantity, price and revenue (log scale)",
            "All three variables are right-skewed, therefore log transforms are used for clustering and modelling.", "EDA")

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=sales, x="category", y="revenue", ax=ax, showfliers=False, palette=ru.COLORS[:sales["category"].nunique()])
ax.tick_params(axis="x", rotation=25); ax.set_title("Line revenue by category (outliers hidden)")
ru.save_fig(fig, "01_13_revenue_box_category", "Line revenue by category (boxplot)", "Shows spread of line values per category.", "EDA")

basket = sales.groupby("invoice")["quantity"].sum()
fig, ax = plt.subplots(figsize=(10, 4.5)); sns.histplot(basket.clip(upper=basket.quantile(0.99)), bins=50, ax=ax, color=ru.PALETTE["purple"])
ax.set_title("Basket size (units per order, clipped at 99th pct)"); ax.set_xlabel("Units per order")
ru.save_fig(fig, "01_14_basket_size", "Basket size distribution", f"Median basket = {basket.median():.0f} units, mean = {basket.mean():.0f} units (wholesale-like behaviour).", "EDA")

### Step 10 - Pareto (80/20) analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
pr = sales.groupby("stock_code")["revenue"].sum().sort_values(ascending=False)
cum = pr.cumsum() / pr.sum() * 100
axes[0].plot(np.arange(1, len(cum) + 1) / len(cum) * 100, cum.values, color=ru.PALETTE["navy"], lw=2.5)
axes[0].axhline(80, color="red", ls="--"); axes[0].set_xlabel("% of products"); axes[0].set_ylabel("Cumulative % revenue"); axes[0].set_title("Product Pareto")
n80 = int((cum <= 80).sum() + 1)
cs = ru.get_customer_sales().groupby("customer_id")["revenue"].sum().sort_values(ascending=False)
cumc = cs.cumsum() / cs.sum() * 100
axes[1].plot(np.arange(1, len(cumc) + 1) / len(cumc) * 100, cumc.values, color=ru.PALETTE["teal"], lw=2.5)
axes[1].axhline(80, color="red", ls="--"); axes[1].set_xlabel("% of customers"); axes[1].set_title("Customer Pareto")
n80c = int((cumc <= 80).sum() + 1)
ru.save_fig(fig, "01_15_pareto", "Pareto analysis of products and customers",
            f"{n80 / len(cum) * 100:.0f}% of products and {n80c / len(cumc) * 100:.0f}% of customers generate 80% of revenue.", "EDA")

### Step 11 - Correlation heatmap

In [ ]:
num = sales[["quantity", "unit_price", "revenue", "hour", "dow", "month", "week", "is_weekend"]]
fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Correlation heatmap of numeric features")
ru.save_fig(fig, "01_16_correlation_heatmap", "Correlation heatmap", "Revenue is driven by quantity; price and quantity are weakly (negatively) related.", "EDA")

### Step 12 - Trend, rolling averages and customer acquisition

In [ ]:
d = sales.groupby("date")["revenue"].sum().asfreq("D", fill_value=0)
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(d.index, d.values, color="#CBD5E1", lw=1, label="Daily revenue")
ax.plot(d.index, d.rolling(7).mean(), color=ru.PALETTE["sky"], lw=2, label="7-day mean")
ax.plot(d.index, d.rolling(30).mean(), color=ru.PALETTE["red"], lw=2.5, label="30-day mean")
ax.legend(); ax.set_title("Daily revenue with rolling averages")
ru.save_fig(fig, "01_17_rolling_revenue", "Daily revenue with 7 and 30-day rolling averages", "Rolling means expose the trend and the seasonal build-up before the holiday season.", "EDA")

cs = ru.get_customer_sales()
first = cs.groupby("customer_id")["invoice_date"].min().dt.to_period("M").astype(str)
tmp = cs[["customer_id", "year_month"]].drop_duplicates()
tmp["is_new"] = tmp["customer_id"].map(first) == tmp["year_month"]
nr = tmp.groupby(["year_month", "is_new"]).size().unstack(fill_value=0).rename(columns={True: "New", False: "Returning"})
fig, ax = plt.subplots(figsize=(12, 4.8)); nr.plot(kind="bar", stacked=True, ax=ax, color=[ru.PALETTE["orange"], ru.PALETTE["navy"]])
ax.set_title("New vs returning customers per month"); ax.set_xlabel("")
ru.save_fig(fig, "01_18_new_vs_returning", "New vs returning customers per month", "Repeat buyers form the backbone of monthly activity.", "EDA")

hm = sales.pivot_table(index="year", columns="month", values="revenue", aggfunc="sum")
fig, ax = plt.subplots(figsize=(11, 3.8)); sns.heatmap(hm / 1000, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax, cbar_kws={"label": "Revenue (thousands)"})
ax.set_title("Revenue heatmap: year x month (thousands)")
ru.save_fig(fig, "01_19_year_month_heatmap", "Year x month revenue heatmap", "Highlights seasonality (autumn / holiday peak).", "EDA")

### Step 13 - Automatic EDA insights

In [ ]:
best_m = sales.groupby("month_name")["revenue"].sum().idxmax()
ins = {
    "peak_month_name": best_m, "top_product": top_rev.index[-1], "top_country_share_pct": float(share),
    "median_basket_units": float(basket.median()), "pct_products_for_80pct_revenue": float(n80 / len(cum) * 100),
}
ru.log_metrics("eda", ins)
print("KEY INSIGHTS")
print(f"1) Revenue peaks in {best_m}; plan stock before that month.")
print(f"2) Top product by revenue: {ins['top_product']}")
print(f"3) {ctry.index[0]} is {share:.1f}% of revenue -> export markets are an upside.")
print(f"4) {ins['pct_products_for_80pct_revenue']:.0f}% of SKUs drive 80% of revenue -> focus inventory on A-class items.")

## Summary of Notebook 01
* Loaded the dataset (Kaggle / upload / demo) and built a **clean sales table** with revenue and calendar features.
* Produced **19 charts**: missing values, data-quality issues, monthly revenue & orders, growth, top products, category share, country revenue, weekday/hour patterns, heatmaps, distributions, Pareto, correlation, rolling trend, new-vs-returning customers.
* **Next -> Notebook 02:** formal cleaning report, validation checks and feature engineering (RFM, lags, rolling statistics).

---
# PART 2 - 02 - Data Cleaning, Validation & Feature Engineering
*Week 1 - Day 2*

## Introduction
Raw retail data is messy: cancellations, zero prices, postage fees, duplicates, guest checkouts and extreme outliers. In this notebook we
1. document **what was removed and why** (cleaning waterfall),
2. run **data-validation expectations** (Great-Expectations-style checks implemented natively so they never break on API changes),
3. engineer features: **RFM**, **lag features**, **rolling statistics** and calendar features,
4. save processed files for the dashboard and **Power BI**.

In [ ]:
bundle = ru.get_data()
sales, returns, rep, raw = bundle["sales"], bundle["returns"], bundle["report"], bundle["raw"]
print("DATA SOURCE :", bundle["source"])
print("Clean rows  :", f"{len(sales):,}", "| Date range:", sales["date"].min().date(), "->", sales["date"].max().date())

### Step 1 - Cleaning waterfall (rows removed at each step)

In [ ]:
steps = [("Raw rows", rep["raw_rows"]), ("Duplicates", -rep["duplicates_removed"]), ("Returns / cancellations", -rep["returns_separated"]),
         ("Invalid qty / price", -rep["invalid_qty_or_price_removed"]), ("Non-product codes", -rep["non_product_codes_removed"]),
         ("Extreme outliers", -rep["extreme_outliers_removed"]), ("Clean rows", rep["final_rows"])]
labels = [s[0] for s in steps]; vals = [s[1] for s in steps]
fig, ax = plt.subplots(figsize=(12, 5)); run = 0
for i, (l, v) in enumerate(steps):
    if i == 0 or i == len(steps) - 1:
        ax.bar(l, v, color=ru.PALETTE["navy"]); run = v if i == 0 else run
    else:
        ax.bar(l, v, bottom=run, color=ru.PALETTE["red"]); run += v
    ax.text(i, (run if 0 < i < len(steps) - 1 else v) + rep["raw_rows"] * 0.01, f"{v:,}", ha="center", fontsize=9)
ax.tick_params(axis="x", rotation=25); ax.set_title("Data cleaning waterfall")
ru.save_fig(fig, "02_01_cleaning_waterfall", "Data cleaning waterfall", "Shows exactly how many rows each cleaning rule removed.", "Cleaning & Feature Engineering")
print(f"Retention: {rep['final_rows'] / rep['raw_rows'] * 100:.1f}% of raw rows kept")
ru.log_metrics("cleaning", {"retention_pct": rep["final_rows"] / rep["raw_rows"] * 100, **rep})

### Step 2 - Before vs after: outlier handling

In [ ]:
raw_pos = raw[(raw["quantity"] > 0) & (raw["unit_price"] > 0)]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].boxplot([np.log10(raw_pos["quantity"]), np.log10(sales["quantity"])], labels=["Before", "After"], patch_artist=True,
                boxprops=dict(facecolor="#BAE6FD")); axes[0].set_title("log10(quantity)")
axes[1].boxplot([np.log10(raw_pos["unit_price"]), np.log10(sales["unit_price"])], labels=["Before", "After"], patch_artist=True,
                boxprops=dict(facecolor="#BBF7D0")); axes[1].set_title("log10(unit price)")
ru.save_fig(fig, "02_02_outliers_before_after", "Outliers before vs after cleaning", f"Max quantity before {raw_pos['quantity'].max():,.0f} -> after {sales['quantity'].max():,.0f}.", "Cleaning & Feature Engineering")

### Step 3 - Data validation (Great-Expectations-style)

In [ ]:
val = ru.validate_data(sales)
display(val)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(val["expectation"], np.ones(len(val)), color=np.where(val["passed"], ru.PALETTE["green"], ru.PALETTE["red"]))
for i, p in enumerate(val["passed"]): ax.text(0.5, i, "PASS" if p else "FAIL", ha="center", va="center", color="white", fontweight="bold")
ax.set_xticks([]); ax.set_title(f"Data validation: {int(val['passed'].sum())}/{len(val)} expectations passed")
ru.save_fig(fig, "02_03_validation_results", "Data validation results", "All checks should pass before the data is used for modelling.", "Cleaning & Feature Engineering")
ru.log_metrics("cleaning", {"validation_passed": int(val["passed"].sum()), "validation_total": int(len(val))})

### Step 4 - Feature engineering: RFM (customer level)
* **Recency** - days since last purchase, **Frequency** - number of orders, **Monetary** - total spend.
* Scores 1-5 via quintiles. These features feed segmentation (03) and churn (09).

In [ ]:
cs = ru.get_customer_sales()
rfm = ru.rfm_scores(ru.rfm_table(cs))
display(rfm.head())
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, c, col in zip(axes, ["recency", "frequency", "monetary"], [ru.PALETTE["red"], ru.PALETTE["teal"], ru.PALETTE["navy"]]):
    sns.histplot(rfm[c].clip(upper=rfm[c].quantile(0.99)), bins=40, ax=ax, color=col); ax.set_title(c.capitalize())
ru.save_fig(fig, "02_04_rfm_distributions", "RFM feature distributions", "Frequency and monetary are heavily right-skewed - log scaling is applied before clustering.", "Cleaning & Feature Engineering")

### Step 5 - Feature engineering: lag features & rolling statistics (demand level)
Lags (1, 2, 3, 7, 14, 21, 28 days), rolling mean / std (7, 14, 28), EWM, expanding mean and calendar flags. All rolling features use `shift(1)` so **no future information leaks** into the features.

In [ ]:
daily = ru.daily_series(sales)
lagf = ru.make_lag_features(daily, "quantity")
display(lagf.head())
print("Feature matrix:", lagf.shape)
corr = lagf.drop(columns=["ds"]).corr()["y"].drop("y").sort_values()
fig, ax = plt.subplots(figsize=(8, 8)); ax.barh(corr.index, corr.values, color=np.where(corr.values >= 0, ru.PALETTE["teal"], ru.PALETTE["red"]))
ax.set_title("Correlation of engineered features with daily demand")
ru.save_fig(fig, "02_05_feature_target_corr", "Feature correlation with demand", "Rolling means and lag-7 carry the strongest signal for next-day demand.", "Cleaning & Feature Engineering")

In [ ]:
cols = ["y", "lag_1", "lag_7", "lag_14", "roll_mean_7", "roll_mean_28", "ewm_7", "expanding_mean"]
fig, ax = plt.subplots(figsize=(8.5, 6.5)); sns.heatmap(lagf[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Lag / rolling feature correlation matrix")
ru.save_fig(fig, "02_06_lag_corr_heatmap", "Lag feature correlation matrix", "Highly correlated rolling features - tree models handle this well, linear models need regularisation.", "Cleaning & Feature Engineering")

fig, ax = plt.subplots(figsize=(13, 4.8))
ax.plot(lagf["ds"], lagf["y"], color="#CBD5E1", label="Daily demand")
ax.plot(lagf["ds"], lagf["roll_mean_7"], color=ru.PALETTE["sky"], label="roll_mean_7 (shifted)")
ax.plot(lagf["ds"], lagf["roll_mean_28"], color=ru.PALETTE["red"], label="roll_mean_28 (shifted)")
ax.legend(); ax.set_title("Raw demand vs rolling-mean features")
ru.save_fig(fig, "02_07_rolling_features", "Raw demand vs rolling-mean features", "Shifted rolling means follow the signal with a lag, which is exactly what a forecaster is allowed to see.", "Cleaning & Feature Engineering")

### Step 6 - Save processed files (dashboard + Power BI)

In [ ]:
keep = ["invoice", "invoice_no", "stock_code", "description", "category", "quantity", "unit_price", "revenue", "order_value", "invoice_date", "customer_id", "country"]
sales[keep].to_csv(ru.DATA_PROC / "transactions_clean.csv.gz", index=False, compression="gzip")
daily.to_csv(ru.DATA_PROC / "daily_series.csv", index=False)
lagf.to_csv(ru.DATA_PROC / "lag_features.csv", index=False)
rfm.reset_index().to_csv(ru.DATA_PROC / "rfm_table.csv", index=False)
pbi = sales[keep + ["year", "month", "month_name", "year_month", "dow", "hour", "week", "quarter"]].copy()
pbi["day_name"] = pbi["invoice_date"].dt.day_name()
pbi.to_csv(ru.DATA_PROC / "powerbi_ready.csv", index=False)
print("Saved:", sorted(p.name for p in ru.DATA_PROC.glob("*.csv*")))

## Summary of Notebook 02
* Cleaning waterfall, before/after outlier handling and **10 validation expectations**.
* Features: **RFM (+scores)**, **7 lags**, **rolling mean/std (7/14/28)**, EWM, expanding mean, calendar flags.
* Saved `transactions_clean.csv.gz`, `daily_series.csv`, `lag_features.csv`, `rfm_table.csv` and `powerbi_ready.csv`.

---
# PART 3 - 03 - Customer Segmentation (RFM, K-Means, DBSCAN, Cohorts)
*Week 1 - Day 3*

## Introduction
Not all customers are equal. We segment customers using **RFM analysis** (rule-based business segments) and **machine-learning clustering** (K-Means and DBSCAN) and then interpret every segment in business language with an action plan.
Acceptance criterion F-02: **6-8 meaningful segments with business interpretation**.

In [ ]:
bundle = ru.get_data()
sales, returns, rep, raw = bundle["sales"], bundle["returns"], bundle["report"], bundle["raw"]
print("DATA SOURCE :", bundle["source"])
print("Clean rows  :", f"{len(sales):,}", "| Date range:", sales["date"].min().date(), "->", sales["date"].max().date())

In [ ]:
cs = ru.get_customer_sales()
rfm = ru.rfm_scores(ru.rfm_table(cs))
print("Customers:", len(rfm))
display(rfm[["recency", "frequency", "monetary", "R", "F", "M", "rule_segment"]].head())

### Step 1 - RFM distributions and scores

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, c, col in zip(axes, ["recency", "frequency", "monetary"], [ru.PALETTE["red"], ru.PALETTE["teal"], ru.PALETTE["navy"]]):
    sns.histplot(np.log1p(rfm[c]), bins=40, ax=ax, color=col, kde=True); ax.set_title(f"log1p({c})")
ru.save_fig(fig, "03_01_rfm_hist", "RFM distributions (log scale)", "Log scaling makes the three RFM variables close to bell-shaped, which suits K-Means.", "Customer Segmentation")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(axes, ["R", "F", "M"]):
    rfm[c].value_counts().sort_index().plot(kind="bar", ax=ax, color=ru.PALETTE["sky"]); ax.set_title(f"{c} score distribution"); ax.set_xlabel("")
ru.save_fig(fig, "03_02_rfm_scores", "RFM score distributions", "Quintile scoring gives roughly equal customer counts per score.", "Customer Segmentation")

### Step 2 - Rule-based RFM segments (Champions, Loyal, At Risk ...)

In [ ]:
seg = rfm.groupby("rule_segment").agg(customers=("monetary", "size"), revenue=("monetary", "sum"), avg_recency=("recency", "mean"),
                                      avg_frequency=("frequency", "mean")).sort_values("revenue", ascending=False)
seg["revenue_share_%"] = seg["revenue"] / seg["revenue"].sum() * 100
display(seg.round(1))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
seg["customers"].sort_values().plot(kind="barh", ax=axes[0], color=ru.PALETTE["teal"]); axes[0].set_title("Customers per RFM segment")
seg["revenue_share_%"].sort_values().plot(kind="barh", ax=axes[1], color=ru.PALETTE["orange"]); axes[1].set_title("Revenue share % per RFM segment")
ru.save_fig(fig, "03_03_rule_segments", "Rule-based RFM segments: customers and revenue share", f"{seg.index[0]} generate the highest revenue share ({seg['revenue_share_%'].iloc[0]:.1f}%).", "Customer Segmentation")

hm = rfm.pivot_table(index="R", columns="F", values="monetary", aggfunc="mean").sort_index(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6)); sns.heatmap(hm, annot=True, fmt=".0f", cmap="YlGn", ax=ax)
ax.set_title("Average monetary value by Recency score (R) x Frequency score (F)")
ru.save_fig(fig, "03_04_rf_heatmap", "Average spend by R x F score", "Recent and frequent buyers (top-right) are the most valuable.", "Customer Segmentation")

### Step 3 - Choosing K for K-Means (elbow, silhouette, Davies-Bouldin)

In [ ]:
ks, X = ru.kmeans_search(rfm)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
axes[0].plot(ks["k"], ks["inertia"], marker="o", color=ru.PALETTE["navy"]); axes[0].set_title("Elbow (inertia)")
axes[1].plot(ks["k"], ks["silhouette"], marker="o", color=ru.PALETTE["teal"]); axes[1].set_title("Silhouette (higher = better)")
axes[2].plot(ks["k"], ks["davies_bouldin"], marker="o", color=ru.PALETTE["red"]); axes[2].set_title("Davies-Bouldin (lower = better)")
for a in axes: a.set_xlabel("k")
ru.save_fig(fig, "03_05_k_selection", "K selection: elbow, silhouette and Davies-Bouldin", "Metrics used to choose the number of clusters.", "Customer Segmentation")
cand = ks[(ks["k"] >= 6) & (ks["k"] <= 8)]
K = int(cand.loc[cand["silhouette"].idxmax(), "k"])
print(f"Chosen K = {K} (best silhouette inside the business range 6-8 required by F-02)")
display(ks.round(3))

### Step 4 - Final K-Means model and PCA visualisation

In [ ]:
from sklearn.decomposition import PCA
seg_df, km, names = ru.fit_kmeans(rfm, X, K)
pc = PCA(n_components=3, random_state=ru.SEED).fit(X); P = pc.transform(X)
fig, ax = plt.subplots(figsize=(10, 7))
for i, (c, n) in enumerate(sorted(names.items())):
    m_ = seg_df["cluster"].values == c; ax.scatter(P[m_, 0], P[m_, 1], s=14, alpha=0.7, label=n, color=ru.COLORS[i % len(ru.COLORS)])
ax.legend(fontsize=8, markerscale=2); ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_title(f"K-Means clusters (K={K}) in PCA space")
ru.save_fig(fig, "03_06_pca_clusters", "K-Means clusters in PCA space", f"PCA explains {pc.explained_variance_ratio_[:2].sum() * 100:.0f}% of RFM variance in 2D.", "Customer Segmentation")

from mpl_toolkits.mplot3d import Axes3D  # noqa
fig = plt.figure(figsize=(9, 7)); ax = fig.add_subplot(111, projection="3d")
ax.scatter(np.log1p(seg_df["recency"]), np.log1p(seg_df["frequency"]), np.log1p(seg_df["monetary"]), c=seg_df["cluster"], cmap="tab10", s=10, alpha=0.7)
ax.set_xlabel("log Recency"); ax.set_ylabel("log Frequency"); ax.set_zlabel("log Monetary"); ax.set_title("3D view of RFM clusters")
ru.save_fig(fig, "03_07_3d_clusters", "3D RFM cluster view", "Each colour is one K-Means cluster in R-F-M space.", "Customer Segmentation")

### Step 5 - Segment profiles and business interpretation

In [ ]:
prof = seg_df.groupby("segment").agg(customers=("monetary", "size"), recency=("recency", "mean"), frequency=("frequency", "mean"),
                                     monetary=("monetary", "mean"), total_revenue=("monetary", "sum")).sort_values("total_revenue", ascending=False)
prof["revenue_share_%"] = prof["total_revenue"] / prof["total_revenue"].sum() * 100
display(prof.round(1))

z = (prof[["recency", "frequency", "monetary"]] - prof[["recency", "frequency", "monetary"]].mean()) / prof[["recency", "frequency", "monetary"]].std().replace(0, 1)
fig, ax = plt.subplots(figsize=(8, 0.7 * len(z) + 2)); sns.heatmap(z, annot=prof[["recency", "frequency", "monetary"]].round(0), fmt="", cmap="RdYlGn_r", center=0, ax=ax, cbar_kws={"label": "z-score"})
ax.set_title("Cluster profile heatmap (values = cluster means)"); ax.set_ylabel("")
ru.save_fig(fig, "03_08_cluster_profile_heatmap", "Cluster profile heatmap", "Low recency + high frequency + high monetary = best customers.", "Customer Segmentation")

# Radar chart
cats = ["recency", "frequency", "monetary"]
norm = (prof[cats] - prof[cats].min()) / (prof[cats].max() - prof[cats].min()).replace(0, 1)
norm["recency"] = 1 - norm["recency"]       # recent = good
ang = np.linspace(0, 2 * np.pi, len(cats), endpoint=False).tolist(); ang += ang[:1]
fig = plt.figure(figsize=(8, 8)); ax = plt.subplot(111, polar=True)
for i, (n, r) in enumerate(norm.iterrows()):
    v = r.tolist() + r.tolist()[:1]; ax.plot(ang, v, label=n, color=ru.COLORS[i % len(ru.COLORS)], lw=2); ax.fill(ang, v, alpha=0.08, color=ru.COLORS[i % len(ru.COLORS)])
ax.set_xticks(ang[:-1]); ax.set_xticklabels(["Recency (recent=good)", "Frequency", "Monetary"]); ax.legend(loc="upper right", bbox_to_anchor=(1.45, 1.1), fontsize=8)
ax.set_title("Segment radar chart")
ru.save_fig(fig, "03_09_segment_radar", "Segment radar chart", "Normalised R, F and M of each K-Means segment.", "Customer Segmentation")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
prof["customers"].sort_values().plot(kind="barh", ax=axes[0], color=ru.PALETTE["purple"]); axes[0].set_title("Customers per segment")
prof["revenue_share_%"].sort_values().plot(kind="barh", ax=axes[1], color=ru.PALETTE["green"]); axes[1].set_title("Revenue share % per segment")
ru.save_fig(fig, "03_10_segment_size_revenue", "Segment size and revenue share", "A few high-value segments contribute most revenue.", "Customer Segmentation")

In [ ]:
ACTIONS = {
    "Champions": "Reward, early access to new products, ask for reviews/referrals.",
    "Loyal Regulars": "Upsell bundles, loyalty points, keep communication frequent.",
    "Big Spenders": "Premium / bulk offers, personal account manager.",
    "New / Promising": "Welcome series, second-purchase incentive.",
    "Can't Lose Them": "Urgent win-back with personal outreach and special discount.",
    "At-Risk Regulars": "Reminder emails, 'we miss you' offer.",
    "Dormant Big Spenders": "High-value reactivation campaign.",
    "Hibernating / Lost": "Low-cost reactivation or sunset from marketing spend.",
}
def action_for(name):
    base = name.split(" - ")[0]
    return ACTIONS.get(base, "Review manually.")
interp = prof.reset_index()[["segment", "customers", "recency", "frequency", "monetary", "revenue_share_%"]].round(1)
interp["recommended_action"] = interp["segment"].map(action_for)
display(interp)
interp.to_csv(ru.DATA_PROC / "segment_interpretation.csv", index=False)

### Step 6 - DBSCAN (density-based) comparison

In [ ]:
labels, eps, kd = ru.fit_dbscan(X)
from sklearn.metrics import silhouette_score
ncl = len(set(labels)) - (1 if -1 in labels else 0); noise = float((labels == -1).mean() * 100)
sil = silhouette_score(X[labels != -1], labels[labels != -1]) if ncl > 1 else float("nan")
print(f"DBSCAN eps={eps:.3f} -> clusters={ncl}, noise={noise:.1f}%, silhouette(excl. noise)={sil:.3f}")
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(kd, color=ru.PALETTE["navy"]); axes[0].axhline(eps, color="red", ls="--", label=f"eps={eps:.2f}"); axes[0].legend(); axes[0].set_title("k-distance plot"); axes[0].set_xlabel("points sorted")
sc = axes[1].scatter(P[:, 0], P[:, 1], c=labels, cmap="tab10", s=12); axes[1].set_title(f"DBSCAN clusters (noise = -1) | {ncl} clusters, {noise:.1f}% noise")
ru.save_fig(fig, "03_11_dbscan", "DBSCAN k-distance and clusters", "DBSCAN isolates outlier customers (noise) that K-Means would force into a cluster.", "Customer Segmentation")
cmp = pd.DataFrame({"K-Means": {"clusters": K, "silhouette": float(ks.loc[ks['k'] == K, 'silhouette'].iloc[0]), "noise_%": 0.0},
                    "DBSCAN": {"clusters": ncl, "silhouette": sil, "noise_%": noise}}).T
display(cmp.round(3))

### Step 7 - Cohort retention analysis

In [ ]:
coh = ru.cohort_matrix(cs)
fig, ax = plt.subplots(figsize=(13, 8)); sns.heatmap(coh.iloc[:, :13], annot=True, fmt=".0f", cmap="Blues", ax=ax, vmin=0, vmax=60, cbar_kws={"label": "% customers active"})
ax.set_title("Monthly cohort retention (% of cohort buying again)"); ax.set_xlabel("Months since first purchase"); ax.set_ylabel("Cohort")
ru.save_fig(fig, "03_12_cohort_retention", "Cohort retention heatmap", "Shows how quickly each acquisition cohort decays - input for churn definition.", "Customer Segmentation")

In [ ]:
out = seg_df.reset_index().merge(rfm[["R", "F", "M", "rule_segment"]].reset_index(), on="customer_id", suffixes=("", "_r"))
out[["customer_id", "recency", "frequency", "monetary", "R", "F", "M", "rule_segment", "cluster", "segment"]].to_csv(ru.DATA_PROC / "rfm_segments.csv", index=False)
ru.log_metrics("segmentation", {"customers": int(len(rfm)), "k_chosen": K, "silhouette_kmeans": float(ks.loc[ks['k'] == K, 'silhouette'].iloc[0]),
                                "dbscan_clusters": int(ncl), "dbscan_noise_pct": noise, "segments": prof["customers"].to_dict(),
                                "revenue_share_pct": prof["revenue_share_%"].round(2).to_dict()})
print("Saved rfm_segments.csv")

## Summary of Notebook 03
* RFM scoring + **10 rule-based segments**; K-Means with K chosen by silhouette inside 6-8; DBSCAN for outliers; cohort retention.
* Business interpretation + recommended marketing actions per segment.
* **12 charts.** Output `rfm_segments.csv` is used by churn analysis and the Streamlit Customer dashboard.

---
# PART 4 - 04 - Time-Series Preparation, Stationarity & Decomposition
*Week 1 - Day 4*

## Introduction
Forecasting needs a clean, regular time series. We build the **daily demand series**, check it for **stationarity** (ADF and KPSS tests), **decompose** trend / seasonality / residual, inspect **autocorrelation (ACF / PACF)** and define a leakage-free **train / validation / test** split (30-day horizon, F-03).

In [ ]:
bundle = ru.get_data()
sales, returns, rep, raw = bundle["sales"], bundle["returns"], bundle["report"], bundle["raw"]
print("DATA SOURCE :", bundle["source"])
print("Clean rows  :", f"{len(sales):,}", "| Date range:", sales["date"].min().date(), "->", sales["date"].max().date())

In [ ]:
daily = ru.daily_series(sales)
TARGET = "quantity"
print(daily.shape, "| zero-demand days:", int((daily[TARGET] == 0).sum()))
display(daily.tail())
daily.to_csv(ru.DATA_PROC / "daily_series.csv", index=False)

### Step 1 - Daily demand, revenue and orders

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
for ax, c, col in zip(axes, ["quantity", "revenue", "orders"], [ru.PALETTE["navy"], ru.PALETTE["teal"], ru.PALETTE["orange"]]):
    ax.plot(daily["ds"], daily[c], color=col, lw=1); ax.set_ylabel(c)
axes[0].set_title("Daily units, revenue and orders")
ru.save_fig(fig, "04_01_daily_series", "Daily units, revenue and orders", "All three series share the same seasonal rhythm.", "Time-Series Preparation")

dd = daily.copy(); dd["dow"] = dd["ds"].dt.dayofweek
zero = dd.groupby("dow")[TARGET].apply(lambda s: (s == 0).mean() * 100)
fig, ax = plt.subplots(figsize=(8, 4)); ax.bar(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], zero.reindex(range(7), fill_value=0).values, color=ru.PALETTE["red"])
ax.set_ylabel("% days with zero demand"); ax.set_title("Zero-demand days by weekday (closed days)")
ru.save_fig(fig, "04_02_zero_days", "Zero-demand days by weekday", f"Closed weekdays detected: {ru.closed_dows(daily, TARGET)}. These are forced to zero in forecasts.", "Time-Series Preparation")

### Step 2 - Rolling statistics and seasonality views

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(daily["ds"], daily[TARGET], color="#CBD5E1"); axes[0].plot(daily["ds"], daily[TARGET].rolling(28).mean(), color=ru.PALETTE["red"], lw=2); axes[0].set_title("Rolling mean (28d)")
axes[1].plot(daily["ds"], daily[TARGET].rolling(28).std(), color=ru.PALETTE["purple"], lw=2); axes[1].set_title("Rolling std (28d) - variance changes -> non-stationary")
ru.save_fig(fig, "04_03_rolling_stats", "Rolling mean and rolling std", "A moving mean and moving variance indicate non-stationarity.", "Time-Series Preparation")

dd["month"] = dd["ds"].dt.month; dd["year"] = dd["ds"].dt.year
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
sns.boxplot(data=dd, x="dow", y=TARGET, ax=axes[0], palette=ru.COLORS); axes[0].set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]); axes[0].set_title("Weekly seasonality")
mm = dd.groupby(["year", "month"])[TARGET].sum().unstack(0)
mm.plot(ax=axes[1], marker="o"); axes[1].set_title("Yearly seasonality (monthly units)"); axes[1].set_xlabel("month")
ru.save_fig(fig, "04_04_seasonality", "Weekly and yearly seasonality", "Clear weekday pattern plus a strong autumn / holiday peak.", "Time-Series Preparation")

### Step 3 - Decomposition (trend / seasonal / residual)

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose, STL
s = daily.set_index("ds")[TARGET]
dec = seasonal_decompose(s, model="additive", period=7)
fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
for ax, (n, v, col) in zip(axes, [("Observed", dec.observed, "navy"), ("Trend", dec.trend, "red"), ("Weekly seasonal", dec.seasonal, "teal"), ("Residual", dec.resid, "grey")]):
    ax.plot(v, color=ru.PALETTE[col], lw=1.2); ax.set_ylabel(n)
axes[0].set_title("Classical decomposition (period = 7)")
ru.save_fig(fig, "04_05_decomposition", "Classical decomposition (weekly)", "Separates trend, weekly seasonality and noise.", "Time-Series Preparation")

stl = STL(np.log1p(s), period=7, robust=True).fit()
fig = stl.plot(); fig.set_size_inches(13, 9); fig.suptitle("STL decomposition of log1p(demand)", y=1.01)
ru.save_fig(fig, "04_06_stl", "STL decomposition (robust)", "Robust STL handles the zero (closed) days better than classical decomposition.", "Time-Series Preparation")

### Step 4 - Stationarity tests (ADF & KPSS) and differencing

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
def tests(x, name):
    a = adfuller(x.dropna(), autolag="AIC"); k = kpss(x.dropna(), regression="c", nlags="auto")
    return {"series": name, "ADF stat": a[0], "ADF p": a[1], "KPSS stat": k[0], "KPSS p": k[1],
            "verdict": "Stationary" if (a[1] < 0.05 and k[1] >= 0.05) else ("Trend/seasonal non-stationary" if a[1] >= 0.05 else "Mixed - difference recommended")}
res = pd.DataFrame([tests(s, "original"), tests(s.diff(1), "diff(1)"), tests(s.diff(7), "diff(7) seasonal")])
display(res.round(4))
ru.log_metrics("timeseries", {"adf_p_original": float(res.loc[0, "ADF p"]), "adf_p_diff1": float(res.loc[1, "ADF p"]), "adf_p_diff7": float(res.loc[2, "ADF p"])})
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
for ax, (n, v) in zip(axes, [("Original", s), ("First difference", s.diff(1)), ("Seasonal difference (7)", s.diff(7))]):
    ax.plot(v, lw=0.9, color=ru.PALETTE["navy"]); ax.set_ylabel(n)
axes[0].set_title("Original vs differenced series")
ru.save_fig(fig, "04_07_differencing", "Original vs differenced series", "Differencing removes trend / weekly seasonality and stabilises the mean.", "Time-Series Preparation")

### Step 5 - ACF / PACF and lag plot

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
plot_acf(s, lags=42, ax=axes[0]); plot_pacf(s, lags=42, ax=axes[1], method="ywm")
axes[0].set_title("ACF - spikes every 7 lags = weekly seasonality"); axes[1].set_title("PACF")
ru.save_fig(fig, "04_08_acf_pacf", "ACF and PACF", "Significant lags at 1, 7, 14 justify the lag features (1, 7, 14, 28).", "Time-Series Preparation")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(s.shift(1), s, s=10, alpha=0.5, color=ru.PALETTE["teal"]); axes[0].set_title("y(t) vs y(t-1)")
axes[1].scatter(s.shift(7), s, s=10, alpha=0.5, color=ru.PALETTE["orange"]); axes[1].set_title("y(t) vs y(t-7)")
ru.save_fig(fig, "04_09_lag_plots", "Lag scatter plots", "The y(t) vs y(t-7) relation is tighter than the 1-day lag.", "Time-Series Preparation")

### Step 6 - Outlier days and train / validation / test split

In [ ]:
z = (s - s.rolling(28, min_periods=7).mean()) / s.rolling(28, min_periods=7).std()
out_days = s[z.abs() > 3]
fig, ax = plt.subplots(figsize=(13, 4.5)); ax.plot(s, color="#94A3B8", lw=1); ax.scatter(out_days.index, out_days.values, color=ru.PALETTE["red"], zorder=3, label=f"|z|>3 ({len(out_days)} days)")
ax.legend(); ax.set_title("Anomalous demand days (rolling z-score)")
ru.save_fig(fig, "04_10_outlier_days", "Anomalous demand days", "Spikes usually correspond to big wholesale orders; they are kept but flagged.", "Time-Series Preparation")

tr, va, te = ru.split_series(daily)
fig, ax = plt.subplots(figsize=(13, 4.5))
for d_, c, n in [(tr, ru.PALETTE["navy"], "Train"), (va, ru.PALETTE["orange"], "Validation (30d)"), (te, ru.PALETTE["red"], "Test (30d)")]:
    ax.plot(d_["ds"], d_[TARGET], color=c, label=f"{n}: {len(d_)} days")
ax.legend(); ax.set_title("Chronological split (no shuffling -> no leakage)")
ru.save_fig(fig, "04_11_train_val_test", "Train / validation / test split", "Validation tunes ensemble weights; test is touched only once for the final score.", "Time-Series Preparation")

## Summary of Notebook 04
* Built a **continuous daily series**, detected **closed weekdays**, decomposed (classical + STL), ran **ADF/KPSS**, ACF/PACF, differencing and outlier detection.
* **11 charts** and a leakage-free **30-day train/validation/test split**.
* **Next -> Notebook 05:** baseline Prophet model.

---
# PART 5 - 05 - Baseline Demand Forecasting with Prophet
*Week 1 - Day 5*

## Introduction
**Prophet** (Meta) is an additive / multiplicative time-series model: `y(t) = trend(t) x seasonality(t) x holidays(t) + noise`.
It handles weekly and yearly seasonality, holidays and changepoints out-of-the-box, which makes it a strong **baseline** for retail demand.

In this notebook we fit Prophet on train + validation, forecast the **next 30 days (test window)**, and compare it with two simple baselines (Seasonal Naive, Holt-Winters).

**Metrics used:** MAPE (daily), MAPE on weekly totals, WAPE, sMAPE, RMSE, MAE and Bias.
> Daily MAPE is naturally high for lumpy wholesale data, so we also report weekly-total MAPE which is what a replenishment planner actually uses.

In [ ]:
bundle = ru.get_data()
sales, returns, rep = bundle["sales"], bundle["returns"], bundle["report"]
print("DATA SOURCE :", bundle["source"], "|", f"{len(sales):,} clean rows")
daily = ru.daily_series(sales)
TARGET = "quantity"                       # demand = units sold per day
H = ru.CONFIG["FORECAST_HORIZON"]         # 30-day horizon (F-03)
tr, va, te = ru.split_series(daily, H)
trv = pd.concat([tr, va])                 # train + validation, used to score the final test window
print(f"Train {len(tr)} d | Validation {len(va)} d | Test {len(te)} d  ({te['ds'].min().date()} -> {te['ds'].max().date()})")

### Step 1 - Fit Prophet and forecast the test window

In [ ]:
res = ru.prophet_forecast(trv, te["ds"], TARGET, return_model=True)
y = te[TARGET].values
print("Engine:", res["engine"])
met = ru.forecast_metrics(y, res["pred"])
display(pd.Series(met, name="Prophet").round(2).to_frame())

In [ ]:
hist = trv.tail(90)
fig, ax = plt.subplots(figsize=(13, 5.2))
ax.plot(hist["ds"], hist[TARGET], color="#94A3B8", label="History (train+val)")
ax.plot(te["ds"], y, color=ru.PALETTE["navy"], lw=2.5, marker="o", ms=4, label="Actual (test)")
ax.plot(te["ds"], res["pred"], color=ru.PALETTE["orange"], lw=2.5, marker="s", ms=4, label=f"{res['engine']} forecast")
ax.fill_between(te["ds"], res["lower"], res["upper"], color=ru.PALETTE["orange"], alpha=0.18, label="80% interval")
ax.axvline(te["ds"].min(), color="grey", ls="--"); ax.legend(); ax.set_title("Prophet: 30-day forecast vs actual demand (units)")
ru.save_fig(fig, "05_01_prophet_forecast", "Prophet 30-day forecast vs actual", f"Daily MAPE {met['MAPE_daily_%']:.1f}%, weekly-total MAPE {met['MAPE_weekly_%']:.1f}%, WAPE {met['WAPE_%']:.1f}%.", "Forecasting - Prophet")

### Step 2 - Model components (trend, weekly, yearly, holidays)

In [ ]:
m = res.get("model")
if m is not None:
    full = m.predict(m.make_future_dataframe(periods=H))
    fig = m.plot_components(full); fig.set_size_inches(11, 11)
    ru.save_fig(fig, "05_02_prophet_components", "Prophet components", "Trend, weekly seasonality, yearly seasonality and UK holiday effects learned by the model.", "Forecasting - Prophet")

    fig, ax = plt.subplots(figsize=(13, 4.8))
    ax.plot(full["ds"], full["trend"], color=ru.PALETTE["red"], lw=2.5, label="Trend")
    ax.scatter(m.history["ds"], m.history["y"], s=4, color="#CBD5E1", label="Observed")
    for cp in m.changepoints: ax.axvline(cp, color=ru.PALETTE["purple"], alpha=0.15)
    ax.legend(); ax.set_title("Prophet trend and automatically detected changepoints")
    ru.save_fig(fig, "05_03_prophet_changepoints", "Prophet trend and changepoints", "Vertical lines = candidate changepoints where the growth rate may change.", "Forecasting - Prophet")
else:
    print("Prophet model object not available (fallback engine used) - component plots skipped.")

### Step 3 - Residual diagnostics

In [ ]:
from scipy import stats
resid = y - res["pred"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].bar(te["ds"], resid, color=np.where(resid >= 0, ru.PALETTE["green"], ru.PALETTE["red"])); axes[0].set_title("Residuals over time (actual - forecast)"); axes[0].tick_params(axis="x", rotation=45)
sns.histplot(resid, bins=15, kde=True, ax=axes[1], color=ru.PALETTE["navy"]); axes[1].set_title("Residual distribution")
stats.probplot(resid, dist="norm", plot=axes[2]); axes[2].set_title("Q-Q plot")
ru.save_fig(fig, "05_04_prophet_residuals", "Prophet residual diagnostics", f"Mean residual (bias) = {resid.mean():.1f} units; residuals should be centred near zero.", "Forecasting - Prophet")

### Step 4 - Baselines comparison

In [ ]:
nv = ru.naive_forecast(trv, te["ds"], TARGET)
hw = ru.holt_winters_forecast(trv, te["ds"], TARGET)
comp = pd.DataFrame({"Seasonal Naive": ru.forecast_metrics(y, nv), "Holt-Winters": ru.forecast_metrics(y, hw), "Prophet": met}).T
display(comp.round(2))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, c in zip(axes, ["MAPE_daily_%", "MAPE_weekly_%", "WAPE_%"]):
    comp[c].plot(kind="bar", ax=ax, color=ru.COLORS[:3]); ax.set_title(c); ax.tick_params(axis="x", rotation=20)
    for i, v in enumerate(comp[c]): ax.text(i, v, f"{v:.1f}", ha="center", va="bottom")
ru.save_fig(fig, "05_05_baseline_comparison", "Prophet vs simple baselines", "Any serious model must beat Seasonal Naive; weekly-total MAPE is the planning metric.", "Forecasting - Prophet")

In [ ]:
dfp = pd.DataFrame({"ds": te["ds"].values, "actual": y, "pred": res["pred"]}); dfp["dow"] = pd.to_datetime(dfp["ds"]).dt.dayofweek
bd = dfp.groupby("dow")[["actual", "pred"]].mean().reindex(range(7)).fillna(0)
fig, ax = plt.subplots(figsize=(10, 4.5)); x = np.arange(7); ax.bar(x - 0.2, bd["actual"], 0.4, label="Actual", color=ru.PALETTE["navy"]); ax.bar(x + 0.2, bd["pred"], 0.4, label="Prophet", color=ru.PALETTE["orange"])
ax.set_xticks(x); ax.set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]); ax.legend(); ax.set_title("Average actual vs forecast demand by weekday (test window)")
ru.save_fig(fig, "05_06_prophet_by_weekday", "Forecast accuracy by weekday", "Checks that the weekly pattern (including closed days) is captured.", "Forecasting - Prophet")

### Step 5 - Hyper-parameter sensitivity (changepoint_prior_scale)

In [ ]:
rows = []
for cps in [0.01, 0.05, 0.2, 0.5]:
    r = ru.prophet_forecast(tr, va["ds"], TARGET, changepoint_prior_scale=cps)
    rows.append({"changepoint_prior_scale": cps, "val_WAPE_%": ru.wape(va[TARGET].values, r["pred"]), "val_MAPE_weekly_%": ru.block_mape(va[TARGET].values, r["pred"])})
sens = pd.DataFrame(rows); display(sens.round(2))
fig, ax = plt.subplots(figsize=(8, 4.5)); ax.plot(sens["changepoint_prior_scale"].astype(str), sens["val_WAPE_%"], marker="o", color=ru.PALETTE["purple"], lw=2.5)
ax.set_xlabel("changepoint_prior_scale"); ax.set_ylabel("Validation WAPE %"); ax.set_title("Prophet sensitivity to trend flexibility")
ru.save_fig(fig, "05_07_prophet_sensitivity", "Prophet hyper-parameter sensitivity", "Too small = under-fits trend changes, too large = over-fits noise.", "Forecasting - Prophet")

In [ ]:
target_ok = met["MAPE_weekly_%"] <= 12
print(f"F-03 acceptance (MAPE <= 12% on weekly totals): {'PASS' if target_ok else 'NOT MET'}  -> {met['MAPE_weekly_%']:.1f}%")
print(f"Daily MAPE (strict, lumpy data): {met['MAPE_daily_%']:.1f}%")
ru.log_metrics("prophet", {**met, "engine": res["engine"], "target_weekly_mape_met": bool(target_ok)})

## Summary of Notebook 05
* Prophet baseline with UK holidays, weekly + yearly seasonality, closed-day handling.
* Components, changepoints, residual diagnostics, baseline comparison, weekday accuracy, hyper-parameter sensitivity (**7 charts**).
* Be honest in reporting: daily MAPE is high on lumpy wholesale data; weekly-total MAPE is the planning metric. **Next -> Notebook 06:** LSTM.

---
# PART 6 - 06 - Deep Learning Forecasting with LSTM (PyTorch)
*Week 1 - Day 6*

## Introduction
An **LSTM** (Long Short-Term Memory) network learns non-linear temporal patterns from the last 28 days of demand. Our design:
* Input window: 28 days of `[log-demand, sin/cos weekday, sin/cos month, closed-day flag]`.
* **Known-future covariates** (weekday / month / closed flag of the day being predicted) are concatenated to the LSTM output.
* Training: Adam, MSE, early stopping on a chronological validation slice, **recursive 30-day forecast**.
* Implemented in plain PyTorch (stable on every Colab runtime); if PyTorch is missing the code automatically falls back to a gradient-boosting lag model so the notebook never fails.

In [ ]:
bundle = ru.get_data()
sales, returns, rep = bundle["sales"], bundle["returns"], bundle["report"]
print("DATA SOURCE :", bundle["source"], "|", f"{len(sales):,} clean rows")
daily = ru.daily_series(sales)
TARGET = "quantity"                       # demand = units sold per day
H = ru.CONFIG["FORECAST_HORIZON"]         # 30-day horizon (F-03)
tr, va, te = ru.split_series(daily, H)
trv = pd.concat([tr, va])                 # train + validation, used to score the final test window
print(f"Train {len(tr)} d | Validation {len(va)} d | Test {len(te)} d  ({te['ds'].min().date()} -> {te['ds'].max().date()})")

### Step 1 - Network architecture

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.8)); ax.set_xlim(0, 13); ax.set_ylim(0, 3.8); ax.axis("off")
ru._box(ax, 0.2, 1.4, 2.2, 1.0, "Input window\n28 days x 6 features", "#DBEAFE", "#1D4ED8", 9, True)
ru._box(ax, 3.0, 1.4, 2.2, 1.0, "LSTM\n(hidden = 32)", "#FEF3C7", "#B45309", 9, True)
ru._box(ax, 5.8, 1.4, 2.0, 1.0, "Last hidden\nstate (32)", "#FCE7F3", "#BE185D", 9, True)
ru._box(ax, 5.8, 0.15, 2.0, 0.85, "Future calendar\nfeatures (5)", "#DCFCE7", "#15803D", 9, True)
ru._box(ax, 8.4, 1.4, 2.0, 1.0, "Dense 16 + ReLU", "#EDE9FE", "#6D28D9", 9, True)
ru._box(ax, 11.0, 1.4, 1.8, 1.0, "Dense 1\nlog-demand t+1", "#FEE2E2", "#B91C1C", 9, True)
for a, b in [(2.4, 3.0), (5.2, 5.8), (7.8, 8.4), (10.4, 11.0)]:
    ax.annotate("", xy=(b, 1.9), xytext=(a, 1.9), arrowprops=dict(arrowstyle="-|>", lw=2, color="#334155"))
ax.annotate("", xy=(8.4, 1.55), xytext=(7.8, 0.6), arrowprops=dict(arrowstyle="-|>", lw=2, color="#15803D"))
ax.text(6.5, 3.3, "Recursive multi-step forecasting: each prediction is fed back into the window", ha="center", fontsize=11, style="italic")
ax.set_title("LSTM architecture used in RetailPulse")
ru.save_fig(fig, "06_01_lstm_architecture", "LSTM architecture", "LSTM encoder + known-future calendar covariates + dense head, applied recursively for 30 days.", "Forecasting - LSTM")

### Step 2 - Train and forecast

In [ ]:
res = ru.lstm_forecast(trv, te["ds"], TARGET, window=28, epochs=60, hidden=32)
y = te[TARGET].values
met = ru.forecast_metrics(y, res["pred"])
print("Engine:", res["engine"], "| epochs run:", len(res["history"]))
display(pd.Series(met, name="LSTM").round(2).to_frame())

In [ ]:
h = res["history"]
if h:
    h = np.array(h)
    fig, ax = plt.subplots(figsize=(9, 4.8)); ax.plot(h[:, 0], label="Train loss", color=ru.PALETTE["navy"], lw=2.5); ax.plot(h[:, 1], label="Validation loss", color=ru.PALETTE["red"], lw=2.5)
    ax.axvline(int(np.argmin(h[:, 1])), color="grey", ls="--", label="best epoch"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE (scaled)"); ax.legend(); ax.set_title("LSTM training curves (early stopping)")
    ru.save_fig(fig, "06_02_lstm_training_curve", "LSTM training and validation loss", "Validation loss stops improving -> early stopping restores the best weights.", "Forecasting - LSTM")
else:
    print("No training curve (fallback model in use).")

hist = trv.tail(90)
fig, ax = plt.subplots(figsize=(13, 5.2)); ax.plot(hist["ds"], hist[TARGET], color="#94A3B8", label="History")
ax.plot(te["ds"], y, color=ru.PALETTE["navy"], lw=2.5, marker="o", ms=4, label="Actual"); ax.plot(te["ds"], res["pred"], color=ru.PALETTE["teal"], lw=2.5, marker="s", ms=4, label=res["engine"])
ax.axvline(te["ds"].min(), color="grey", ls="--"); ax.legend(); ax.set_title("LSTM: 30-day recursive forecast vs actual")
ru.save_fig(fig, "06_03_lstm_forecast", "LSTM 30-day forecast vs actual", f"Daily MAPE {met['MAPE_daily_%']:.1f}%, weekly MAPE {met['MAPE_weekly_%']:.1f}%, WAPE {met['WAPE_%']:.1f}%.", "Forecasting - LSTM")

In [ ]:
from sklearn.metrics import r2_score
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
axes[0].scatter(y, res["pred"], color=ru.PALETTE["purple"], s=30); lim = [0, max(y.max(), res["pred"].max()) * 1.05]; axes[0].plot(lim, lim, "k--"); axes[0].set_xlabel("Actual"); axes[0].set_ylabel("Predicted")
axes[0].set_title(f"Actual vs predicted (R2 = {r2_score(y, res['pred']):.2f})")
resid = y - res["pred"]
axes[1].bar(te["ds"], resid, color=np.where(resid >= 0, ru.PALETTE["green"], ru.PALETTE["red"])); axes[1].set_title("Residuals over time"); axes[1].tick_params(axis="x", rotation=45)
sns.histplot(resid, bins=15, kde=True, ax=axes[2], color=ru.PALETTE["navy"]); axes[2].set_title("Residual distribution")
ru.save_fig(fig, "06_04_lstm_diagnostics", "LSTM diagnostics: scatter and residuals", "Points close to the diagonal and centred residuals indicate a well-calibrated model.", "Forecasting - LSTM")

### Step 3 - Sensitivity to the input window length

In [ ]:
rows = []
for w in [14, 28, 56]:
    r = ru.lstm_forecast(tr, va["ds"], TARGET, window=w, epochs=30)
    rows.append({"window_days": w, "val_WAPE_%": ru.wape(va[TARGET].values, r["pred"]), "val_MAPE_weekly_%": ru.block_mape(va[TARGET].values, r["pred"])})
sens = pd.DataFrame(rows); display(sens.round(2))
fig, ax = plt.subplots(figsize=(8, 4.5)); ax.bar(sens["window_days"].astype(str), sens["val_WAPE_%"], color=ru.COLORS[:3])
for i, v in enumerate(sens["val_WAPE_%"]): ax.text(i, v, f"{v:.1f}", ha="center", va="bottom")
ax.set_xlabel("Input window (days)"); ax.set_ylabel("Validation WAPE %"); ax.set_title("LSTM window-length sensitivity")
ru.save_fig(fig, "06_05_lstm_window_sensitivity", "LSTM window-length sensitivity", "Chooses the look-back window using validation data only.", "Forecasting - LSTM")

### Step 4 - LSTM vs Prophet

In [ ]:
pr = ru.prophet_forecast(trv, te["ds"], TARGET)
cmp_ = pd.DataFrame({"Prophet": ru.forecast_metrics(y, pr["pred"]), "LSTM": met}).T; display(cmp_.round(2))
fig, ax = plt.subplots(figsize=(13, 5)); ax.plot(te["ds"], y, color=ru.PALETTE["navy"], lw=3, label="Actual")
ax.plot(te["ds"], pr["pred"], color=ru.PALETTE["orange"], lw=2, label="Prophet"); ax.plot(te["ds"], res["pred"], color=ru.PALETTE["teal"], lw=2, label="LSTM")
ax.legend(); ax.set_title("Prophet vs LSTM on the test window")
ru.save_fig(fig, "06_06_prophet_vs_lstm", "Prophet vs LSTM", "The two models make different errors - which is exactly why an ensemble helps (Notebook 08).", "Forecasting - LSTM")
ru.log_metrics("lstm", {**met, "engine": res["engine"]})

## Summary of Notebook 06
* LSTM with calendar covariates, early stopping, recursive 30-day forecast, window-length study, comparison to Prophet (**6 charts**).
* **Next -> Notebook 07:** log baseline experiments in MLflow.

---
# PART 7 - 07 - MLflow Experiment Tracking (Week 1 Baselines)
*Week 1 - Day 7 (checkpoint)*

## Introduction
**MLflow** gives us reproducibility: every run stores parameters, metrics, tags and artifacts (charts, CSVs). In the Week-1 checkpoint we log all baseline models:
data cleaning, Seasonal Naive, Holt-Winters, Prophet, LSTM and K-Means segmentation into the experiment **RetailPulse-Week1** (local SQLite backend, no paid service or key needed).

In [ ]:
bundle = ru.get_data()
sales, returns, rep = bundle["sales"], bundle["returns"], bundle["report"]
print("DATA SOURCE :", bundle["source"], "|", f"{len(sales):,} clean rows")
daily = ru.daily_series(sales)
TARGET = "quantity"                       # demand = units sold per day
H = ru.CONFIG["FORECAST_HORIZON"]         # 30-day horizon (F-03)
tr, va, te = ru.split_series(daily, H)
trv = pd.concat([tr, va])                 # train + validation, used to score the final test window
print(f"Train {len(tr)} d | Validation {len(va)} d | Test {len(te)} d  ({te['ds'].min().date()} -> {te['ds'].max().date()})")

In [ ]:
y = te[TARGET].values
runs = []
def log(name, params, metrics, artifacts=None):
    rid = ru.mlflow_log("RetailPulse-Week1", name, params, metrics, artifacts, tags={"stage": "baseline"})
    runs.append({"run": name, **{k: v for k, v in metrics.items()}}); print(f"logged {name:<22} run_id={str(rid)[:8]}")

log("data_cleaning", {"source": bundle["source"], "dedupe": True, "outlier_pct": 99.95},
    {"raw_rows": rep["raw_rows"], "clean_rows": rep["final_rows"], "retention_pct": rep["final_rows"] / rep["raw_rows"] * 100})
nv = ru.naive_forecast(trv, te["ds"], TARGET)
log("seasonal_naive", {"weeks": 4}, ru.forecast_metrics(y, nv))
hw = ru.holt_winters_forecast(trv, te["ds"], TARGET)
log("holt_winters", {"trend": "damped additive", "seasonal_periods": 7}, ru.forecast_metrics(y, hw))
pr = ru.prophet_forecast(trv, te["ds"], TARGET)
log("prophet_baseline", {"engine": pr["engine"], "changepoint_prior_scale": 0.05, "seasonality_mode": "multiplicative"}, ru.forecast_metrics(y, pr["pred"]))
ls = ru.lstm_forecast(trv, te["ds"], TARGET, epochs=40)
log("lstm_baseline", {"engine": ls["engine"], "window": 28, "hidden": 32}, ru.forecast_metrics(y, ls["pred"]))
cs = ru.get_customer_sales(); rfm = ru.rfm_scores(ru.rfm_table(cs)); ks, X = ru.kmeans_search(rfm, range(4, 9))
best = ks.loc[ks["silhouette"].idxmax()]
log("kmeans_rfm", {"k_range": "4-8", "best_k": int(best["k"])}, {"silhouette": float(best["silhouette"]), "davies_bouldin": float(best["davies_bouldin"])})

### Compare the logged runs

In [ ]:
tbl = pd.DataFrame(runs).set_index("run"); display(tbl.round(2))
fc = tbl.dropna(subset=["MAPE_weekly_%"])
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
fc["MAPE_weekly_%"].plot(kind="bar", ax=axes[0], color=ru.COLORS[:len(fc)]); axes[0].set_title("MLflow runs: weekly-total MAPE %"); axes[0].tick_params(axis="x", rotation=25)
fc["WAPE_%"].plot(kind="bar", ax=axes[1], color=ru.COLORS[:len(fc)]); axes[1].set_title("MLflow runs: WAPE %"); axes[1].tick_params(axis="x", rotation=25)
ru.save_fig(fig, "07_01_mlflow_forecast_runs", "MLflow: baseline forecasting runs", "All baseline runs are tracked and comparable in one place.", "MLOps - MLflow")

fig, ax = plt.subplots(figsize=(8, 4.2)); ax.bar(ks["k"].astype(str), ks["silhouette"], color=ru.PALETTE["teal"]); ax.set_xlabel("k"); ax.set_ylabel("silhouette"); ax.set_title("Silhouette by k (K-Means run)")
ru.save_fig(fig, "07_02_mlflow_kmeans", "K-Means silhouette scores logged", "Segmentation experiment logged with the same tooling as forecasting.", "MLOps - MLflow")

In [ ]:
df_runs = ru.mlflow_runs(["RetailPulse-Week1"])
cols = [c for c in df_runs.columns if c in ("tags.mlflow.runName", "run", "status", "params.engine") or c.startswith("metrics.MAPE_weekly")]
print("Runs stored in MLflow:", len(df_runs))
display(df_runs[cols].head(10) if cols else df_runs.head(10))
print("\nTo open the MLflow UI on your own PC:  mlflow ui --backend-store-uri sqlite:///retailpulse_work/mlflow.db")
ru.log_metrics("mlflow", {"week1_runs": int(len(df_runs))})

## Summary of Notebook 07
* 6 baseline runs logged (cleaning, Seasonal Naive, Holt-Winters, Prophet, LSTM, K-Means) with params, metrics and tags.
* **Week-1 checkpoint reached:** EDA, cleaned dataset and baseline models logged in MLflow.

---
# PART 8 - 08 - Hybrid Forecasting Ensemble (Prophet + LSTM)
*Week 2 - Day 8*

## Introduction
Prophet captures trend + calendar effects; LSTM captures non-linear short-term dynamics. A **weighted ensemble** combines both:
`Hybrid(t) = w x Prophet(t) + (1 - w) x LSTM(t)`.
The weight `w` is chosen on the **validation** window (minimising WAPE) and then both models are re-fitted on train + validation to score the **test** window - no leakage.
Finally the hybrid is refit on all data to produce the **30-day-ahead production forecast** used by inventory optimisation and the dashboard.

In [ ]:
bundle = ru.get_data()
sales, returns, rep = bundle["sales"], bundle["returns"], bundle["report"]
print("DATA SOURCE :", bundle["source"], "|", f"{len(sales):,} clean rows")
daily = ru.daily_series(sales)
TARGET = "quantity"                       # demand = units sold per day
H = ru.CONFIG["FORECAST_HORIZON"]         # 30-day horizon (F-03)
tr, va, te = ru.split_series(daily, H)
trv = pd.concat([tr, va])                 # train + validation, used to score the final test window
print(f"Train {len(tr)} d | Validation {len(va)} d | Test {len(te)} d  ({te['ds'].min().date()} -> {te['ds'].max().date()})")

In [ ]:
fb = ru.forecast_bundle(daily, TARGET, H)
print("Engines:", fb["engines"], "| Prophet weight w =", round(fb["weight_prophet"], 2))
display(fb["metrics"].round(2))
y = fb["test"][TARGET].values

### Step 1 - Blend weight search

In [ ]:
bc = fb["blend_curve"]
fig, ax = plt.subplots(figsize=(9, 4.8)); ax.plot(bc["w_prophet"], bc["WAPE_%"], color=ru.PALETTE["purple"], lw=2.5, marker="o")
ax.axvline(fb["weight_prophet"], color="red", ls="--", label=f"best w = {fb['weight_prophet']:.2f}"); ax.legend()
ax.set_xlabel("Weight on Prophet (1 - w on LSTM)"); ax.set_ylabel("Validation WAPE %"); ax.set_title("Ensemble weight optimisation (validation window)")
ru.save_fig(fig, "08_01_blend_weight", "Ensemble weight search", f"Best Prophet weight = {fb['weight_prophet']:.2f}.", "Forecasting - Hybrid")

### Step 2 - Test-window forecasts of all models

In [ ]:
te = fb["test"]; P = fb["test_preds"]
fig, ax = plt.subplots(figsize=(13, 5.5)); ax.plot(te["ds"], y, color=ru.PALETTE["navy"], lw=3.2, label="Actual")
for (n, v), c in zip(P.items(), [ "#94A3B8", "#CBD5E1", ru.PALETTE["orange"], ru.PALETTE["teal"], ru.PALETTE["red"]]):
    ax.plot(te["ds"], v, lw=2.8 if n == "Hybrid" else 1.4, ls="-" if n == "Hybrid" else "--", color=c, label=n)
ax.legend(ncol=3); ax.set_title("All models vs actual - 30-day test window")
ru.save_fig(fig, "08_02_all_models_test", "All models vs actual (test window)", "The hybrid line combines Prophet and LSTM.", "Forecasting - Hybrid")

met = fb["metrics"]
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, c in zip(axes, ["MAPE_daily_%", "MAPE_weekly_%", "WAPE_%"]):
    met[c].plot(kind="bar", ax=ax, color=ru.COLORS[:len(met)]); ax.set_title(c); ax.tick_params(axis="x", rotation=30)
    for i, v in enumerate(met[c]): ax.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=9)
ru.save_fig(fig, "08_03_model_comparison", "Model comparison on test window", "Lower is better. The best model by weekly MAPE is highlighted in the table above.", "Forecasting - Hybrid")

In [ ]:
errs = pd.DataFrame({n: y - v for n, v in P.items()})
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
sns.boxplot(data=errs, ax=axes[0], palette=ru.COLORS); axes[0].axhline(0, color="black", lw=0.8); axes[0].set_title("Residual distribution per model"); axes[0].tick_params(axis="x", rotation=25)
errs.abs().cumsum().set_index(te["ds"].values).plot(ax=axes[1], color=ru.COLORS[:5]); axes[1].set_title("Cumulative absolute error")
ru.save_fig(fig, "08_04_residuals_cumulative", "Residuals and cumulative error per model", "Hybrid should show a smaller, tighter error distribution than its parts.", "Forecasting - Hybrid")

k = (len(y) // 7) * 7
wk = pd.DataFrame({"week": [f"Week {i + 1}" for i in range(k // 7)], "Actual": y[:k].reshape(-1, 7).sum(1), "Hybrid": P["Hybrid"][:k].reshape(-1, 7).sum(1)})
fig, ax = plt.subplots(figsize=(9, 4.8)); x = np.arange(len(wk)); ax.bar(x - 0.2, wk["Actual"], 0.4, label="Actual", color=ru.PALETTE["navy"]); ax.bar(x + 0.2, wk["Hybrid"], 0.4, label="Hybrid", color=ru.PALETTE["teal"])
ax.set_xticks(x); ax.set_xticklabels(wk["week"]); ax.legend(); ax.set_title("Weekly demand totals: actual vs hybrid forecast")
ru.save_fig(fig, "08_05_weekly_totals", "Weekly totals: actual vs hybrid forecast", f"Weekly-total MAPE of hybrid = {met.loc['Hybrid (Prophet+LSTM)', 'MAPE_weekly_%']:.1f}%.", "Forecasting - Hybrid")

### Step 3 - Production forecast: next 30 days beyond the last date

In [ ]:
fu = fb["future"]; hist = daily.tail(75)
fig, ax = plt.subplots(figsize=(13, 5.2)); ax.plot(hist["ds"], hist[TARGET], color=ru.PALETTE["navy"], lw=1.8, label="History")
ax.plot(fu["ds"], fu["hybrid"], color=ru.PALETTE["red"], lw=2.8, marker="o", ms=4, label="Hybrid forecast"); ax.fill_between(fu["ds"], fu["lower"], fu["upper"], color=ru.PALETTE["red"], alpha=0.15, label="80% interval")
ax.axvline(fu["ds"].min(), color="grey", ls="--"); ax.legend(); ax.set_title(f"30-day-ahead demand forecast ({fu['ds'].min().date()} -> {fu['ds'].max().date()})")
ru.save_fig(fig, "08_06_future_forecast", "30-day-ahead production forecast", f"Expected total demand next 30 days: {fu['hybrid'].sum():,.0f} units.", "Forecasting - Hybrid")
fu.to_csv(ru.DATA_PROC / "forecast_future.csv", index=False)
pd.DataFrame({"ds": te["ds"].values, "actual": y, **{k_: v for k_, v in P.items()}}).to_csv(ru.DATA_PROC / "forecast_test.csv", index=False)
hy = met.loc["Hybrid (Prophet+LSTM)"]
ok = hy["MAPE_weekly_%"] <= 12
print(f"F-03 target MAPE <= 12% (weekly totals): {'PASS' if ok else 'NOT MET'} -> {hy['MAPE_weekly_%']:.1f}% | daily MAPE {hy['MAPE_daily_%']:.1f}% | WAPE {hy['WAPE_%']:.1f}%")
ru.log_metrics("hybrid", {**hy.to_dict(), "weight_prophet": fb["weight_prophet"], "weekly_target_met": bool(ok), "future_30d_units": float(fu["hybrid"].sum())})

## Summary of Notebook 08
* Validation-tuned Prophet + LSTM ensemble, leakage-free test scoring, residual analysis and the **30-day production forecast**.
* Saved `forecast_future.csv` and `forecast_test.csv` (inputs for inventory + dashboard). **6 charts.**

---
# PART 9 - 09 - Churn Prediction with XGBoost + SHAP Explainability
*Week 2 - Day 9*

## Introduction
**Churn** = a customer who bought before but makes **no purchase in the next 90 days**. Predicting churn early lets marketing launch win-back campaigns.

**Leakage-free design:** features are computed only from history *before* a cut-off date (last 90 days are held out); the label looks at the 90 days *after* the cut-off.
**Models:** Logistic Regression (baseline), Random Forest, **XGBoost** (main). **Explainability:** SHAP.
**Acceptance (F-04):** AUC-ROC >= 0.88 and precision@top-20% >= 0.75.

In [ ]:
bundle = ru.get_data()
sales, returns, rep = bundle["sales"], bundle["returns"], bundle["report"]
print("DATA SOURCE :", bundle["source"], "|", f"{len(sales):,} clean rows")
daily = ru.daily_series(sales); TARGET = "quantity"; H = ru.CONFIG["FORECAST_HORIZON"]

In [ ]:
cs = ru.get_customer_sales()
X, y, cutoff = ru.build_churn_dataset(cs, returns)
res = ru.train_churn_models(X, y)
Xtr, Xva, Xte, ytr, yva, yte = res["splits"]
XN = res["xgb_name"]; xgb = res["models"][XN]
print(f"Customers: {len(X)} | features: {X.shape[1]} | churn rate: {y.mean() * 100:.1f}% | label window: last {ru.CONFIG['CHURN_WINDOW_DAYS']} days (cutoff {cutoff.date()})")

### Step 1 - Label balance and feature behaviour

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5)); vc = y.value_counts().sort_index(); ax.bar(["Active (0)", "Churned (1)"], vc.values, color=[ru.PALETTE["green"], ru.PALETTE["red"]])
for i, v in enumerate(vc.values): ax.text(i, v, f"{v} ({v / len(y) * 100:.0f}%)", ha="center", va="bottom")
ax.set_title("Churn label distribution")
ru.save_fig(fig, "09_01_label_distribution", "Churn label distribution", f"{y.mean() * 100:.1f}% of customers churned in the 90-day window.", "Churn Prediction")

feats = ["recency", "frequency", "monetary", "avg_gap_days", "orders_last_90d", "recency_gap_ratio"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, f in zip(axes.ravel(), feats):
    for lab, col in [(0, ru.PALETTE["green"]), (1, ru.PALETTE["red"])]:
        v = X.loc[y == lab, f]; sns.kdeplot(np.log1p(v.clip(lower=0)), ax=ax, color=col, fill=True, alpha=0.3, label="Churned" if lab else "Active")
    ax.set_title(f"log1p({f})"); ax.legend()
ru.save_fig(fig, "09_02_feature_by_churn", "Key features: churned vs active", "Churners have higher recency and lower recent activity.", "Churn Prediction")

top = X.assign(churn=y).corr()["churn"].drop("churn").abs().sort_values(ascending=False).head(14).index.tolist() + ["churn"]
fig, ax = plt.subplots(figsize=(10, 8)); sns.heatmap(X.assign(churn=y)[top].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, annot_kws={"size": 7})
ax.set_title("Correlation heatmap (top features vs churn)")
ru.save_fig(fig, "09_03_churn_corr", "Churn feature correlation heatmap", "Recency-type features correlate most with churn.", "Churn Prediction")

### Step 2 - Model comparison

In [ ]:
mt = res["metrics"]; display(mt.round(3))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, c in zip(axes, ["AUC-ROC", "PR-AUC", "F1"]):
    mt[c].plot(kind="bar", ax=ax, color=ru.COLORS[:3]); ax.set_title(c); ax.set_ylim(0, 1.05); ax.tick_params(axis="x", rotation=20)
    for i, v in enumerate(mt[c]): ax.text(i, v + 0.01, f"{v:.3f}", ha="center")
ru.save_fig(fig, "09_04_model_comparison", "Churn model comparison", "AUC-ROC, PR-AUC and F1 on the held-out test set.", "Churn Prediction")

from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix
fig, ax = plt.subplots(figsize=(7.5, 6))
for (n, p), c in zip(res["probs"].items(), ru.COLORS): fpr, tpr, _ = roc_curve(yte, p); ax.plot(fpr, tpr, lw=2.5, color=c, label=f"{n} (AUC {mt.loc[n, 'AUC-ROC']:.3f})")
ax.plot([0, 1], [0, 1], "k--"); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.legend(); ax.set_title("ROC curves")
ru.save_fig(fig, "09_05_roc_curves", "ROC curves", "Higher and further left is better.", "Churn Prediction")
fig, ax = plt.subplots(figsize=(7.5, 6))
for (n, p), c in zip(res["probs"].items(), ru.COLORS): pr, rc, _ = precision_recall_curve(yte, p); ax.plot(rc, pr, lw=2.5, color=c, label=n)
ax.axhline(yte.mean(), color="grey", ls="--", label="baseline"); ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.legend(); ax.set_title("Precision-Recall curves")
ru.save_fig(fig, "09_06_pr_curves", "Precision-Recall curves", "Useful for imbalanced classes.", "Churn Prediction")

In [ ]:
p_x = res["probs"][XN]; th = res["thresholds"][XN]
cm = confusion_matrix(yte, p_x >= th)
fig, ax = plt.subplots(figsize=(5.8, 5)); sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, xticklabels=["Pred active", "Pred churn"], yticklabels=["Actual active", "Actual churn"])
ax.set_title(f"Confusion matrix ({XN}, threshold {th:.2f})")
ru.save_fig(fig, "09_07_confusion_matrix", "Confusion matrix", "Threshold chosen on the validation set to maximise F1.", "Churn Prediction")

order = np.argsort(-p_x); cumg = np.cumsum(np.asarray(yte)[order]) / max(np.sum(yte), 1); pct = np.arange(1, len(yte) + 1) / len(yte)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(pct * 100, cumg * 100, color=ru.PALETTE["red"], lw=2.5, label="Model"); axes[0].plot([0, 100], [0, 100], "k--", label="Random"); axes[0].set_xlabel("% customers contacted (highest risk first)"); axes[0].set_ylabel("% churners captured"); axes[0].legend(); axes[0].set_title("Cumulative gain curve")
axes[1].plot(pct * 100, cumg / pct, color=ru.PALETTE["purple"], lw=2.5); axes[1].axhline(1, color="k", ls="--"); axes[1].set_xlabel("% customers contacted"); axes[1].set_ylabel("Lift"); axes[1].set_title("Lift curve")
ru.save_fig(fig, "09_08_gain_lift", "Gain and lift curves", f"Contacting the top 20% riskiest customers captures {ru.recall_at_k(yte, p_x, 0.2) * 100:.0f}% of churners.", "Churn Prediction")

from sklearn.calibration import calibration_curve
fig, ax = plt.subplots(figsize=(6.5, 6))
for (n, p), c in zip(res["probs"].items(), ru.COLORS):
    fr, mp = calibration_curve(yte, p, n_bins=8, strategy="quantile"); ax.plot(mp, fr, marker="o", lw=2, color=c, label=n)
ax.plot([0, 1], [0, 1], "k--"); ax.set_xlabel("Predicted probability"); ax.set_ylabel("Observed churn rate"); ax.legend(); ax.set_title("Calibration (reliability) curves")
ru.save_fig(fig, "09_09_calibration", "Calibration curves", "Shows whether predicted probabilities can be trusted as real probabilities.", "Churn Prediction")

from sklearn.metrics import f1_score
ths = np.linspace(0.05, 0.95, 91); f1s = [f1_score(yva, res["models"][XN].predict_proba(Xva)[:, 1] >= t) for t in ths]
fig, ax = plt.subplots(figsize=(8, 4.5)); ax.plot(ths, f1s, color=ru.PALETTE["teal"], lw=2.5); ax.axvline(th, color="red", ls="--", label=f"best = {th:.2f}"); ax.legend(); ax.set_xlabel("Threshold"); ax.set_ylabel("F1 (validation)"); ax.set_title("Threshold selection")
ru.save_fig(fig, "09_10_threshold_f1", "Threshold selection curve", "Decision threshold is tuned on validation data only.", "Churn Prediction")

### Step 3 - Explainability: feature importance and SHAP

In [ ]:
imp = pd.Series(getattr(xgb, "feature_importances_", np.zeros(X.shape[1])), index=X.columns).sort_values().tail(15)
fig, ax = plt.subplots(figsize=(9, 6.5)); ax.barh(imp.index, imp.values, color=ru.PALETTE["navy"]); ax.set_title(f"{XN} feature importance (top 15)")
ru.save_fig(fig, "09_11_feature_importance", "Churn model feature importance", f"Most important feature: {imp.index[-1]}.", "Churn Prediction")

try:
    import shap
    explainer = shap.TreeExplainer(xgb); sv = explainer.shap_values(Xte)
    sv = sv[1] if isinstance(sv, list) else sv
    plt.figure(figsize=(10, 7)); shap.summary_plot(sv, Xte, show=False, max_display=15); plt.title("SHAP summary (impact on churn probability)")
    ru.save_fig(plt.gcf(), "09_12_shap_summary", "SHAP summary plot", "Red = high feature value. Right = pushes the customer towards churn.", "Churn Prediction")
    plt.figure(figsize=(9, 6)); shap.summary_plot(sv, Xte, plot_type="bar", show=False, max_display=15); plt.title("SHAP mean |value|")
    ru.save_fig(plt.gcf(), "09_13_shap_bar", "SHAP global importance", "Average absolute SHAP value per feature.", "Churn Prediction")
    i = int(np.argmax(p_x)); ev = explainer.expected_value; ev = float(np.ravel(ev)[0])
    ex = shap.Explanation(values=sv[i], base_values=ev, data=Xte.iloc[i].values, feature_names=list(Xte.columns))
    plt.figure(); shap.plots.waterfall(ex, show=False, max_display=12); plt.gcf().set_size_inches(9, 6)
    ru.save_fig(plt.gcf(), "09_14_shap_waterfall", "SHAP explanation of the highest-risk customer", "Shows why one specific customer is flagged - useful for the retention team.", "Churn Prediction")
except Exception as e:
    print(f"[INFO] SHAP plots skipped ({type(e).__name__}: {str(e)[:80]})")

### Step 4 - Score all customers and attach segments

In [ ]:
scores = ru.score_all_customers(xgb, cs, returns)
fig, ax = plt.subplots(figsize=(9, 4.8)); sns.histplot(scores["churn_probability"], bins=40, ax=ax, color=ru.PALETTE["red"]); ax.set_title("Distribution of churn probability (all customers)")
ru.save_fig(fig, "09_15_prob_distribution", "Churn probability distribution", f"High-risk customers (p > 0.7): {(scores['churn_probability'] > 0.7).sum()}.", "Churn Prediction")

rfm = ru.rfm_scores(ru.rfm_table(cs)).reset_index()[["customer_id", "rule_segment"]]
sc2 = scores.merge(rfm, on="customer_id", how="left")
seg = sc2.groupby("rule_segment")["churn_probability"].mean().sort_values()
fig, ax = plt.subplots(figsize=(9, 5)); ax.barh(seg.index, seg.values, color=ru.PALETTE["orange"]); ax.set_xlabel("Average churn probability"); ax.set_title("Average churn risk by RFM segment")
ru.save_fig(fig, "09_16_risk_by_segment", "Average churn risk by RFM segment", "Segments with low recency scores carry the highest risk.", "Churn Prediction")
sc2.to_csv(ru.DATA_PROC / "churn_scores.csv", index=False)
import joblib; joblib.dump(xgb, ru.MODELS / "churn_model.joblib")
m_ = mt.loc[XN]; print(f"AUC {m_['AUC-ROC']:.3f} (target >= 0.88: {'PASS' if m_['AUC-ROC'] >= 0.88 else 'NOT MET'}) | precision@top20% {m_['Precision@top20%']:.3f} (target >= 0.75: {'PASS' if m_['Precision@top20%'] >= 0.75 else 'NOT MET'})")
ru.log_metrics("churn", {"model": XN, **{k: float(v) for k, v in m_.items()}, "churn_rate_pct": float(y.mean() * 100), "n_customers": int(len(X)),
                         "table": mt.round(4).to_dict(orient="index"), "high_risk_customers": int((scores["churn_probability"] > 0.7).sum())})

## Summary of Notebook 09
* Time-based churn label, 3 models, ROC/PR/gain/lift/calibration/threshold charts, **SHAP** explainability, scores for every customer (`churn_scores.csv`). **16 charts.**
* Always read the printed PASS / NOT MET lines - results depend on the real dataset and are never faked.

---
# PART 10 - 10 - Inventory Optimization using Forecasted Demand
*Week 2 - Day 10*

## Introduction
Turning forecasts into decisions. For the top SKUs we compute:
* **Safety stock** `SS = z x sigma x sqrt(L + R)`  (z from the service level, L = lead time, R = review period)
* **Reorder point** `ROP = mu x L + z x sigma x sqrt(L)`
* **Order-up-to level** `S = mu x (L + R) + SS`   and **EOQ** `sqrt(2 D S_cost / H)`
* **Recommended order quantity** = `S - stock on hand`.

Demand `mu` is the history mean scaled by the **hybrid forecast growth ratio** (top-down forecasting). 
> **Important:** retail transaction datasets have *no stock-on-hand column*, so current stock is **simulated** for the demo (clearly labelled). With a real inventory table, replace `current_stock_simulated`.
Then a **back-test** replays the last 30 real days to compare a naive buffer rule against the forecast-driven policy (F-05).

In [ ]:
bundle = ru.get_data()
sales, returns, rep = bundle["sales"], bundle["returns"], bundle["report"]
print("DATA SOURCE :", bundle["source"], "|", f"{len(sales):,} clean rows")
daily = ru.daily_series(sales); TARGET = "quantity"; H = ru.CONFIG["FORECAST_HORIZON"]

In [ ]:
fb = ru.forecast_bundle(daily, TARGET, H)
recent = daily[TARGET].tail(90).mean()
ratio = float(np.clip(fb["future"]["hybrid"].mean() / max(recent, 1e-9), 0.5, 2.0))
print(f"Forecast growth ratio (next 30d mean / last 90d mean) = {ratio:.2f}")
plan = ru.inventory_plan(sales, growth_ratio=ratio)
display(plan.head(10).round(1))

### Step 1 - Stock status and recommended orders

In [ ]:
st = plan["status"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(st.values, labels=st.index, autopct="%1.0f%%", colors=[ru.PALETTE["red"], ru.PALETTE["green"], ru.PALETTE["orange"]][:len(st)], wedgeprops=dict(width=0.5, edgecolor="white")); axes[0].set_title("Stock status of top SKUs")
t10 = plan.nlargest(10, "order_value").sort_values("order_value"); axes[1].barh(t10["description"].str[:28], t10["order_value"], color=ru.PALETTE["teal"]); axes[1].set_title("Top 10 recommended purchase orders (value)")
ru.save_fig(fig, "10_01_status_orders", "Stock status and top recommended orders", f"{int((plan['status'] == 'Reorder Now').sum())} SKUs need reordering now.", "Inventory Optimization")

fig, ax = plt.subplots(figsize=(9, 4.8)); sns.histplot(plan["days_of_cover"].clip(upper=120), bins=30, ax=ax, color=ru.PALETTE["purple"]); ax.axvline(45, color="red", ls="--", label="overstock > 45 days"); ax.legend(); ax.set_title("Days of cover distribution")
ru.save_fig(fig, "10_02_days_of_cover", "Days of cover distribution", "SKUs right of the red line are overstocked.", "Inventory Optimization")

fig, ax = plt.subplots(figsize=(8.5, 6.5)); cmap = {"Reorder Now": ru.PALETTE["red"], "Healthy": ru.PALETTE["green"], "Overstock": ru.PALETTE["orange"]}
for s_, g in plan.groupby("status"): ax.scatter(g["reorder_point"], g["current_stock_simulated"], s=60, color=cmap[s_], label=s_, alpha=0.85)
lim = max(plan["reorder_point"].max(), plan["current_stock_simulated"].max()) * 1.05; ax.plot([0, lim], [0, lim], "k--", label="stock = ROP"); ax.legend(); ax.set_xlabel("Reorder point"); ax.set_ylabel("Current stock (simulated)"); ax.set_title("Current stock vs reorder point")
ru.save_fig(fig, "10_03_stock_vs_rop", "Current stock vs reorder point", "Points below the diagonal are under the reorder point.", "Inventory Optimization")

### Step 2 - Demand variability, ABC analysis and EOQ

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6)); sc = ax.scatter(plan["avg_daily_demand_hist"], plan["safety_stock"], c=plan["demand_std"], cmap="viridis", s=70); plt.colorbar(sc, label="demand std")
ax.set_xlabel("Average daily demand"); ax.set_ylabel("Safety stock"); ax.set_title("Safety stock grows with demand variability")
ru.save_fig(fig, "10_04_safety_stock", "Safety stock vs demand and variability", "More volatile SKUs need more buffer.", "Inventory Optimization")

rev = sales.groupby("stock_code")["revenue"].sum().sort_values(ascending=False); cum = rev.cumsum() / rev.sum()
abc = pd.Series(np.where(cum <= 0.8, "A", np.where(cum <= 0.95, "B", "C")), index=rev.index)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
abc.value_counts().reindex(["A", "B", "C"]).plot(kind="bar", ax=axes[0], color=[ru.PALETTE["red"], ru.PALETTE["orange"], ru.PALETTE["green"]]); axes[0].set_title("SKU count per ABC class"); axes[0].tick_params(axis="x", rotation=0)
rev.groupby(abc).sum().reindex(["A", "B", "C"]).plot(kind="bar", ax=axes[1], color=[ru.PALETTE["red"], ru.PALETTE["orange"], ru.PALETTE["green"]]); axes[1].set_title("Revenue per ABC class"); axes[1].tick_params(axis="x", rotation=0)
ru.save_fig(fig, "10_05_abc_analysis", "ABC inventory classification", "A-class SKUs (80% of revenue) deserve tight control and high service levels.", "Inventory Optimization")

t = plan.nlargest(12, "EOQ").sort_values("EOQ"); fig, ax = plt.subplots(figsize=(9, 5.5)); ax.barh(t["description"].str[:28], t["EOQ"], color=ru.PALETTE["navy"]); ax.set_title("Economic order quantity (EOQ) - top 12 SKUs")
ru.save_fig(fig, "10_06_eoq", "Economic order quantity", "Balances ordering cost against holding cost.", "Inventory Optimization")

### Step 3 - Sensitivity: service level and lead time

In [ ]:
rows = []
for sl in [0.85, 0.90, 0.95, 0.975, 0.99]:
    p_ = ru.inventory_plan(sales, ratio, service_level=sl); rows.append({"service_level": sl, "safety_stock_value": float((p_["safety_stock"] * p_["unit_price"] * 0.6).sum())})
sl_df = pd.DataFrame(rows)
rows = []
for L in [2, 5, 7, 10, 14]:
    p_ = ru.inventory_plan(sales, ratio, lead_time=L); rows.append({"lead_time": L, "safety_stock_value": float((p_["safety_stock"] * p_["unit_price"] * 0.6).sum())})
lt_df = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].plot(sl_df["service_level"] * 100, sl_df["safety_stock_value"], marker="o", color=ru.PALETTE["red"], lw=2.5); axes[0].set_xlabel("Service level %"); axes[0].set_ylabel("Safety-stock investment"); axes[0].set_title("Cost of higher service level")
axes[1].plot(lt_df["lead_time"], lt_df["safety_stock_value"], marker="o", color=ru.PALETTE["teal"], lw=2.5); axes[1].set_xlabel("Lead time (days)"); axes[1].set_title("Cost of longer supplier lead time")
ru.save_fig(fig, "10_07_sensitivity", "Safety-stock sensitivity", "Each extra point of service level and each extra lead-time day costs working capital.", "Inventory Optimization")

### Step 4 - Back-test: naive rule vs forecast-driven policy (last 30 real days)

In [ ]:
# the back-test must use the forecast that was available for THAT window (hybrid test forecast), not the future ratio
ratio_bt = float(np.clip(fb["test_preds"]["Hybrid"].mean() / max(daily[TARGET].iloc[-(H + 90):-H].mean(), 1e-9), 0.5, 2.0))
print(f"Back-test growth ratio from the hybrid forecast for the test window = {ratio_bt:.2f}")
bt = ru.inventory_backtest(sales, ratio_bt, horizon=H)
tot_dem = float(ru.sku_daily_matrix(sales).iloc[-H:].sum().sum())
S = {}
for tag in ("naive", "opt"):
    S[tag] = {"lost_units": bt[f"{tag}_lost_units"].sum(), "stockout_days": bt[f"{tag}_stockout_days"].sum(), "avg_inventory": bt[f"{tag}_avg_inventory"].sum(),
              "overstock_days": bt[f"{tag}_overstock_days"].sum(), "cost": bt[f"{tag}_cost"].sum(), "fill_rate_%": (1 - bt[f"{tag}_lost_units"].sum() / tot_dem) * 100}
sm = pd.DataFrame(S).T; sm.index = ["Naive buffer rule", "Forecast-driven (RetailPulse)"]; display(sm.round(1))
so = (1 - S["opt"]["lost_units"] / max(S["naive"]["lost_units"], 1e-9)) * 100
cost = (1 - S["opt"]["cost"] / max(S["naive"]["cost"], 1e-9)) * 100
inv = (S["opt"]["avg_inventory"] / max(S["naive"]["avg_inventory"], 1e-9) - 1) * 100
fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))
for ax, (c, t) in zip(axes, [("lost_units", "Lost sales (units)"), ("stockout_days", "SKU stock-out days"), ("avg_inventory", "Average inventory (units)"), ("cost", "Total cost (lost margin + holding)")]):
    ax.bar(["Naive", "RetailPulse"], [S["naive"][c], S["opt"][c]], color=[ru.PALETTE["grey"], ru.PALETTE["teal"]]); ax.set_title(t)
    for i, v in enumerate([S["naive"][c], S["opt"][c]]): ax.text(i, v, f"{v:,.0f}", ha="center", va="bottom")
ru.save_fig(fig, "10_08_backtest", "Inventory policy back-test", f"Stock-out units change {-so:+.0f}%, total cost change {-cost:+.0f}%, inventory change {inv:+.0f}%.", "Inventory Optimization")
print(f"Lost-sales reduction: {so:.1f}% | Total cost reduction: {cost:.1f}% | Inventory level change: {inv:+.1f}% (more safety stock is the price of fewer stock-outs)")
print("F-05 target (25-40% reduction of overstock/understock cost):", "PASS" if cost >= 25 else "NOT MET")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6)); ax.scatter(bt["naive_lost_units"], bt["opt_lost_units"], s=55, color=ru.PALETTE["purple"], alpha=0.8)
lim = max(bt["naive_lost_units"].max(), bt["opt_lost_units"].max()) * 1.05 + 1; ax.plot([0, lim], [0, lim], "k--"); ax.set_xlabel("Lost units - naive rule"); ax.set_ylabel("Lost units - RetailPulse"); ax.set_title("Per-SKU lost sales (below the line = improvement)")
ru.save_fig(fig, "10_09_sku_lost_scatter", "Per-SKU lost sales comparison", "Most SKUs lie below the diagonal -> fewer stock-outs.", "Inventory Optimization")
plan.to_csv(ru.DATA_PROC / "inventory_plan.csv", index=False); bt.to_csv(ru.DATA_PROC / "inventory_backtest.csv", index=False)
ru.log_metrics("inventory", {"growth_ratio": ratio, "lost_sales_reduction_pct": float(so), "cost_reduction_pct": float(cost), "inventory_change_pct": float(inv),
                             "skus": int(len(plan)), "reorder_now": int((plan["status"] == "Reorder Now").sum()), "overstock": int((plan["status"] == "Overstock").sum()),
                             "fill_rate_naive_pct": float(S["naive"]["fill_rate_%"]), "fill_rate_opt_pct": float(S["opt"]["fill_rate_%"])})

## Summary of Notebook 10
* Safety stock, reorder point, EOQ, order-up-to and recommended order quantity for the top SKUs, ABC analysis, sensitivity analysis and a **30-day policy back-test**. **9 charts.**
* Stock-on-hand is simulated (no such column in public retail data) - replace with a real inventory feed in production.

---
# PART 11 - 11 - Hyper-parameter Tuning with Optuna & Feature Importance
*Week 2 - Day 11*

## Introduction
**Optuna** performs Bayesian (TPE) hyper-parameter search. We tune:
1. the **XGBoost churn model** (objective = 3-fold cross-validated AUC on the training set only),
2. **Prophet** (changepoint / seasonality prior and mode, objective = validation WAPE).
The test set is used once at the end to compare *default vs tuned*.

In [ ]:
bundle = ru.get_data()
sales, returns, rep = bundle["sales"], bundle["returns"], bundle["report"]
print("DATA SOURCE :", bundle["source"], "|", f"{len(sales):,} clean rows")
daily = ru.daily_series(sales); TARGET = "quantity"; H = ru.CONFIG["FORECAST_HORIZON"]

In [ ]:
cs = ru.get_customer_sales()
X, y, cutoff = ru.build_churn_dataset(cs, returns)
res = ru.train_churn_models(X, y)
Xtr, Xva, Xte, ytr, yva, yte = res["splits"]
XN = res["xgb_name"]; xgb = res["models"][XN]
print(f"Customers: {len(X)} | features: {X.shape[1]} | churn rate: {y.mean() * 100:.1f}% | label window: last {ru.CONFIG['CHURN_WINDOW_DAYS']} days (cutoff {cutoff.date()})")

In [ ]:
import optuna, json
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score
cv = StratifiedKFold(3, shuffle=True, random_state=ru.SEED)
def objective(trial):
    params = dict(n_estimators=trial.suggest_int("n_estimators", 100, 500, step=50), max_depth=trial.suggest_int("max_depth", 2, 7),
                  learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True), subsample=trial.suggest_float("subsample", 0.6, 1.0),
                  colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0), min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
                  reg_lambda=trial.suggest_float("reg_lambda", 0.1, 10.0, log=True))
    m, _ = ru.make_xgb(**params)
    return cross_val_score(m, Xtr, ytr, cv=cv, scoring="roc_auc").mean()
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=ru.SEED))
study.optimize(objective, n_trials=25, show_progress_bar=False)
print("Best CV AUC:", round(study.best_value, 4)); print("Best params:", study.best_params)

### Step 1 - Optimisation history and parameter importance

In [ ]:
tdf = study.trials_dataframe(); tdf["best_so_far"] = tdf["value"].cummax()
fig, ax = plt.subplots(figsize=(10, 4.8)); ax.scatter(tdf["number"], tdf["value"], color=ru.PALETTE["sky"], label="Trial AUC"); ax.plot(tdf["number"], tdf["best_so_far"], color=ru.PALETTE["red"], lw=2.5, label="Best so far")
ax.legend(); ax.set_xlabel("Trial"); ax.set_ylabel("CV AUC"); ax.set_title("Optuna optimisation history (XGBoost churn)")
ru.save_fig(fig, "11_01_optuna_history", "Optuna optimisation history", f"Best CV AUC = {study.best_value:.4f} after {len(tdf)} trials.", "Tuning")
try:
    imp = optuna.importance.get_param_importances(study); s = pd.Series(imp).sort_values()
    fig, ax = plt.subplots(figsize=(8.5, 5)); ax.barh(s.index, s.values, color=ru.PALETTE["purple"]); ax.set_title("Hyper-parameter importance (fANOVA)")
    ru.save_fig(fig, "11_02_param_importance", "Hyper-parameter importance", f"Most influential parameter: {s.index[-1]}.", "Tuning")
except Exception as e:
    print("Param importance skipped:", type(e).__name__)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sc = axes[0].scatter(tdf["params_learning_rate"], tdf["value"], c=tdf["params_max_depth"], cmap="viridis", s=60); axes[0].set_xscale("log"); axes[0].set_xlabel("learning_rate"); axes[0].set_ylabel("CV AUC"); plt.colorbar(sc, ax=axes[0], label="max_depth"); axes[0].set_title("Learning rate vs AUC")
axes[1].scatter(tdf["params_n_estimators"], tdf["value"], c=tdf["params_subsample"], cmap="plasma", s=60); axes[1].set_xlabel("n_estimators"); axes[1].set_title("Trees vs AUC (colour = subsample)")
ru.save_fig(fig, "11_03_optuna_slices", "Optuna parameter slices", "Shows which regions of the search space perform best.", "Tuning")

### Step 2 - Default vs tuned on the untouched test set

In [ ]:
dm, _ = ru.make_xgb(scale_pos_weight=float((ytr == 0).sum() / max((ytr == 1).sum(), 1))); dm.fit(Xtr, ytr)
tm, _ = ru.make_xgb(**study.best_params, scale_pos_weight=float((ytr == 0).sum() / max((ytr == 1).sum(), 1))); tm.fit(Xtr, ytr)
a0, a1 = roc_auc_score(yte, dm.predict_proba(Xte)[:, 1]), roc_auc_score(yte, tm.predict_proba(Xte)[:, 1])
p0, p1 = ru.precision_at_k(yte, dm.predict_proba(Xte)[:, 1]), ru.precision_at_k(yte, tm.predict_proba(Xte)[:, 1])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, (t, v) in zip(axes, [("Test AUC-ROC", [a0, a1]), ("Precision@top20%", [p0, p1])]):
    ax.bar(["Default", "Tuned"], v, color=[ru.PALETTE["grey"], ru.PALETTE["teal"]]); ax.set_ylim(0, 1.05); ax.set_title(t)
    for i, x in enumerate(v): ax.text(i, x + 0.01, f"{x:.3f}", ha="center")
ru.save_fig(fig, "11_04_default_vs_tuned", "Default vs tuned XGBoost", f"AUC {a0:.3f} -> {a1:.3f}. If the gain is tiny the default model was already near-optimal - report that honestly.", "Tuning")
print(f"Default AUC {a0:.4f} | Tuned AUC {a1:.4f}")

### Step 3 - Tune Prophet (validation WAPE)

In [ ]:
tr, va, te = ru.split_series(daily, H)
def obj_p(trial):
    r = ru.prophet_forecast(tr, va["ds"], TARGET, changepoint_prior_scale=trial.suggest_float("changepoint_prior_scale", 0.005, 0.5, log=True),
                            seasonality_prior_scale=trial.suggest_float("seasonality_prior_scale", 0.1, 20.0, log=True),
                            seasonality_mode=trial.suggest_categorical("seasonality_mode", ["additive", "multiplicative"]))
    return ru.wape(va[TARGET].values, r["pred"])
sp = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=ru.SEED)); sp.optimize(obj_p, n_trials=10)
print("Best Prophet params:", sp.best_params, "| val WAPE:", round(sp.best_value, 2))
pdf = sp.trials_dataframe(); pdf["best"] = pdf["value"].cummin()
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8)); axes[0].scatter(pdf["number"], pdf["value"], color=ru.PALETTE["sky"]); axes[0].plot(pdf["number"], pdf["best"], color=ru.PALETTE["red"], lw=2.5); axes[0].set_title("Prophet tuning history (validation WAPE %)"); axes[0].set_xlabel("Trial")
axes[1].scatter(pdf["params_changepoint_prior_scale"], pdf["value"], c=(pdf["params_seasonality_mode"] == "multiplicative").astype(int), cmap="coolwarm", s=70); axes[1].set_xscale("log"); axes[1].set_xlabel("changepoint_prior_scale"); axes[1].set_title("Changepoint prior vs WAPE (colour = mode)")
ru.save_fig(fig, "11_05_prophet_tuning", "Prophet tuning with Optuna", f"Best validation WAPE {sp.best_value:.2f}%.", "Tuning")
json.dump({"xgboost": study.best_params, "prophet": sp.best_params}, open(ru.MODELS / "best_params.json", "w"), indent=2)
ru.log_metrics("tuning", {"xgb_best_cv_auc": study.best_value, "xgb_default_test_auc": a0, "xgb_tuned_test_auc": a1, "prophet_best_val_wape": sp.best_value, "xgb_best_params": study.best_params, "prophet_best_params": sp.best_params})

## Summary of Notebook 11
* Optuna TPE search for XGBoost (25 trials) and Prophet (10 trials), parameter importance, slices and a clean default-vs-tuned comparison on the untouched test set. **5 charts.** Best parameters saved to `models/best_params.json`.

---
# PART 12 - 12 - Data & Model Drift Detection (PSI, KS, Evidently)
*Week 2 - Day 12*

## Introduction
Models degrade when the world changes. We monitor:
* **Data drift** - PSI (Population Stability Index: <0.1 stable, 0.1-0.2 warning, >0.2 drift) and the Kolmogorov-Smirnov test, reference period vs current period.
* **Model-performance drift** - rolling forecast error.
* **Real drift simulation** (required by the Zidio guidelines): we deliberately inject a price increase and demand drop, then show the monitor firing and the **retrain trigger**.

In [ ]:
bundle = ru.get_data()
sales, returns, rep = bundle["sales"], bundle["returns"], bundle["report"]
print("DATA SOURCE :", bundle["source"], "|", f"{len(sales):,} clean rows")
daily = ru.daily_series(sales); TARGET = "quantity"; H = ru.CONFIG["FORECAST_HORIZON"]

In [ ]:
cut = sales["date"].min() + (sales["date"].max() - sales["date"].min()) * 0.7
ref, cur = sales[sales["date"] <= cut].copy(), sales[sales["date"] > cut].copy()
FEATS = ["quantity", "unit_price", "revenue", "hour"]
d0 = ru.drift_table(ref, cur, FEATS)
print(f"Reference: {ref['date'].min().date()} -> {ref['date'].max().date()} ({len(ref):,} rows) | Current: {cur['date'].min().date()} -> {cur['date'].max().date()} ({len(cur):,} rows)")
display(d0.round(4))

### Step 1 - Natural drift (no manipulation)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for ax, f in zip(axes, FEATS):
    a, b = np.log1p(ref[f]), np.log1p(cur[f]); sns.kdeplot(a, ax=ax, color=ru.PALETTE["navy"], fill=True, alpha=0.3, label="Reference"); sns.kdeplot(b, ax=ax, color=ru.PALETTE["red"], fill=True, alpha=0.3, label="Current")
    ax.set_title(f"{f}  (PSI {d0.set_index('feature').loc[f, 'PSI']:.3f})"); ax.legend()
ru.save_fig(fig, "12_01_distribution_overlay", "Reference vs current distributions", "Overlapping curves = stable data.", "Drift Detection")
fig, ax = plt.subplots(figsize=(8.5, 4.5)); ax.bar(d0["feature"], d0["PSI"], color=[ru.PALETTE["green"] if s == "OK" else ru.PALETTE["orange"] if s == "WARNING" else ru.PALETTE["red"] for s in d0["status"]])
ax.axhline(0.1, color="orange", ls="--"); ax.axhline(0.2, color="red", ls="--"); ax.set_ylabel("PSI"); ax.set_title("Natural data drift (PSI per feature)")
ru.save_fig(fig, "12_02_psi_natural", "PSI per feature (natural drift)", "Green = stable, orange = warning, red = drift.", "Drift Detection")

### Step 2 - Real drift simulation (injected shift)

In [ ]:
sim = cur.copy()
rng = np.random.default_rng(ru.SEED)
sim["unit_price"] = sim["unit_price"] * 1.35                                  # price shock
sim["quantity"] = np.maximum(1, np.round(sim["quantity"] * rng.uniform(0.4, 0.8, len(sim)))).astype(int)   # demand collapse
sim["hour"] = np.clip(sim["hour"] + 2, 0, 23)                                 # shifted ordering time
sim["revenue"] = sim["quantity"] * sim["unit_price"]
d1 = ru.drift_table(ref, sim, FEATS); display(d1.round(4))
x = np.arange(len(FEATS)); fig, ax = plt.subplots(figsize=(10, 4.8)); ax.bar(x - 0.2, d0["PSI"], 0.4, label="Natural", color=ru.PALETTE["green"]); ax.bar(x + 0.2, d1["PSI"], 0.4, label="After injected drift", color=ru.PALETTE["red"])
ax.axhline(0.2, color="red", ls="--", label="drift threshold 0.2"); ax.set_xticks(x); ax.set_xticklabels(FEATS); ax.legend(); ax.set_title("PSI before vs after the simulated drift")
ru.save_fig(fig, "12_03_psi_simulated", "PSI before and after simulated drift", "The monitor correctly flags the injected price / quantity shift.", "Drift Detection")
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, f in zip(axes, ["quantity", "unit_price", "hour"]):
    sns.kdeplot(np.log1p(ref[f]), ax=ax, color=ru.PALETTE["navy"], fill=True, alpha=0.3, label="Reference"); sns.kdeplot(np.log1p(sim[f]), ax=ax, color=ru.PALETTE["red"], fill=True, alpha=0.3, label="Drifted"); ax.set_title(f); ax.legend()
ru.save_fig(fig, "12_04_distribution_drifted", "Distribution shift after simulated drift", "Visible shift of the red curves away from the reference.", "Drift Detection")

### Step 3 - Customer-level feature drift (churn model inputs)

In [ ]:
cs = ru.get_customer_sales(); end = cs["invoice_date"].max().normalize()
snapA, snapB = end - pd.Timedelta(days=240), end - pd.Timedelta(days=90)
fa = ru.customer_features(cs[cs["invoice_date"] < snapA], snapA); fb_ = ru.customer_features(cs[cs["invoice_date"] < snapB], snapB)
CF = ["recency", "frequency", "monetary", "avg_gap_days", "avg_order_value", "n_products"]
d2 = ru.drift_table(fa, fb_, CF); display(d2.round(3))
fig, ax = plt.subplots(figsize=(9.5, 4.8)); ax.bar(d2["feature"], d2["PSI"], color=[ru.PALETTE["green"] if s == "OK" else ru.PALETTE["orange"] if s == "WARNING" else ru.PALETTE["red"] for s in d2["status"]])
ax.axhline(0.1, color="orange", ls="--"); ax.axhline(0.2, color="red", ls="--"); ax.tick_params(axis="x", rotation=20); ax.set_title("Customer feature drift (snapshot -240d vs -90d)")
ru.save_fig(fig, "12_05_customer_feature_drift", "Customer feature drift", "Customers naturally age (recency / tenure move) - PSI quantifies how much.", "Drift Detection")

### Step 4 - Model-performance drift and the retraining trigger

In [ ]:
fb = ru.forecast_bundle(daily, TARGET, H)
te = fb["test"]; yv = te[TARGET].values; pv = fb["test_preds"]["Hybrid"]
good = pd.Series(np.abs(yv - pv), index=te["ds"]).rolling(7, min_periods=3).sum() / pd.Series(yv, index=te["ds"]).rolling(7, min_periods=3).sum() * 100
yd = yv.copy().astype(float); yd[len(yd) // 2:] *= 1.6                          # simulated demand shock in 2nd half
bad = pd.Series(np.abs(yd - pv), index=te["ds"]).rolling(7, min_periods=3).sum() / pd.Series(yd, index=te["ds"]).rolling(7, min_periods=3).sum() * 100
fig, ax = plt.subplots(figsize=(12, 4.8)); ax.plot(good, color=ru.PALETTE["green"], lw=2.5, label="Normal operation"); ax.plot(bad, color=ru.PALETTE["red"], lw=2.5, label="After simulated demand shock")
thr = float(max(good.max() * 1.25, 30)); ax.axhline(thr, color="black", ls="--", label=f"retrain threshold ({thr:.0f}% rolling WAPE)"); ax.legend(); ax.set_ylabel("Rolling 7-day WAPE %"); ax.set_title("Forecast performance monitoring")
ru.save_fig(fig, "12_06_performance_monitor", "Forecast performance monitoring", "Rolling error crosses the threshold after the simulated shock -> alert.", "Drift Detection")
perf_alert = bool(bad.max() > thr)
dec = ru.retrain_decision(d1, float(bad.iloc[-1]), mape_limit=thr / 2.5)
print("Natural data drift :", ru.retrain_decision(d0, float(good.iloc[-1]), mape_limit=thr / 2.5))
print("Simulated drift    :", dec, "| performance alert:", perf_alert)
rp = ru.evidently_report(ref[FEATS].reset_index(drop=True), sim[FEATS].reset_index(drop=True), str(ru.REPORTS / "evidently_drift_report.html"))
print("Evidently HTML report:", rp or "skipped (native PSI/KS report used)")
ru.log_metrics("drift", {"psi_natural": d0.set_index("feature")["PSI"].to_dict(), "psi_simulated": d1.set_index("feature")["PSI"].to_dict(),
                         "retrain_triggered_on_simulation": bool(dec["retrain"] or perf_alert), "reasons": dec["reasons"], "evidently_report": bool(rp)})

## Summary of Notebook 12
* PSI + KS drift tables, natural vs **simulated** drift (price shock, demand collapse), customer-feature drift, rolling error monitor and an automatic **retrain decision**. Optional Evidently AI HTML report. **6 charts.**

---
# PART 13 - 13 - Model Refinement & Automated Retraining Pipeline (Airflow)
*Week 2 - Day 13*

## Introduction
Refinement = squeeze reliable gains **without overfitting**:
1. **Probability calibration** (isotonic) so churn scores behave like real probabilities,
2. **Business-cost threshold** - choose the cut-off that maximises campaign profit instead of F1,
3. **Feature-group ablation** - which feature families really matter,
4. **Churn ensemble** (LR + RF + XGBoost average) and **learning curve**,
5. **Forecast bias correction** by weekday,
6. The **Airflow retraining DAG** (written to disk and syntax-checked).

In [ ]:
bundle = ru.get_data()
sales, returns, rep = bundle["sales"], bundle["returns"], bundle["report"]
print("DATA SOURCE :", bundle["source"], "|", f"{len(sales):,} clean rows")
daily = ru.daily_series(sales); TARGET = "quantity"; H = ru.CONFIG["FORECAST_HORIZON"]

In [ ]:
cs = ru.get_customer_sales()
X, y, cutoff = ru.build_churn_dataset(cs, returns)
res = ru.train_churn_models(X, y)
Xtr, Xva, Xte, ytr, yva, yte = res["splits"]
XN = res["xgb_name"]; xgb = res["models"][XN]
print(f"Customers: {len(X)} | features: {X.shape[1]} | churn rate: {y.mean() * 100:.1f}% | label window: last {ru.CONFIG['CHURN_WINDOW_DAYS']} days (cutoff {cutoff.date()})")

### Step 1 - Calibration

In [ ]:
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score
pv, pt = xgb.predict_proba(Xva)[:, 1], xgb.predict_proba(Xte)[:, 1]
iso = IsotonicRegression(out_of_bounds="clip").fit(pv, yva); pt_cal = iso.predict(pt)
b0, b1 = brier_score_loss(yte, pt), brier_score_loss(yte, pt_cal)
fig, ax = plt.subplots(figsize=(6.5, 6))
for p, n, c in [(pt, "XGBoost raw", ru.PALETTE["red"]), (pt_cal, "XGBoost + isotonic", ru.PALETTE["teal"])]:
    fr, mp = calibration_curve(yte, p, n_bins=8, strategy="quantile"); ax.plot(mp, fr, marker="o", lw=2, color=c, label=n)
ax.plot([0, 1], [0, 1], "k--"); ax.legend(); ax.set_xlabel("Predicted"); ax.set_ylabel("Observed"); ax.set_title(f"Calibration (Brier {b0:.3f} -> {b1:.3f})")
ru.save_fig(fig, "13_01_calibration", "Calibration before vs after isotonic", f"Brier score {b0:.3f} -> {b1:.3f} (lower is better); AUC {roc_auc_score(yte, pt):.3f} -> {roc_auc_score(yte, pt_cal):.3f}.", "Model Refinement")

### Step 2 - Profit-based threshold

In [ ]:
mon = X.loc[Xte.index, "monetary"].values; AVG_VALUE = float(np.median(mon)) * 0.2; CONTACT_COST = 2.0; SUCCESS = 0.25
rows = []
for t in np.linspace(0.05, 0.95, 46):
    flag = pt_cal >= t; tp = (flag & (np.asarray(yte) == 1)).sum()
    rows.append({"threshold": t, "profit": tp * AVG_VALUE * SUCCESS - flag.sum() * CONTACT_COST, "contacted": int(flag.sum())})
pf = pd.DataFrame(rows); best = pf.loc[pf["profit"].idxmax()]
fig, ax = plt.subplots(figsize=(9, 4.8)); ax.plot(pf["threshold"], pf["profit"], color=ru.PALETTE["green"], lw=2.5); ax.axvline(best["threshold"], color="red", ls="--", label=f"best = {best['threshold']:.2f}"); ax.legend()
ax.set_xlabel("Threshold"); ax.set_ylabel("Expected campaign profit"); ax.set_title("Profit-optimal churn threshold (contact cost 2, 25% save rate)")
ru.save_fig(fig, "13_02_profit_threshold", "Profit-optimal threshold", f"Best threshold {best['threshold']:.2f} contacts {int(best['contacted'])} customers; expected profit {best['profit']:,.0f}.", "Model Refinement")

### Step 3 - Feature-group ablation, ensemble and learning curve

In [ ]:
groups = {"Recency features": ["recency", "recency_gap_ratio"], "Frequency / monetary": ["frequency", "monetary", "avg_order_value", "n_products"],
          "Trend / recent activity": ["orders_last_30d", "orders_last_90d", "orders_prev_90d", "order_trend", "revenue_last_90d"],
          "Gap behaviour": ["avg_gap_days", "std_gap_days", "single_order"], "Tenure": ["tenure_days"]}
base = roc_auc_score(yte, pt); rows = []
for g, cols in groups.items():
    keep = [c for c in X.columns if c not in cols]; m, _ = ru.make_xgb(); m.fit(Xtr[keep], ytr); rows.append({"dropped_group": g, "AUC": roc_auc_score(yte, m.predict_proba(Xte[keep])[:, 1])})
ab = pd.DataFrame(rows); ab["delta_vs_full"] = ab["AUC"] - base; display(ab.round(4))
fig, ax = plt.subplots(figsize=(9, 4.8)); ax.barh(ab["dropped_group"], ab["delta_vs_full"], color=np.where(ab["delta_vs_full"] < 0, ru.PALETTE["red"], ru.PALETTE["green"])); ax.axvline(0, color="black"); ax.set_xlabel("AUC change when the group is removed"); ax.set_title("Feature-group ablation")
ru.save_fig(fig, "13_03_ablation", "Feature-group ablation", "Negative bars = the group helps the model.", "Model Refinement")

ens = np.mean([res["probs"][n] for n in res["probs"]], axis=0); a_e = roc_auc_score(yte, ens)
cmpa = pd.Series({**{n: roc_auc_score(yte, p) for n, p in res["probs"].items()}, "Average ensemble": a_e})
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8)); cmpa.plot(kind="bar", ax=axes[0], color=ru.COLORS[:4]); axes[0].set_ylim(max(0, cmpa.min() - 0.05), 1); axes[0].set_title("Churn: single models vs ensemble (AUC)"); axes[0].tick_params(axis="x", rotation=20)
for i, v in enumerate(cmpa): axes[0].text(i, v, f"{v:.3f}", ha="center", va="bottom")
fr = [0.2, 0.4, 0.6, 0.8, 1.0]; tr_a, te_a = [], []
for f in fr:
    n = max(int(len(Xtr) * f), 30); m, _ = ru.make_xgb(); m.fit(Xtr.iloc[:n], ytr.iloc[:n]); tr_a.append(roc_auc_score(ytr.iloc[:n], m.predict_proba(Xtr.iloc[:n])[:, 1])); te_a.append(roc_auc_score(yte, m.predict_proba(Xte)[:, 1]))
axes[1].plot(fr, tr_a, marker="o", label="Train AUC", color=ru.PALETTE["navy"]); axes[1].plot(fr, te_a, marker="o", label="Test AUC", color=ru.PALETTE["red"]); axes[1].legend(); axes[1].set_xlabel("Fraction of training data"); axes[1].set_title("Learning curve")
ru.save_fig(fig, "13_04_ensemble_learning_curve", "Churn ensemble and learning curve", "A gap between train and test AUC signals over-fitting.", "Model Refinement")

### Step 4 - Forecast bias correction by weekday

In [ ]:
tr_, va_, te_ = ru.split_series(daily, H); trv = pd.concat([tr_, va_])
pv_ = ru.prophet_forecast(tr_, va_["ds"], TARGET)["pred"]; pt_ = ru.prophet_forecast(trv, te_["ds"], TARGET)["pred"]
dv = pd.DataFrame({"dow": va_["ds"].dt.dayofweek.values, "a": va_[TARGET].values, "p": pv_}); fac = (dv.groupby("dow")["a"].sum() / dv.groupby("dow")["p"].sum().replace(0, np.nan)).fillna(1.0)
fac = 1 + 0.5 * (fac.clip(0.7, 1.4) - 1)                        # shrink correction by 50% to avoid over-fitting
corr = pt_ * te_["ds"].dt.dayofweek.map(fac).fillna(1.0).values
w0, w1 = ru.wape(te_[TARGET].values, pt_), ru.wape(te_[TARGET].values, corr)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8)); axes[0].bar(["Prophet", "Prophet + bias fix"], [w0, w1], color=[ru.PALETTE["grey"], ru.PALETTE["teal"]]); axes[0].set_title("Test WAPE %")
for i, v in enumerate([w0, w1]): axes[0].text(i, v, f"{v:.1f}", ha="center", va="bottom")
axes[1].plot(te_["ds"], te_[TARGET], color=ru.PALETTE["navy"], lw=2.5, label="Actual"); axes[1].plot(te_["ds"], pt_, "--", color=ru.PALETTE["orange"], label="Prophet"); axes[1].plot(te_["ds"], corr, color=ru.PALETTE["teal"], label="Corrected"); axes[1].legend(); axes[1].tick_params(axis="x", rotation=40)
ru.save_fig(fig, "13_05_bias_correction", "Weekday bias correction", f"WAPE {w0:.1f}% -> {w1:.1f}%. Keep the correction only if it helps on validation AND test.", "Model Refinement")
ru.log_metrics("refinement", {"brier_raw": b0, "brier_calibrated": b1, "best_profit_threshold": float(best["threshold"]), "ensemble_auc": float(a_e), "bias_fix_wape_before": w0, "bias_fix_wape_after": w1})

### Step 5 - Automated retraining pipeline (Airflow DAG)

In [ ]:
DAG_SRC = '"""\nRetailPulse - automated retraining DAG (Apache Airflow 2.x)\nFlow: ingest_clean -> drift_check -> (retrain | skip) -> evaluate_and_register\nRun `python scripts/retrain_pipeline.py` for the same logic without Airflow.\nAuthor: Santosh Kumar\n"""\nfrom __future__ import annotations\n\nimport sys\nfrom datetime import datetime, timedelta\nfrom pathlib import Path\n\ntry:  # Airflow is optional - the file stays importable for tests / linting\n    from airflow import DAG\n    from airflow.operators.python import BranchPythonOperator, PythonOperator\n    from airflow.operators.empty import EmptyOperator\nexcept Exception:  # pragma: no cover\n    DAG = None\n\nREPO = Path(__file__).resolve().parents[2]\nsys.path.insert(0, str(REPO / "src"))\nsys.path.insert(0, str(REPO / "scripts"))\n\n\ndef ingest_clean(**_):\n    import retailpulse_utils as ru\n    ru.get_data(force=True)\n\n\ndef drift_check(**_):\n    from retrain_pipeline import drift_needs_retrain\n    return "retrain" if drift_needs_retrain() else "skip_retrain"\n\n\ndef retrain(**_):\n    from retrain_pipeline import run_retrain\n    run_retrain()\n\n\ndef evaluate_and_register(**_):\n    import retailpulse_utils as ru\n    print("Latest metrics:", ru.get_metrics().get("retrain", {}))\n\n\nif DAG is not None:\n    default_args = {"owner": "santosh.kumar", "retries": 1, "retry_delay": timedelta(minutes=5)}\n    with DAG("retailpulse_retraining", default_args=default_args, start_date=datetime(2026, 3, 1),\n             schedule="@weekly", catchup=False, tags=["retailpulse", "mlops"]) as dag:\n        t1 = PythonOperator(task_id="ingest_clean", python_callable=ingest_clean)\n        t2 = BranchPythonOperator(task_id="drift_check", python_callable=drift_check)\n        t3 = PythonOperator(task_id="retrain", python_callable=retrain)\n        t4 = EmptyOperator(task_id="skip_retrain")\n        t5 = PythonOperator(task_id="evaluate_and_register", python_callable=evaluate_and_register, trigger_rule="none_failed_min_one_success")\n        t1 >> t2 >> [t3, t4] >> t5\n'
p = ru.ROOT / "airflow_retailpulse_retrain_dag.py"; p.write_text(DAG_SRC)
import py_compile; py_compile.compile(str(p), doraise=True)
print("DAG written and syntax-checked ->", p)
print("Schedule: weekly | tasks: ingest_clean -> drift_check -> (retrain | skip) -> evaluate_and_register")
print("Same logic without Airflow:  python scripts/retrain_pipeline.py --force")

## Summary of Notebook 13
* Calibration, profit-optimal threshold, ablation, ensemble, learning curve, weekday bias fix (**5 charts**) and the **Airflow retraining DAG** (weekly, drift-gated).

---
# PART 14 - 14 - MLflow Week-2 Tracking, Acceptance Checklist & Model Card
*Week 2 - Day 14 (checkpoint)*

## Introduction
Week-2 checkpoint: log the advanced models (hybrid forecaster, churn models, tuned XGBoost, inventory policy) into **RetailPulse-Week2**, compare them with Week 1, evaluate every **acceptance criterion from the project brief** honestly and write a **model card**.

In [ ]:
bundle = ru.get_data()
sales, returns, rep = bundle["sales"], bundle["returns"], bundle["report"]
print("DATA SOURCE :", bundle["source"], "|", f"{len(sales):,} clean rows")
daily = ru.daily_series(sales); TARGET = "quantity"; H = ru.CONFIG["FORECAST_HORIZON"]

In [ ]:
cs = ru.get_customer_sales()
X, y, cutoff = ru.build_churn_dataset(cs, returns)
res = ru.train_churn_models(X, y)
Xtr, Xva, Xte, ytr, yva, yte = res["splits"]
XN = res["xgb_name"]; xgb = res["models"][XN]
print(f"Customers: {len(X)} | features: {X.shape[1]} | churn rate: {y.mean() * 100:.1f}% | label window: last {ru.CONFIG['CHURN_WINDOW_DAYS']} days (cutoff {cutoff.date()})")

In [ ]:
fb = ru.forecast_bundle(daily, TARGET, H); met = fb["metrics"]
runs = []
def log(name, params, metrics):
    rid = ru.mlflow_log("RetailPulse-Week2", name, params, metrics, tags={"stage": "advanced"}); runs.append({"run": name, **metrics}); print(f"logged {name:<26} {str(rid)[:8]}")
for n in ["Prophet", "LSTM", "Hybrid (Prophet+LSTM)"]:
    log(f"forecast_{n.split()[0].lower()}", {"horizon": H, "engines": fb["engines"], "w_prophet": round(fb["weight_prophet"], 2)}, met.loc[n].to_dict())
for n, r in res["metrics"].iterrows():
    log(f"churn_{n.replace(' ', '_').lower()}", {"label_window_days": ru.CONFIG["CHURN_WINDOW_DAYS"]}, r.to_dict())
m_all = ru.get_metrics()
if "tuning" in m_all: log("churn_xgb_tuned", m_all["tuning"].get("xgb_best_params", {}), {"test_auc": m_all["tuning"].get("xgb_tuned_test_auc", float("nan"))})
if "inventory" in m_all: log("inventory_policy", {"service_level": ru.CONFIG["SERVICE_LEVEL"], "lead_time": ru.CONFIG["LEAD_TIME_DAYS"]}, {k: v for k, v in m_all["inventory"].items() if isinstance(v, (int, float))})

In [ ]:
tb = pd.DataFrame(runs).set_index("run")
f = tb.dropna(subset=["MAPE_weekly_%"]); c = tb.dropna(subset=["AUC-ROC"])
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
f["MAPE_weekly_%"].plot(kind="bar", ax=axes[0], color=ru.COLORS[:len(f)]); axes[0].axhline(12, color="red", ls="--", label="target 12%"); axes[0].legend(); axes[0].set_title("Week-2 forecasting runs (weekly MAPE %)"); axes[0].tick_params(axis="x", rotation=25)
c["AUC-ROC"].plot(kind="bar", ax=axes[1], color=ru.COLORS[:len(c)]); axes[1].axhline(0.88, color="red", ls="--", label="target 0.88"); axes[1].legend(); axes[1].set_ylim(0, 1.05); axes[1].set_title("Week-2 churn runs (AUC-ROC)"); axes[1].tick_params(axis="x", rotation=25)
ru.save_fig(fig, "14_01_week2_runs", "MLflow Week-2 runs vs targets", "Dashed red lines show the acceptance targets from the project brief.", "MLOps - MLflow")
df_runs = ru.mlflow_runs(["RetailPulse-Week1", "RetailPulse-Week2"]); print("Total runs tracked in MLflow:", len(df_runs))
ru.log_metrics("mlflow", {"total_runs": int(len(df_runs))})

### Acceptance checklist (honest evaluation)

In [ ]:
M_ = ru.get_metrics(); h = met.loc["Hybrid (Prophet+LSTM)"]; ch = res["metrics"].loc[XN]
inv = M_.get("inventory", {}); seg = M_.get("segmentation", {})
k_ = seg.get("k_chosen"); cr_ = inv.get("cost_reduction_pct")
chk = pd.DataFrame([
    ("F-01", "Automated ETL + data-quality checks", f"{M_.get('cleaning', {}).get('validation_passed', 'n/a')}/{M_.get('cleaning', {}).get('validation_total', 'n/a')} checks passed", True if M_.get("cleaning") else None),
    ("F-02", "6-8 meaningful segments", f"K = {k_}" if k_ else "run notebook 03 first", (6 <= int(k_) <= 8) if k_ else None),
    ("F-03", "MAPE <= 12% (30-day, weekly totals)", f"{h['MAPE_weekly_%']:.1f}% weekly | {h['MAPE_daily_%']:.1f}% daily", bool(h["MAPE_weekly_%"] <= 12)),
    ("F-04a", "AUC-ROC >= 0.88", f"{ch['AUC-ROC']:.3f}", bool(ch["AUC-ROC"] >= 0.88)),
    ("F-04b", "Precision@top-20% >= 0.75", f"{ch['Precision@top20%']:.3f}", bool(ch["Precision@top20%"] >= 0.75)),
    ("F-05", "Overstock/understock cost reduced 25-40%", f"{cr_:.1f}% cost, {inv.get('lost_sales_reduction_pct', float('nan')):.1f}% fewer lost sales" if cr_ is not None else "run notebook 10 first", (cr_ >= 25) if cr_ is not None else None),
    ("F-06", "Streamlit dashboard", "app/streamlit_app.py (5 pages)", True),
    ("NFR", "Drift detection + MLflow", f"{M_.get('mlflow', {}).get('total_runs', 'n/a')} runs, retrain on simulated drift: {M_.get('drift', {}).get('retrain_triggered_on_simulation', 'run notebook 12')}", True)],
    columns=["ID", "Requirement", "Result", "Met"])
display(chk)
colmap = {True: ru.PALETTE["green"], False: ru.PALETTE["orange"], None: ru.PALETTE["grey"]}
fig, ax = plt.subplots(figsize=(10, 4.8)); ax.barh(chk["ID"], np.ones(len(chk)), color=[colmap[m] for m in chk["Met"]])
for i, (r, m) in enumerate(zip(chk["Result"], chk["Met"])): ax.text(0.5, i, ({True: "MET  ", False: "NOT MET  ", None: "NOT RUN  "}[m]) + r, ha="center", va="center", color="white", fontsize=9, fontweight="bold")
ax.set_xticks([]); ax.invert_yaxis(); ax.set_title("Acceptance criteria scorecard (computed from real results)")
ru.save_fig(fig, "14_02_acceptance_scorecard", "Acceptance criteria scorecard", "Green = target met on this dataset; orange = honestly reported as not met; grey = earlier notebook not run yet.", "MLOps - MLflow")
ru.log_metrics("acceptance", {r.ID: {"requirement": r.Requirement, "result": r.Result, "met": (None if r.Met is None else bool(r.Met))} for r in chk.itertuples()})

### Model card

In [ ]:
import json
card = {"project": ru.PROJECT, "author": ru.AUTHOR, "data_source": bundle["source"], "forecast_model": {"type": "Prophet + LSTM weighted ensemble", "weight_prophet": fb["weight_prophet"], "horizon_days": H, "engines": fb["engines"],
        "test_metrics": h.round(3).to_dict()}, "churn_model": {"type": XN, "label": "no purchase in next 90 days", "test_metrics": ch.round(3).to_dict()},
        "intended_use": "Retail demand planning, retention campaigns, replenishment recommendations", "limitations": ["Stock on hand is simulated for public data", "Daily MAPE is high for lumpy wholesale demand", "Retrain when PSI > 0.2 on 2+ features"]}
json.dump(card, open(ru.MODELS / "model_card.json", "w"), indent=2, default=str); print(json.dumps(card, indent=2, default=str)[:1500])

## Summary of Notebook 14
* Week-2 runs in MLflow, **acceptance scorecard computed from real results (never faked)** and a JSON model card. **Week-2 checkpoint reached.**

---
# PART 15 - 15 - Streamlit Dashboard Launch (Colab) & Final DOCX Report
*Week 3-4 - Dashboard & final report*

## Introduction
This notebook (a) writes the **Streamlit dashboard** (Sales, Customer, Forecast, Inventory, Monitoring pages) and shows how to **open it from Google Colab**, and (b) builds the **final project report (DOCX)** automatically from the results of this run, so the numbers always match your real dataset.

In [ ]:
bundle = ru.get_data()
sales, returns, rep = bundle["sales"], bundle["returns"], bundle["report"]
print("DATA SOURCE :", bundle["source"], "|", f"{len(sales):,} clean rows")
daily = ru.daily_series(sales); TARGET = "quantity"; H = ru.CONFIG["FORECAST_HORIZON"]

### Step 1 - Make sure the dashboard input files exist

In [ ]:
need = ["transactions_clean.csv.gz", "forecast_future.csv", "forecast_test.csv", "churn_scores.csv", "rfm_segments.csv", "inventory_plan.csv"]
missing = [n for n in need if not (ru.DATA_PROC / n).exists()]
print("Missing processed files:", missing or "none")
if missing:
    print("Creating the missing files now (this runs the fast parts of the pipeline) ...")
    keep = ["invoice", "invoice_no", "stock_code", "description", "category", "quantity", "unit_price", "revenue", "order_value", "invoice_date", "customer_id", "country"]
    sales[keep].to_csv(ru.DATA_PROC / "transactions_clean.csv.gz", index=False, compression="gzip")
    daily = ru.daily_series(sales); fb = ru.forecast_bundle(daily, "quantity", ru.CONFIG["FORECAST_HORIZON"], lstm_epochs=30)
    fb["future"].to_csv(ru.DATA_PROC / "forecast_future.csv", index=False)
    pd.DataFrame({"ds": fb["test"]["ds"].values, "actual": fb["test"]["quantity"].values, **fb["test_preds"]}).to_csv(ru.DATA_PROC / "forecast_test.csv", index=False)
    cs = ru.get_customer_sales(); rfm = ru.rfm_scores(ru.rfm_table(cs)); ks, X = ru.kmeans_search(rfm, range(6, 9)); K = int(ks.loc[ks["silhouette"].idxmax(), "k"])
    seg_df, _, _ = ru.fit_kmeans(rfm, X, K); seg_df.reset_index().to_csv(ru.DATA_PROC / "rfm_segments.csv", index=False)
    Xc, yc, _ = ru.build_churn_dataset(cs, bundle["returns"]); rr = ru.train_churn_models(Xc, yc)
    ru.score_all_customers(rr["models"][rr["xgb_name"]], cs, bundle["returns"]).to_csv(ru.DATA_PROC / "churn_scores.csv", index=False)
    ru.inventory_plan(sales).to_csv(ru.DATA_PROC / "inventory_plan.csv", index=False)
print("Files:", sorted(p.name for p in ru.DATA_PROC.glob("*") if p.suffix in (".csv", ".gz")))

### Step 2 - Write the Streamlit app

In [ ]:
%%writefile streamlit_app.py
"""
RetailPulse - Streamlit Dashboard (5 pages)
Run:  streamlit run app/streamlit_app.py
Author: Santosh Kumar | Zidio Development - Data Science & Analytics (March 2026)

Data order: (1) file uploaded in sidebar  (2) env RETAILPULSE_DATA folder  (3) ./retailpulse_work/data/processed
            (4) ./data/processed (shipped demo)  (5) built-in demo data.
"""
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

HERE = Path(__file__).resolve().parent
for p in (HERE, HERE.parent / "src", Path.cwd(), Path.cwd() / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import retailpulse_utils as ru  # noqa: E402

st.set_page_config(page_title="RetailPulse | Santosh Kumar", page_icon="📊", layout="wide")

CANDIDATE_DIRS = [Path(os.environ["RETAILPULSE_DATA"])] if os.environ.get("RETAILPULSE_DATA") else []
CANDIDATE_DIRS += [Path.cwd() / "retailpulse_work" / "data" / "processed", HERE.parent / "data" / "processed", Path.cwd() / "data" / "processed"]


def find_dir():
    for d in CANDIDATE_DIRS:
        if (d / "transactions_clean.csv.gz").exists():
            return d
    return None


@st.cache_data(show_spinner="Loading transactions ...")
def load_transactions(path, uploaded_bytes=None, uploaded_name=None):
    if uploaded_bytes is not None:
        import io
        buf = io.BytesIO(uploaded_bytes)
        raw = pd.read_csv(buf) if uploaded_name.lower().endswith(".csv") else pd.read_excel(buf)
        sales, _, _ = ru.clean(ru.standardize(raw), verbose=False)
        return sales
    if path:
        df = pd.read_csv(Path(path) / "transactions_clean.csv.gz", parse_dates=["invoice_date"])
        d = df["invoice_date"]
        df["date"] = d.dt.normalize()
        df["year_month"] = d.dt.to_period("M").astype(str)
        df["month_name"] = d.dt.strftime("%b")
        df["dow"], df["hour"] = d.dt.dayofweek, d.dt.hour
        df["month"], df["year"], df["day"] = d.dt.month, d.dt.year, d.dt.day
        df["is_weekend"] = (df["dow"] >= 5).astype(int)
        df["week"] = d.dt.isocalendar().week.astype(int)
        df["quarter"] = d.dt.quarter
        return df
    sales, _, _ = ru.clean(ru.standardize(ru.generate_demo_data()), verbose=False)
    return sales


@st.cache_data(show_spinner="Segmenting customers ...")
def customer_tables(_sales, key):
    cs = _sales[_sales["customer_id"].notna()]
    rfm = ru.rfm_scores(ru.rfm_table(cs))
    ks, X = ru.kmeans_search(rfm, range(6, 9))
    K = int(ks.loc[ks["silhouette"].idxmax(), "k"])
    seg, _, _ = ru.fit_kmeans(rfm, X, K)
    return rfm, seg.reset_index(), K


@st.cache_data(show_spinner="Scoring churn risk ...")
def churn_table(_sales, key):
    cs = _sales[_sales["customer_id"].notna()]
    X, y, _ = ru.build_churn_dataset(cs, None)
    res = ru.train_churn_models(X, y)
    model = res["models"][res["xgb_name"]]
    return ru.score_all_customers(model, cs, None), res["metrics"].loc[res["xgb_name"]]


@st.cache_data(show_spinner="Forecasting ...")
def live_forecast(_sales, key, horizon):
    daily = ru.daily_series(_sales)
    fut_dates = pd.date_range(daily["ds"].max() + pd.Timedelta(days=1), periods=horizon, freq="D")
    p = ru.holt_winters_forecast(daily, fut_dates, "quantity")
    sd = float(daily["quantity"].tail(60).std())
    return pd.DataFrame({"ds": fut_dates, "hybrid": p, "lower": np.clip(p - 1.28 * sd, 0, None), "upper": p + 1.28 * sd})


# ------------------------------------------------------------------ sidebar
st.sidebar.title("📊 RetailPulse")
st.sidebar.caption("AI-Powered Customer Analytics & Demand Forecasting\n\nMade by **Santosh Kumar**")
page = st.sidebar.radio("Dashboard page", ["1 · Sales", "2 · Customers", "3 · Forecast", "4 · Inventory", "5 · Monitoring", "ℹ️ About"])
up = st.sidebar.file_uploader("Optional: upload your retail CSV / XLSX", type=["csv", "xlsx", "xls"])
data_dir = find_dir()
try:
    if up is not None:
        sales = load_transactions(None, up.getvalue(), up.name)
    else:
        sales = load_transactions(str(data_dir) if data_dir else None)
except Exception as e:  # noqa
    st.sidebar.error(f"Upload could not be read ({e}). Showing demo data.")
    up = None
    sales = load_transactions(str(data_dir) if data_dir else None)
key = f"{len(sales)}_{sales['revenue'].sum():.0f}"
st.sidebar.success(f"{len(sales):,} rows | {sales['date'].min().date()} → {sales['date'].max().date()}")

st.sidebar.markdown("### Filters")
d0, d1 = sales["date"].min().date(), sales["date"].max().date()
date_rng = st.sidebar.date_input("Date range", (d0, d1), min_value=d0, max_value=d1)
countries = st.sidebar.multiselect("Country", sorted(sales["country"].unique()), default=[])
cats = st.sidebar.multiselect("Category", sorted(sales["category"].unique()), default=[])
f = sales.copy()
if isinstance(date_rng, tuple) and len(date_rng) == 2:
    f = f[(f["date"].dt.date >= date_rng[0]) & (f["date"].dt.date <= date_rng[1])]
if countries:
    f = f[f["country"].isin(countries)]
if cats:
    f = f[f["category"].isin(cats)]

# ------------------------------------------------------------------ page 1
if page.startswith("1"):
    st.title("🛒 Sales Dashboard")
    if f.empty:
        st.warning("No rows for the selected filters.")
        st.stop()
    c = st.columns(5)
    orders = f["invoice"].nunique()
    c[0].metric("Revenue", f"{f['revenue'].sum():,.0f}")
    c[1].metric("Orders", f"{orders:,}")
    c[2].metric("Avg order value", f"{f['revenue'].sum() / max(orders, 1):,.1f}")
    c[3].metric("Customers", f"{f['customer_id'].nunique():,}")
    c[4].metric("Units sold", f"{int(f['quantity'].sum()):,}")
    m = f.groupby("year_month").agg(revenue=("revenue", "sum"), orders=("invoice", "nunique")).reset_index()
    fig = go.Figure()
    fig.add_bar(x=m["year_month"], y=m["revenue"], name="Revenue", marker_color="#0EA5E9")
    fig.add_scatter(x=m["year_month"], y=m["orders"], name="Orders", yaxis="y2", line=dict(color="#F59E0B", width=3))
    fig.update_layout(title="Monthly revenue and orders", yaxis2=dict(overlaying="y", side="right"), height=420, legend=dict(orientation="h"))
    st.plotly_chart(fig, width="stretch")
    a, b = st.columns(2)
    top = f.groupby("description")["revenue"].sum().nlargest(10).reset_index()
    a.plotly_chart(px.bar(top, x="revenue", y="description", orientation="h", title="Top 10 products", color_discrete_sequence=["#14B8A6"]).update_layout(yaxis=dict(autorange="reversed")), width="stretch")
    cat = f.groupby("category")["revenue"].sum().reset_index()
    b.plotly_chart(px.pie(cat, names="category", values="revenue", hole=0.5, title="Revenue by category"), width="stretch")
    a, b = st.columns(2)
    ct = f.groupby("country")["revenue"].sum().nlargest(10).reset_index()
    a.plotly_chart(px.bar(ct, x="country", y="revenue", title="Top countries", color_discrete_sequence=["#6366F1"]), width="stretch")
    hm = pd.crosstab(f["dow"], f["hour"]).reindex(range(7), fill_value=0)
    hm.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    b.plotly_chart(px.imshow(hm, aspect="auto", color_continuous_scale="YlGnBu", title="Order lines: weekday × hour"), width="stretch")
    st.download_button("⬇️ Download filtered data (CSV)", f.to_csv(index=False).encode(), "retailpulse_filtered_sales.csv", "text/csv")

# ------------------------------------------------------------------ page 2
elif page.startswith("2"):
    st.title("👥 Customer Dashboard — Segmentation & Churn")
    if sales["customer_id"].nunique() < 50:
        st.warning("Too few identified customers in this dataset.")
        st.stop()
    rfm, seg, K = customer_tables(sales, key)
    scores, met = churn_table(sales, key)
    c = st.columns(4)
    c[0].metric("Customers", f"{len(rfm):,}")
    c[1].metric("Segments (K-Means)", K)
    c[2].metric("High-risk customers", f"{int((scores['churn_probability'] > 0.7).sum()):,}")
    c[3].metric("Churn model AUC", f"{met['AUC-ROC']:.3f}")
    t1, t2, t3 = st.tabs(["RFM segments", "Churn risk", "Customer lookup"])
    with t1:
        prof = seg.groupby("segment").agg(customers=("monetary", "size"), recency=("recency", "mean"), frequency=("frequency", "mean"), monetary=("monetary", "mean"), revenue=("monetary", "sum")).reset_index()
        a, b = st.columns(2)
        a.plotly_chart(px.bar(prof.sort_values("customers"), x="customers", y="segment", orientation="h", title="Customers per segment", color_discrete_sequence=["#14B8A6"]), width="stretch")
        b.plotly_chart(px.pie(prof, names="segment", values="revenue", hole=0.45, title="Revenue share by segment"), width="stretch")
        st.plotly_chart(px.scatter(seg, x="recency", y="monetary", size="frequency", color="segment", log_y=True, title="Recency vs monetary (bubble = frequency)", opacity=0.7), width="stretch")
        st.dataframe(prof.round(1), width="stretch")
        st.download_button("⬇️ Download segments (CSV)", seg.to_csv(index=False).encode(), "retailpulse_segments.csv", "text/csv")
    with t2:
        a, b = st.columns(2)
        a.plotly_chart(px.histogram(scores, x="churn_probability", nbins=40, color_discrete_sequence=["#EF4444"], title="Churn probability distribution"), width="stretch")
        rb = scores["risk_band"].astype(str).value_counts().reset_index()
        rb.columns = ["risk_band", "customers"]
        b.plotly_chart(px.pie(rb, names="risk_band", values="customers", hole=0.5, title="Risk bands", color="risk_band", color_discrete_map={"Low": "#22C55E", "Medium": "#F59E0B", "High": "#EF4444"}), width="stretch")
        th = st.slider("Show customers with churn probability above", 0.0, 1.0, 0.7, 0.05)
        top = scores[scores["churn_probability"] >= th].sort_values("churn_probability", ascending=False)
        st.write(f"**{len(top):,} customers** above {th:.2f}")
        st.dataframe(top.head(200).round(3), width="stretch")
        st.download_button("⬇️ Download at-risk list (CSV)", top.to_csv(index=False).encode(), "retailpulse_at_risk_customers.csv", "text/csv")
    with t3:
        cid = st.selectbox("Customer ID", scores["customer_id"].astype(int).tolist()[:2000])
        row = scores[scores["customer_id"] == cid].iloc[0]
        s_ = seg[seg["customer_id"] == cid]
        st.json({"customer_id": int(cid), "churn_probability": round(float(row["churn_probability"]), 3), "risk_band": str(row["risk_band"]),
                 "recency_days": int(row["recency"]), "orders": int(row["frequency"]), "spend": round(float(row["monetary"]), 2),
                 "segment": None if s_.empty else str(s_["segment"].iloc[0])})

# ------------------------------------------------------------------ page 3
elif page.startswith("3"):
    st.title("🔮 Demand Forecast Dashboard")
    fut_file = (data_dir / "forecast_future.csv") if data_dir else None
    test_file = (data_dir / "forecast_test.csv") if data_dir else None
    daily = ru.daily_series(sales)
    horizon = st.slider("Forecast horizon (days)", 7, 60, 30)
    if fut_file is not None and fut_file.exists() and up is None:
        fut = pd.read_csv(fut_file, parse_dates=["ds"]).head(horizon)
        st.success("Showing the **Prophet + LSTM hybrid** forecast produced by the notebooks.")
    else:
        fut = live_forecast(sales, key, horizon)
        st.info("Showing a fast **Holt-Winters** forecast computed live. Run the notebooks for the Prophet + LSTM hybrid.")
    st.subheader("What-if analysis")
    a, b, c_ = st.columns(3)
    promo = a.slider("Promotion uplift %", -30, 80, 0)
    price = b.slider("Price change %", -20, 30, 0)
    elasticity = c_.slider("Price elasticity", -3.0, 0.0, -1.2, 0.1)
    mult = (1 + promo / 100) * (1 + price / 100) ** elasticity
    adj = fut.copy()
    for col in ("hybrid", "lower", "upper"):
        adj[col] = adj[col] * mult
    hist = daily.tail(90)
    fig = go.Figure()
    fig.add_scatter(x=hist["ds"], y=hist["quantity"], name="History", line=dict(color="#1F3A5F"))
    fig.add_scatter(x=fut["ds"], y=fut["upper"], line=dict(width=0), showlegend=False)
    fig.add_scatter(x=fut["ds"], y=fut["lower"], fill="tonexty", fillcolor="rgba(239,68,68,0.15)", line=dict(width=0), name="80% interval")
    fig.add_scatter(x=fut["ds"], y=fut["hybrid"], name="Baseline forecast", line=dict(color="#EF4444", width=3))
    if abs(mult - 1) > 1e-6:
        fig.add_scatter(x=adj["ds"], y=adj["hybrid"], name="What-if scenario", line=dict(color="#14B8A6", width=3, dash="dash"))
    fig.update_layout(title="Demand forecast (units per day)", height=470, legend=dict(orientation="h"))
    st.plotly_chart(fig, width="stretch")
    c = st.columns(3)
    c[0].metric("Baseline demand", f"{fut['hybrid'].sum():,.0f} units")
    c[1].metric("Scenario demand", f"{adj['hybrid'].sum():,.0f} units", f"{(mult - 1) * 100:+.1f}%")
    c[2].metric("Demand multiplier", f"{mult:.3f}")
    if test_file is not None and test_file.exists() and up is None:
        tt = pd.read_csv(test_file, parse_dates=["ds"])
        st.subheader("Back-test on the last 30 days")
        cols = [x for x in tt.columns if x != "ds"]
        st.plotly_chart(px.line(tt, x="ds", y=cols, title="Actual vs model forecasts"), width="stretch")
        rows = [{"model": x, "WAPE %": ru.wape(tt["actual"], tt[x]), "MAPE weekly %": ru.block_mape(tt["actual"], tt[x]), "MAPE daily %": ru.mape(tt["actual"], tt[x])} for x in cols if x != "actual"]
        st.dataframe(pd.DataFrame(rows).round(2), width="stretch")
    st.download_button("⬇️ Download forecast (CSV)", adj.to_csv(index=False).encode(), "retailpulse_forecast.csv", "text/csv")

# ------------------------------------------------------------------ page 4
elif page.startswith("4"):
    st.title("📦 Inventory Optimization")
    st.caption("Current stock is **simulated** because public retail data has no stock-on-hand column. Connect your real stock table in production.")
    a, b, c_, d = st.columns(4)
    L = a.slider("Lead time (days)", 1, 21, ru.CONFIG["LEAD_TIME_DAYS"])
    R = b.slider("Review period (days)", 1, 14, ru.CONFIG["REVIEW_PERIOD_DAYS"])
    SL = c_.slider("Service level %", 80, 99, 95)
    growth = d.slider("Demand growth vs history", 0.5, 2.0, 1.0, 0.05)
    plan = ru.inventory_plan(sales, growth_ratio=growth, lead_time=L, review=R, service_level=SL / 100, top_n=40)
    c = st.columns(4)
    c[0].metric("SKUs analysed", len(plan))
    c[1].metric("Reorder now", int((plan["status"] == "Reorder Now").sum()))
    c[2].metric("Overstocked", int((plan["status"] == "Overstock").sum()))
    c[3].metric("Recommended order value", f"{plan['order_value'].sum():,.0f}")
    cmap = {"Reorder Now": "#EF4444", "Healthy": "#22C55E", "Overstock": "#F59E0B"}
    a, b = st.columns(2)
    a.plotly_chart(px.pie(plan, names="status", hole=0.5, title="Stock status", color="status", color_discrete_map=cmap), width="stretch")
    b.plotly_chart(px.scatter(plan, x="reorder_point", y="current_stock_simulated", color="status", hover_name="description", title="Stock vs reorder point", color_discrete_map=cmap), width="stretch")
    st.plotly_chart(px.bar(plan.nlargest(15, "order_value"), x="order_value", y="description", orientation="h", title="Top recommended purchase orders", color_discrete_sequence=["#14B8A6"]).update_layout(yaxis=dict(autorange="reversed")), width="stretch")
    st.dataframe(plan.round(1), width="stretch")
    st.download_button("⬇️ Download reorder plan (CSV)", plan.to_csv(index=False).encode(), "retailpulse_inventory_plan.csv", "text/csv")

# ------------------------------------------------------------------ page 5
elif page.startswith("5"):
    st.title("🛰️ Monitoring — Data Drift")
    cut = sales["date"].min() + (sales["date"].max() - sales["date"].min()) * 0.7
    ref, cur = sales[sales["date"] <= cut], sales[sales["date"] > cut]
    st.write(f"Reference: **{ref['date'].min().date()} → {ref['date'].max().date()}** | Current: **{cur['date'].min().date()} → {cur['date'].max().date()}**")
    sim = st.checkbox("Simulate drift (price +35%, quantity −40%)")
    cur2 = cur.copy()
    if sim:
        cur2["unit_price"] = cur2["unit_price"] * 1.35
        cur2["quantity"] = np.maximum(1, (cur2["quantity"] * 0.6).round())
        cur2["revenue"] = cur2["quantity"] * cur2["unit_price"]
    d = ru.drift_table(ref, cur2, ["quantity", "unit_price", "revenue", "hour"])
    st.plotly_chart(px.bar(d, x="feature", y="PSI", color="status", title="Population Stability Index", color_discrete_map={"OK": "#22C55E", "WARNING": "#F59E0B", "DRIFT": "#EF4444"}), width="stretch")
    st.dataframe(d.round(4), width="stretch")
    dec = ru.retrain_decision(d, 0.0)
    if dec["retrain"]:
        st.error("RETRAIN RECOMMENDED: " + "; ".join(dec["reasons"]))
    else:
        st.success("Model is stable - no retraining required.")

# ------------------------------------------------------------------ about
else:
    st.title("ℹ️ About RetailPulse")
    st.markdown("""
**RetailPulse – AI-Powered Customer Analytics & Demand Forecasting Platform**
*Made by **Santosh Kumar** · Zidio Development – Data Science & Analytics · March 2026*

**Pages:** 1 Sales · 2 Customers (RFM + churn) · 3 Forecast (Prophet + LSTM, what-if) · 4 Inventory (ROP / safety stock / EOQ) · 5 Monitoring (PSI drift).

**How to use:** pick filters in the sidebar, or upload your own retail file with columns like *Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID, Country*.

**Stack:** Python · Pandas · Scikit-learn · XGBoost · Prophet · PyTorch LSTM · MLflow · Streamlit · FastAPI · Docker · Kubernetes.
""")

### Step 3 - Open the dashboard from Google Colab
Colab cannot show `localhost` directly, so we use a free **localtunnel** (no account / key needed). Run the next cell, copy the **URL**, and when the page asks for a *tunnel password* paste the IP address printed below.

In [ ]:
import subprocess, time, os, urllib.request, shutil, sys
os.environ["RETAILPULSE_DATA"] = str(ru.DATA_PROC)
LAUNCH = False        # <- set True in Colab to start the dashboard
if LAUNCH:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit", "plotly"], check=False)
    subprocess.Popen([sys.executable, "-m", "streamlit", "run", "streamlit_app.py", "--server.port", "8501", "--server.headless", "true"], stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(6)
    try:
        print("Tunnel password (your public IP):", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())
    except Exception: pass
    if shutil.which("npx"):
        subprocess.Popen("npx --yes localtunnel --port 8501", shell=True)
        print("Look for 'your url is: https://....loca.lt' in the output above / below.")
    else:
        print("Node/npx not found. Alternative: !pip install pyngrok and use an ngrok token, or deploy on Streamlit Community Cloud (see README).")
else:
    print("Set LAUNCH = True and re-run this cell in Colab to start the dashboard.\nLocal PC: pip install -r requirements.txt  &&  streamlit run app/streamlit_app.py")

### Step 4 - Build the final project report (DOCX)

In [ ]:
%%writefile retailpulse_report.py
"""
RetailPulse - automatic project report (DOCX) generator.
Reads outputs/metrics.json + outputs/charts_index.json produced by the notebooks, so every number
and every chart in the report comes from the real run (nothing is hard-coded).

    import retailpulse_report as rr
    rr.build_report("RetailPulse_Project_Report_FULL.docx", compact=False)   # every chart, long
    rr.build_report("RetailPulse_Zidio_Submission_Report.docx", compact=True)  # ~15 pages for the Zidio PDF

Author: Santosh Kumar
"""
from __future__ import annotations

import io
import json
import subprocess
import sys
from pathlib import Path

try:
    import docx  # noqa: F401
except ImportError:  # Colab: install on demand
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-docx"], check=False)

from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_BREAK
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

import retailpulse_utils as ru

AUTHOR = "Santosh Kumar"
NAVY, TEAL, GREY = RGBColor(0x1F, 0x3A, 0x5F), RGBColor(0x0F, 0x76, 0x6E), RGBColor(0x64, 0x74, 0x8B)

SECTION_TEXT = {
    "EDA": ("Exploratory Data Analysis (Notebook 01)",
            "EDA was the first step: understand the raw table, its quality problems, the seasonality of demand, where revenue comes from "
            "(products, categories, countries, customers) and how variables relate. Every later model depends on these findings."),
    "Cleaning & Feature Engineering": ("Data Cleaning, Validation & Feature Engineering (Notebook 02)",
            "Cancellations, invalid prices, non-product codes, duplicates and extreme outliers were removed with a documented waterfall. "
            "Ten validation expectations (Great-Expectations style) were checked. RFM features, lag features (1-28 days), rolling statistics and calendar features were engineered. "
            "Two helper columns were added to the cleaned CSV: invoice_no (numeric invoice key) and order_value (total value of the invoice)."),
    "Customer Segmentation": ("Customer Segmentation (Notebook 03)",
            "Customers were segmented with RFM scoring (rule-based), K-Means (K chosen with silhouette inside the business range 6-8) and DBSCAN (outlier discovery). "
            "Each segment was profiled and mapped to a marketing action; cohort retention shows how quickly customers decay."),
    "Time-Series Preparation": ("Time-Series Preparation (Notebook 04)",
            "A continuous daily demand series was built, closed weekdays detected, the series decomposed (classical + STL), stationarity tested (ADF / KPSS), "
            "ACF/PACF inspected, outliers flagged and a leakage-free train / validation / test split of 30 days each created."),
    "Forecasting - Prophet": ("Baseline Forecasting with Prophet (Notebook 05)",
            "Prophet with weekly + yearly seasonality and UK holidays was fitted as the baseline, compared with Seasonal Naive and Holt-Winters, diagnosed with residual plots and tested for hyper-parameter sensitivity."),
    "Forecasting - LSTM": ("Deep-Learning Forecasting with LSTM (Notebook 06)",
            "A PyTorch LSTM uses a 28-day window plus known-future calendar covariates and produces a recursive 30-day forecast with early stopping."),
    "MLOps - MLflow": ("Experiment Tracking with MLflow (Notebooks 07 and 14)",
            "All baseline and advanced runs were logged (parameters, metrics, artifacts) in a local SQLite-backed MLflow store, compared against the acceptance targets and summarised in a scorecard."),
    "Forecasting - Hybrid": ("Hybrid Prophet + LSTM Ensemble (Notebook 08)",
            "The blend weight was optimised on the validation window; models were re-fitted on train + validation to score the test window and finally refit on all data for the 30-day production forecast."),
    "Churn Prediction": ("Churn Prediction with XGBoost + SHAP (Notebook 09)",
            "A leakage-free churn label (no purchase in the next 90 days) was modelled with Logistic Regression, Random Forest and XGBoost. Performance, calibration, gain/lift and SHAP explanations are shown."),
    "Inventory Optimization": ("Inventory Optimization (Notebook 10)",
            "Safety stock, reorder point, EOQ and order-up-to level were computed per SKU from forecast demand and a 30-day policy back-test compared a naive buffer rule with the forecast-driven policy."),
    "Tuning": ("Hyper-parameter Tuning with Optuna (Notebook 11)",
            "Bayesian (TPE) optimisation tuned XGBoost (cross-validated AUC) and Prophet (validation WAPE); results were compared with the defaults on the untouched test set."),
    "Drift Detection": ("Drift Detection & Monitoring (Notebook 12)",
            "PSI and KS tests compare a reference and a current period; an injected price / demand shock demonstrates drift detection and the automatic retrain trigger."),
    "Model Refinement": ("Model Refinement & Retraining Pipeline (Notebook 13)",
            "Calibration, profit-optimal thresholding, feature-group ablation, ensembling, learning curves and weekday bias correction were evaluated; an Airflow DAG automates retraining."),
}
SECTION_ORDER = ["EDA", "Cleaning & Feature Engineering", "Customer Segmentation", "Time-Series Preparation", "Forecasting - Prophet",
                 "Forecasting - LSTM", "Forecasting - Hybrid", "Churn Prediction", "Inventory Optimization", "Tuning", "Drift Detection",
                 "Model Refinement", "MLOps - MLflow"]

COMPACT_KEYS = ["00_architecture_diagram", "01_03_monthly_revenue_orders", "01_15_pareto", "02_01_cleaning_waterfall", "03_06_pca_clusters",
                "03_08_cluster_profile_heatmap", "04_05_decomposition", "05_01_prophet_forecast", "08_02_all_models_test", "08_06_future_forecast",
                "09_05_roc_curves", "09_12_shap_summary", "10_08_backtest", "12_03_psi_simulated", "14_02_acceptance_scorecard", "11_04_default_vs_tuned"]


# ----------------------------------------------------------------------------------- helpers
def g(d, *keys, default="n/a"):
    for k in keys:
        if isinstance(d, dict) and k in d:
            d = d[k]
        else:
            return default
    return d


def fnum(x, nd=2, default="n/a"):
    try:
        return f"{float(x):,.{nd}f}"
    except Exception:
        return default


def shade(cell, hex_color):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear")
    shd.set(qn("w:color"), "auto")
    shd.set(qn("w:fill"), hex_color)
    tcPr.append(shd)


def add_field(paragraph, instr):
    run = paragraph.add_run()
    for t, txt in (("begin", None), (None, instr), ("end", None)):
        if t:
            el = OxmlElement("w:fldChar")
            el.set(qn("w:fldCharType"), t)
        else:
            el = OxmlElement("w:instrText")
            el.set(qn("xml:space"), "preserve")
            el.text = txt
        run._r.append(el)


def heading(doc, text, level=1):
    h = doc.add_heading(text, level=level)
    for r in h.runs:
        r.font.color.rgb = NAVY if level <= 2 else TEAL
        r.font.name = "Calibri"
    return h


def para(doc, text, bold=False, italic=False, size=10.5, color=None, align=None, space_after=6):
    p = doc.add_paragraph()
    r = p.add_run(text)
    r.bold, r.italic = bold, italic
    r.font.size = Pt(size)
    if color is not None:
        r.font.color.rgb = color
    if align:
        p.alignment = align
    p.paragraph_format.space_after = Pt(space_after)
    return p


def bullets(doc, items, size=10.5):
    for it in items:
        p = doc.add_paragraph(style="List Bullet")
        r = p.add_run(it)
        r.font.size = Pt(size)
        p.paragraph_format.space_after = Pt(2)


def table(doc, header, rows, widths=None, font=9, head_fill="1F3A5F"):
    t = doc.add_table(rows=1, cols=len(header))
    t.style = "Table Grid"
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    for i, h in enumerate(header):
        c = t.rows[0].cells[i]
        c.text = ""
        r = c.paragraphs[0].add_run(str(h))
        r.bold, r.font.size, r.font.color.rgb = True, Pt(font), RGBColor(255, 255, 255)
        shade(c, head_fill)
    for ri, row in enumerate(rows):
        cells = t.add_row().cells
        for i, v in enumerate(row):
            cells[i].text = ""
            r = cells[i].paragraphs[0].add_run(str(v))
            r.font.size = Pt(font)
            if ri % 2 == 1:
                shade(cells[i], "F1F5F9")
    if widths:
        for row in t.rows:
            for i, w in enumerate(widths):
                row.cells[i].width = Inches(w)
    doc.add_paragraph().paragraph_format.space_after = Pt(4)
    return t


def df_rows(df, nd=2):
    return [[str(idx)] + [fnum(v, nd) if isinstance(v, (int, float)) else str(v) for v in row] for idx, row in zip(df.index, df.values.tolist())]


def jpg_bytes(path, max_w=1500, q=86):
    from PIL import Image
    im = Image.open(path).convert("RGB")
    if im.width > max_w:
        im = im.resize((max_w, int(im.height * max_w / im.width)))
    buf = io.BytesIO()
    im.save(buf, "JPEG", quality=q, optimize=True)
    buf.seek(0)
    return buf


class Fig:
    n = 0


def figure(doc, key, idx, width=6.3, caption=True):
    meta = idx.get(key)
    if not meta or not Path(meta["file"]).exists():
        return False
    Fig.n += 1
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.add_run().add_picture(jpg_bytes(meta["file"]), width=Inches(width))
    p.paragraph_format.space_after = Pt(2)
    p.paragraph_format.keep_with_next = True
    if caption:
        c = doc.add_paragraph()
        c.alignment = WD_ALIGN_PARAGRAPH.CENTER
        r = c.add_run(f"Figure {Fig.n}: {meta['title']}")
        r.bold, r.font.size = True, Pt(9)
        if meta.get("note"):
            c2 = doc.add_paragraph()
            c2.alignment = WD_ALIGN_PARAGRAPH.CENTER
            r2 = c2.add_run(meta["note"])
            r2.italic, r2.font.size, r2.font.color.rgb = True, Pt(8.5), GREY
            c2.paragraph_format.space_after = Pt(8)
    return True


def figure_grid(doc, keys, idx, per_row=2, width=3.15):
    keys = [k for k in keys if k in idx and Path(idx[k]["file"]).exists()]
    for i in range(0, len(keys), per_row):
        t = doc.add_table(rows=2, cols=per_row)
        t.alignment = WD_TABLE_ALIGNMENT.CENTER
        for j, k in enumerate(keys[i:i + per_row]):
            Fig.n += 1
            cell = t.rows[0].cells[j]
            cell.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
            cell.paragraphs[0].add_run().add_picture(jpg_bytes(idx[k]["file"], 1100), width=Inches(width))
            cap = t.rows[1].cells[j]
            cap.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
            r = cap.paragraphs[0].add_run(f"Fig. {Fig.n}: {idx[k]['title']}")
            r.bold, r.font.size = True, Pt(8)
            if idx[k].get("note"):
                r2 = cap.add_paragraph().add_run(idx[k]["note"][:170])
                r2.italic, r2.font.size, r2.font.color.rgb = True, Pt(7.5), GREY
        doc.add_paragraph().paragraph_format.space_after = Pt(2)


def page_break(doc):
    doc.add_paragraph().add_run().add_break(WD_BREAK.PAGE)


# ----------------------------------------------------------------------------------- main
def build_report(out_path: str, compact: bool = False) -> str:
    Fig.n = 0
    M = ru.get_metrics()
    idx = json.loads(ru.CHART_INDEX.read_text()) if ru.CHART_INDEX.exists() else {}
    data, fc, hy, ch = M.get("data", {}), M.get("forecast", {}), M.get("hybrid", {}), M.get("churn", {})
    inv, seg, tun, drift, clean_, ts = M.get("inventory", {}), M.get("segmentation", {}), M.get("tuning", {}), M.get("drift", {}), M.get("cleaning", {}), M.get("timeseries", {})
    acc = M.get("acceptance", {})

    doc = Document()
    sec = doc.sections[0]
    sec.page_width, sec.page_height = Inches(8.27), Inches(11.69)   # A4
    sec.left_margin = sec.right_margin = Inches(0.8)
    sec.top_margin = sec.bottom_margin = Inches(0.8)
    st = doc.styles["Normal"]
    st.font.name, st.font.size = "Calibri", Pt(10.5)

    hp = sec.header.paragraphs[0]
    hp.text = f"RetailPulse  |  Project made by {AUTHOR}  |  Zidio Development - Data Science & Analytics"
    hp.runs[0].font.size, hp.runs[0].font.color.rgb = Pt(8), GREY
    fp = sec.footer.paragraphs[0]
    fp.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = fp.add_run("Page ")
    r.font.size = Pt(8)
    add_field(fp, "PAGE")

    # ---------------- cover
    if "00_project_banner" in idx and Path(idx["00_project_banner"]["file"]).exists():
        p = doc.add_paragraph()
        p.add_run().add_picture(jpg_bytes(idx["00_project_banner"]["file"]), width=Inches(6.6))
    para(doc, "RetailPulse", bold=True, size=34, color=NAVY, space_after=0)
    para(doc, "AI-Powered Customer Analytics & Demand Forecasting Platform", bold=True, size=17, color=TEAL, space_after=4)
    para(doc, "Predictive Demand  •  Customer Segmentation  •  Churn Analysis  •  Inventory Optimization", size=11, color=GREY, space_after=18)
    table(doc, ["Item", "Details"], [
        ["Project made by", AUTHOR],
        ["Prepared for", "Zidio Development - Data Science & Analytics Domain"],
        ["Document", "Project Report (compact Zidio submission edition)" if compact else "Complete Project Report (every chart explained)"],
        ["Date / Version", "March 2026  |  2.0 - Industry Edition"],
        ["Dataset used", str(g(data, "source"))],
        ["Data period", f"{g(data, 'date_min')}  to  {g(data, 'date_max')}"],
    ], widths=[1.8, 4.8], font=10)
    para(doc, "Tagline: From raw transactions to forecasts, retention actions and replenishment plans - in one reproducible, monitored pipeline.", italic=True, color=GREY)
    page_break(doc)

    # ---------------- project summary
    heading(doc, "Project Summary", 1)
    para(doc, "RetailPulse is an end-to-end Data Science & Analytics platform that helps retail businesses make smarter decisions. "
              "It ingests sales, customer and inventory data and delivers four solutions: (1) demand forecasting for the next 30 days with a Prophet + LSTM hybrid, "
              "(2) customer segmentation with RFM, K-Means and DBSCAN, (3) churn prediction with XGBoost and SHAP explainability, and (4) inventory optimisation "
              "(safety stock, reorder point, EOQ and recommended order quantities). Results are delivered through a 5-page Streamlit dashboard, a FastAPI service and a "
              "12-page Power BI report, and the system follows MLOps practice: MLflow tracking, PSI/KS drift detection, an Airflow retraining DAG, Docker, Kubernetes and GitHub Actions CI/CD.")
    heading(doc, "Key results of this run (computed, not hard-coded)", 2)
    h = hy if hy else {}
    cm = ch if ch else {}
    table(doc, ["Area", "Metric", "Result", "Target"], [
        ["Data", "Clean rows / customers / products", f"{fnum(g(data, 'final_rows', default=None), 0)} / {fnum(g(data, 'customers', default=None), 0)} / {fnum(g(data, 'products', default=None), 0)}", "-"],
        ["Data", "Total revenue", fnum(g(data, "total_revenue", default=None), 0), "-"],
        ["Segmentation", "K-Means clusters (silhouette)", f"{g(seg, 'k_chosen')} ({fnum(g(seg, 'silhouette_kmeans', default=None), 3)})", "6-8 segments"],
        ["Forecasting", "Hybrid MAPE weekly totals", f"{fnum(g(h, 'MAPE_weekly_%', default=None), 1)} %", "<= 12 %"],
        ["Forecasting", "Hybrid MAPE daily / WAPE", f"{fnum(g(h, 'MAPE_daily_%', default=None), 1)} % / {fnum(g(h, 'WAPE_%', default=None), 1)} %", "informational"],
        ["Churn", "XGBoost AUC-ROC", fnum(g(cm, "AUC-ROC", default=None), 3), ">= 0.88"],
        ["Churn", "Precision @ top 20 %", fnum(g(cm, "Precision@top20%", default=None), 3), ">= 0.75"],
        ["Inventory", "Cost reduction / lost-sales reduction (back-test)", f"{fnum(g(inv, 'cost_reduction_pct', default=None), 1)} % / {fnum(g(inv, 'lost_sales_reduction_pct', default=None), 1)} %", "25-40 %"],
        ["MLOps", "Drift simulation triggers retrain", str(g(drift, "retrain_triggered_on_simulation")), "Yes"],
    ], widths=[1.1, 2.6, 1.8, 1.1], font=9)
    para(doc, "Honesty note: targets are taken from the project brief. Where a target is not met on the dataset used, the report states it openly "
              "(see the acceptance scorecard). Daily MAPE is high for lumpy wholesale demand, therefore weekly-total MAPE is reported as the planning metric.", italic=True, size=9.5, color=GREY)
    page_break(doc)

    # ---------------- contents
    heading(doc, "Contents", 1)
    toc = ["Project Summary", "1. Introduction & Business Case", "2. Project Overview", "3. Key Features (F-01 to F-06)", "4. Technology Stack", "5. Architecture & Workflow",
           "6. Dataset, Cleaning & New Columns", "7. 28-Day Execution Timeline"] + [f"{8 + i}. {SECTION_TEXT[s][0]}" for i, s in enumerate([s for s in SECTION_ORDER if any(v['section'] == s for v in idx.values())])]
    n_ = 8 + len([s for s in SECTION_ORDER if any(v['section'] == s for v in idx.values())])
    toc += [f"{n_}. Acceptance Scorecard", f"{n_ + 1}. Deployment & Operations", f"{n_ + 2}. Power BI Dashboards (12 pages)", f"{n_ + 3}. Challenges & How They Were Solved",
            f"{n_ + 4}. Personal Reflection", "Appendix A - Chart index", "Appendix B - Repository structure & how to run"]
    bullets(doc, toc)
    page_break(doc)

    # ---------------- 1-2
    heading(doc, "1. Introduction & Business Case", 1)
    para(doc, "Retailers lose billions because of poor demand forecasting and stock mismanagement: shelves run empty when demand is under-estimated and capital is locked in slow stock when it is over-estimated. "
              "At the same time, acquiring a new customer costs far more than keeping an existing one, so identifying customers who are about to leave is valuable.")
    para(doc, "Mission: build an end-to-end data science platform that ingests sales, customer and inventory data and delivers accurate demand forecasts, customer segmentation, churn prediction and inventory optimisation recommendations.")
    bullets(doc, ["Reduce stock-outs through accurate demand forecasting", "Increase revenue through better inventory decisions", "Improve retention by identifying at-risk customers early",
                  "Process 10M+ transactions per month with daily batch jobs under 5 minutes (design target; large files are sampled by customer in Colab)"])
    heading(doc, "2. Project Overview", 1)
    table(doc, ["Topic", "Description"], [
        ["Vision", "One analytics platform that turns retail transactions into forecasts, customer actions and replenishment plans."],
        ["Target users", "Supermarket chains, fashion retailers and e-commerce companies: merchandisers, supply-chain planners, CRM / marketing teams, management."],
        ["Business value", "Fewer stock-outs, lower holding cost, targeted retention campaigns, one source of truth dashboard."],
        ["Non-functional goals", "MAPE <= 12 % (weekly totals), daily batch < 5 min, scalable to 10M+ transactions / month, full MLflow tracking and drift detection."],
    ], widths=[1.4, 5.2])
    heading(doc, "3. Key Features", 1)
    mp = lambda k: ("Met" if g(acc, k, "met", default=None) else ("Not met" if g(acc, k, "met", default=None) is False else "n/a"))
    table(doc, ["ID", "Feature", "Description", "Acceptance criteria", "Status"], [
        ["F-01", "Data ingestion & cleaning", "Kaggle / upload / demo loaders, ETL, validation", "Automated pipeline + quality checks", mp("F-01")],
        ["F-02", "Customer segmentation", "RFM + K-Means / DBSCAN", "6-8 segments with interpretation", mp("F-02")],
        ["F-03", "Demand forecasting", "Prophet + LSTM hybrid, 30 days", "MAPE <= 12 %", mp("F-03")],
        ["F-04", "Churn prediction", "XGBoost + SHAP", "AUC >= 0.88, precision@20 % >= 0.75", f"{mp('F-04a')} / {mp('F-04b')}"],
        ["F-05", "Inventory optimisation", "Safety stock, ROP, EOQ, order qty", "Reduce over/under-stock 25-40 %", mp("F-05")],
        ["F-06", "Interactive dashboard", "Streamlit (5 pages) + Power BI (12 pages)", "Insights, what-if, exports", "Delivered"],
    ], widths=[0.5, 1.3, 1.9, 1.9, 0.9], font=8.5)
    heading(doc, "4. Technology Stack", 1)
    table(doc, ["Category", "Technology", "Rationale / alternatives"], [
        ["Language", "Python 3.11", "Data-science ecosystem"], ["Data processing", "Pandas, NumPy, Scikit-learn", "Core manipulation and ML"],
        ["Forecasting", "Prophet + LSTM (PyTorch)", "Hybrid: calendar effects + non-linear dynamics"], ["Churn / explainability", "XGBoost, SHAP", "Best tabular accuracy + transparent drivers"],
        ["Tuning", "Optuna", "Bayesian search, faster than grid search"], ["Dashboard", "Streamlit, Plotly, Power BI", "Fast interactive analytics / business reporting"],
        ["Serving", "FastAPI", "Typed, fast REST endpoints"], ["Experiment tracking", "MLflow (SQLite)", "Versioning and reproducibility, no paid service"],
        ["Monitoring", "PSI / KS, Evidently AI, Prometheus + Grafana", "Drift and performance monitoring"], ["Orchestration", "Airflow DAG, GitHub Actions", "Retraining and CI/CD"],
        ["Deployment", "Docker, Kubernetes, Streamlit Cloud", "Consistent, scalable deployment"]], widths=[1.4, 2.3, 2.9])

    heading(doc, "5. Architecture & Workflow", 1)
    para(doc, "RetailPulse has five layers: Data -> Feature Engineering (RFM, lags, rolling averages) -> Models (segmentation, forecasting, churn, inventory, MLflow registry) -> Serving (FastAPI, Streamlit, Power BI) -> Monitoring (drift, performance tracking) with a retraining feedback loop.")
    figure(doc, "00_architecture_diagram", idx, 6.4)
    figure(doc, "00_workflow", idx, 6.4)

    heading(doc, "6. Dataset, Cleaning & New Columns", 1)
    para(doc, f"Data source used for this run: {g(data, 'source')}. Raw rows: {fnum(g(data, 'raw_rows', default=None), 0)}; clean sales rows: {fnum(g(data, 'final_rows', default=None), 0)} "
              f"({fnum(g(clean_, 'retention_pct', default=None), 1)} % retained). Customers: {fnum(g(data, 'customers', default=None), 0)}, products: {fnum(g(data, 'products', default=None), 0)}, countries: {fnum(g(data, 'countries', default=None), 0)}.")
    table(doc, ["Cleaning step", "Rows"], [[k.replace("_", " "), fnum(g(data, k, default=None), 0)] for k in
                                           ["raw_rows", "duplicates_removed", "returns_separated", "invalid_qty_or_price_removed", "non_product_codes_removed", "extreme_outliers_removed", "rows_without_customer_id", "final_rows"]], widths=[4, 2.2])
    para(doc, "Two columns were added to the cleaned CSV / Power BI file:", bold=True)
    bullets(doc, ["invoice_no - numeric part of the invoice id. Effect: a clean integer key for distinct-order counts, joins with returns and Power BI relationships (removes prefixes such as 'C' or 'A').",
                  "order_value - total revenue of the whole invoice repeated on each line. Effect: average-order-value, basket-value bands and order-level filters work without re-grouping, and visuals stay fast."])
    page_break(doc)

    heading(doc, "7. 28-Day Execution Timeline", 1)
    table(doc, ["Week", "Days", "Deliverables", "Notebook(s)"], [
        ["1 - Exploration & preparation", "1-7", "EDA, cleaning, RFM segmentation, time-series prep, Prophet, LSTM, MLflow baselines", "01-07"],
        ["2 - Advanced modelling", "8-14", "Hybrid forecaster, churn + SHAP, inventory, Optuna, drift detection, retraining pipeline", "08-14"],
        ["3 - Dashboard & analytics", "15-21", "Streamlit 5 pages, what-if, alerts, CSV/PDF export, Power BI (12 pages)", "15 + app/"],
        ["4 - Deployment & polish", "22-28", "Docker, Kubernetes, GitHub Actions, cloud deploy, monitoring, load test, report, demo video", "Dockerfile, k8s/, .github/"]],
        widths=[1.7, 0.6, 3.5, 0.9], font=8.5)

    # ---------------- technical sections
    n = 8
    present = [s for s in SECTION_ORDER if any(v["section"] == s for v in idx.values())]
    for s in present:
        title, text = SECTION_TEXT.get(s, (s, ""))
        heading(doc, f"{n}. {title}", 1)
        para(doc, text)
        if s in ("Forecasting - LSTM", "Forecasting - Hybrid") and "fallback" in str(fc.get("engines", "")).lower():
            para(doc, "Note for this run: PyTorch was not installed in the environment that produced these numbers, so a gradient-boosting lag model stood in for the LSTM "
                      "(the engine is printed in the notebook). Re-run the notebooks in Google Colab, where PyTorch is pre-installed, to obtain the true LSTM and ensemble results.", italic=True, size=9.5, color=GREY)
        if s == "Forecasting - Hybrid" and fc.get("table"):
            tb = fc["table"]
            cols = ["MAPE_daily_%", "MAPE_weekly_%", "WAPE_%", "sMAPE_%", "RMSE", "MAE"]
            table(doc, ["Model"] + cols, [[m] + [fnum(v.get(c), 2) for c in cols] for m, v in tb.items()], font=8)
            para(doc, f"Blend weight (Prophet) = {fnum(fc.get('blend_weight_prophet'), 2)}; engines: {fc.get('engines')}.", italic=True, size=9, color=GREY)
        if s == "Churn Prediction" and ch.get("table"):
            cols = ["AUC-ROC", "PR-AUC", "F1", "Precision@top20%", "Recall@top20%", "Brier"]
            table(doc, ["Model"] + cols, [[m] + [fnum(v.get(c), 3) for c in cols] for m, v in ch["table"].items()], font=8)
        if s == "Customer Segmentation" and seg.get("segments"):
            table(doc, ["Segment", "Customers", "Revenue share %"], [[k, fnum(v, 0), fnum(g(seg, "revenue_share_pct", k, default=None), 1)] for k, v in seg["segments"].items()], widths=[3, 1.5, 1.5])
        if s == "Inventory Optimization" and inv:
            table(doc, ["Metric", "Value"], [["Lost-sales reduction (back-test)", f"{fnum(inv.get('lost_sales_reduction_pct'), 1)} %"], ["Total cost reduction", f"{fnum(inv.get('cost_reduction_pct'), 1)} %"],
                                              ["Inventory level change", f"{fnum(inv.get('inventory_change_pct'), 1)} %"], ["Fill rate naive -> RetailPulse", f"{fnum(inv.get('fill_rate_naive_pct'), 1)} % -> {fnum(inv.get('fill_rate_opt_pct'), 1)} %"],
                                              ["SKUs: reorder now / overstock", f"{inv.get('reorder_now', 'n/a')} / {inv.get('overstock', 'n/a')}"]], widths=[3.5, 2.5])
        if s == "Tuning" and tun:
            table(doc, ["Item", "Value"], [["XGBoost default test AUC", fnum(tun.get("xgb_default_test_auc"), 4)], ["XGBoost tuned test AUC", fnum(tun.get("xgb_tuned_test_auc"), 4)],
                                           ["Prophet best validation WAPE", f"{fnum(tun.get('prophet_best_val_wape'), 2)} %"]], widths=[3.5, 2.5])
        keys = [k for k, v in idx.items() if v["section"] == s]
        if compact:
            keys = [k for k in keys if k in COMPACT_KEYS]
            figure_grid(doc, keys, idx, 2, 3.15)
        else:
            for k in keys:
                figure(doc, k, idx, 6.0)
        n += 1

    # ---------------- acceptance
    heading(doc, f"{n}. Acceptance Scorecard", 1)
    if acc:
        table(doc, ["ID", "Requirement", "Result", "Status"], [[k, v.get("requirement"), v.get("result"), {True: "MET", False: "NOT MET", None: "NOT RUN"}[v.get("met")]] for k, v in acc.items()], widths=[0.6, 2.5, 2.6, 0.9], font=8.5)
    else:
        para(doc, "Run notebook 14 to generate the acceptance scorecard.")
    figure(doc, "14_02_acceptance_scorecard", idx, 5.6) if not compact else None
    n += 1

    heading(doc, f"{n}. Deployment & Operations", 1)
    bullets(doc, ["Platform: Streamlit Community Cloud (free, HTTPS) or Hugging Face Spaces; container image for AWS / GCP.",
                  "Docker: multi-stage image (builder + slim runtime) running Streamlit on port 8501 with a health check.",
                  "Kubernetes: Deployment (2 replicas, liveness / readiness probes), Service and Ingress manifests in k8s/.",
                  "CI/CD: GitHub Actions runs ruff, pytest and a smoke test of the pipeline on every push / pull request, then builds the Docker image.",
                  "Serving: FastAPI endpoints /health, /forecast, /churn/predict and /segment.",
                  "Monitoring: PSI / KS drift tables, rolling WAPE monitor, Evidently HTML report, weekly Airflow DAG that retrains when drift is detected; Prometheus + Grafana recommended for production metrics.",
                  "Security: customer data anonymisation, role-based access in the dashboard (planned), JWT-secured API endpoints (recommended), audit logging of sensitive operations."])
    n += 1
    heading(doc, f"{n}. Power BI Dashboards (12 pages)", 1)
    para(doc, "A Power BI-ready file (powerbi_ready.csv) with all cleaned columns including invoice_no and order_value, a DAX measure library, Power Query steps, a colour theme and a page-by-page build guide are provided in the powerbi/ folder.")
    table(doc, ["Page", "Dashboard", "Key visuals"], [
        ["1", "Executive Summary", "KPI cards, revenue trend, top country, top product"], ["2", "Sales Trend Analysis", "Line + MoM / YoY, moving average"],
        ["3", "Product Performance", "Top / bottom products, category treemap, Pareto"], ["4", "Customer Analytics", "Customers, new vs returning, RFM segments"],
        ["5", "Region / Country Sales", "Map, country ranking"], ["6", "Monthly & Seasonal Sales", "Month x weekday matrix, seasonality"],
        ["7", "Customer Behaviour", "Frequency, basket size, cohort matrix"], ["8", "Profit & Revenue Analysis", "Revenue, estimated margin, price bands"],
        ["9", "Inventory Risk", "Reorder-now SKUs, days of cover, ABC"], ["10", "Order & Transaction", "Orders (distinct invoice_no), order_value bands"],
        ["11", "Advanced Analytics", "Forecast vs actual, churn risk, decomposition"], ["12", "Interactive Filters", "Slicers, drill-through, bookmarks"]], widths=[0.5, 2.0, 4.1], font=8.5)
    n += 1
    heading(doc, f"{n}. Challenges & How They Were Solved", 1)
    table(doc, ["Challenge", "Solution"], [
        ["Non-stationary, lumpy daily demand with closed weekdays", "Decomposition, ADF/KPSS, closed-day detection, weekly-total MAPE as planning metric, hybrid model"],
        ["Churn label leakage", "Features built only from history before a cut-off, label from the following 90 days, chronological logic"],
        ["Accuracy vs interpretability", "XGBoost with SHAP global and per-customer explanations"],
        ["No inventory table in public data", "Stock on hand simulated and clearly labelled; policy evaluated with a real-demand back-test"],
        ["Huge files (10M rows) in Colab", "Customer-level hash sampling keeps whole customer histories while limiting memory"],
        ["Library API changes (Evidently, Great Expectations)", "Native PSI/KS and validation checks plus optional Evidently with version fallback"]], widths=[2.7, 3.9], font=8.5)
    n += 1
    heading(doc, f"{n}. Personal Reflection", 1)
    para(doc, f"By {AUTHOR}. (Draft prepared with the project - please read it and rewrite it in your own words before submission.)", italic=True, size=9.5, color=GREY)
    para(doc, "Key learnings: a good forecast starts with understanding the calendar - closed days, weekly rhythm and holiday peaks mattered more than model complexity. "
              "Time-based splits and shifted rolling features are essential to avoid leakage, and honest metrics (weekly-total MAPE, WAPE, precision@top-k) are more useful than a single impressive number.")
    para(doc, "Industry best practices applied: reproducible environment, MLflow tracking, validation checks, drift monitoring with an automatic retrain trigger, explainable models and CI/CD.")
    para(doc, "Future roadmap: hierarchical (SKU x store) forecasting, promotion and price features, real inventory feed, uplift modelling for retention offers, feature store, streaming ingestion and automated A/B evaluation of campaigns.")
    page_break(doc)

    heading(doc, "Appendix A - Chart index", 1)
    rows = [[i + 1, v["title"], v["section"], k] for i, (k, v) in enumerate(sorted(idx.items()))]
    table(doc, ["#", "Chart", "Section", "File"], rows, widths=[0.4, 3.0, 1.7, 1.6], font=7.5)
    heading(doc, "Appendix B - Repository structure & how to run", 1)
    para(doc, "RetailPulse/ - notebooks/ (00 MAIN + 01-15), src/ (retailpulse_utils.py, retailpulse_report.py), app/ (Streamlit), api/ (FastAPI), data/ (sample + processed), powerbi/, k8s/, airflow/, scripts/, tests/, .github/workflows/, Dockerfile, requirements.txt, README.md.", size=10)
    bullets(doc, ["Colab: open notebooks/00_RetailPulse_MAIN.ipynb -> Runtime -> Run all (dataset loads from Kaggle, upload or demo).", "Local: pip install -r requirements.txt; streamlit run app/streamlit_app.py.",
                  "Tests: pytest -q.   Retrain: python scripts/retrain_pipeline.py --force."])
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    doc.save(out_path)
    return out_path


if __name__ == "__main__":
    out = sys.argv[1] if len(sys.argv) > 1 else str(ru.REPORTS / "RetailPulse_Project_Report_FULL.docx")
    print(build_report(out, compact="--compact" in sys.argv))

In [ ]:
import importlib, retailpulse_report as rr; importlib.reload(rr)
full = rr.build_report(str(ru.REPORTS / "RetailPulse_Project_Report_FULL.docx"), compact=False)
short = rr.build_report(str(ru.REPORTS / "RetailPulse_Zidio_Submission_Report.docx"), compact=True)
print("Created:", full, "\n         ", short)
try:
    from google.colab import files
    files.download(full); files.download(short)
except Exception:
    print("(Not in Colab - find the reports in", ru.REPORTS, ")")

## Summary of Notebook 15
* Streamlit dashboard written + Colab launch recipe; **FULL report** (every chart explained) and a **compact Zidio-submission report** (10-18 pages) generated from this run's real numbers.